# Setup

In [1]:

# STEP 1: Fresh WDT Multiclass Agent Environment

from pathlib import Path
import os
import sys
import getpass

import pandas as pd
import numpy as np
import xgboost as xgb

from dotenv import load_dotenv
from huggingface_hub import InferenceClient

# --------------------------------------------------
# 1. Project setup
# --------------------------------------------------

PROJECT_DIR = Path(r"C:\WDT")
WORK_DIR = PROJECT_DIR / "agentic_multiclass"

WORK_DIR.mkdir(parents=True, exist_ok=True)

# Same raw data, completely new experiment
NETWORK_DIR = PROJECT_DIR / "cyber"

print("Python:", sys.executable)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("XGBoost:", xgb.__version__)

# --------------------------------------------------
# 2. Hugging Face authentication
# --------------------------------------------------

load_dotenv(
    dotenv_path=PROJECT_DIR / ".env",
    override=False
)

HF_TOKEN = (
    os.getenv("HF_TOKEN")
    or os.getenv("HUGGINGFACEHUB_API_TOKEN")
    or os.getenv("HUGGING_FACE_HUB_TOKEN")
)

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass(
        "Enter your Hugging Face token: "
    )

if not HF_TOKEN:
    raise RuntimeError(
        "Hugging Face authentication is required."
    )

# --------------------------------------------------
# 3. Initialize Qwen3
# --------------------------------------------------

MODEL_ID = "Qwen/Qwen3-32B"

client = InferenceClient(
    api_key=HF_TOKEN
)

# Reasoning configuration for future calls.
# No inference request is made in this cell.

AGENT_CONFIG = {
    "model": MODEL_ID,
    "temperature": 0.6,
    "top_p": 0.95,
    "max_tokens": 8000,
    "thinking_preference": "enabled_if_supported",
    "role": "independent_multiclass_preprocessing_agent"
}

# --------------------------------------------------
# 4. Locate raw data
# --------------------------------------------------

physical_files = sorted(
    PROJECT_DIR.rglob("phy_*.csv")
)

network_files = sorted(
    NETWORK_DIR.glob("*.csv")
)

print("\nRAW PHYSICAL FILES")
for path in physical_files:
    print(path.name)

print("\nRAW NETWORK FILES")
for path in network_files:
    print(path.name)

# --------------------------------------------------
# 5. Verify initialization
# --------------------------------------------------

print("\n" + "=" * 50)
print("WDT MULTICLASS EXPERIMENT INITIALIZED")
print("=" * 50)

print("Workspace:", WORK_DIR)
print("Agent:", MODEL_ID)
print("HF token: Available (value hidden)")
print("Maximum output tokens:", AGENT_CONFIG["max_tokens"])
print("Thinking preference:", AGENT_CONFIG["thinking_preference"])
print("Inference request made: No")
print("Previous preprocessing loaded: No")
print("Previous model loaded: No")

print("\nStep 1 complete.")


Python: c:\WDT\.venv\Scripts\python.exe
Pandas: 3.1.0
NumPy: 2.5.4
XGBoost: 3.4.1

RAW PHYSICAL FILES
phy_att_1.csv
phy_att_2.csv
phy_att_3.csv
phy_att_4.csv
phy_norm.csv

RAW NETWORK FILES
attack_1.csv
attack_2.csv
attack_3.csv
attack_4.csv
normal.csv

WDT MULTICLASS EXPERIMENT INITIALIZED
Workspace: C:\WDT\agentic_multiclass
Agent: Qwen/Qwen3-32B
HF token: Available (value hidden)
Maximum output tokens: 8000
Thinking preference: enabled_if_supported
Inference request made: No
Previous preprocessing loaded: No
Previous model loaded: No

Step 1 complete.


In [4]:

# STEP 2: Inspect raw WDT multiclass labels and structure

import pandas as pd

physical_data = {}
physical_label_results = []
physical_schema = {}

for name, path in sorted(physical_paths.items()):

    # File-specific encoding and separator
    if name == "phy_att_4":
        encoding = "utf-8-sig"
        separator = ","
    else:
        encoding = "utf-16"
        separator = "\t"

    df = pd.read_csv(
        path,
        encoding=encoding,
        sep=separator,
        low_memory=False
    )

    df.columns = df.columns.str.strip()
    physical_data[name] = df

    physical_schema[name] = {
        "rows": len(df),
        "columns": list(df.columns),
        "dtypes": {
            col: str(dtype)
            for col, dtype in df.dtypes.items()
        }
    }

    label_columns = [
        col for col in df.columns
        if col.lower() in {"label", "label_n", "lable_n"}
    ]

    for label_col in label_columns:
        for label, count in df[label_col].value_counts(
            dropna=False
        ).items():
            physical_label_results.append({
                "acquisition": name,
                "label_column": label_col,
                "label_value": str(label),
                "count": int(count)
            })

physical_labels = pd.DataFrame(physical_label_results)

print("PHYSICAL LABEL DISTRIBUTIONS")
display(physical_labels)

print("\nPHYSICAL DATASET STRUCTURE")
display(pd.DataFrame([
    {
        "acquisition": name,
        "rows": info["rows"],
        "columns": len(info["columns"])
    }
    for name, info in physical_schema.items()
]))

# Network files: sample only, without processing
network_label_results = []
network_schema = {}

for path in network_files:

    sample = pd.read_csv(
        path,
        nrows=10_000,
        low_memory=False
    )

    sample.columns = sample.columns.str.strip()

    network_schema[path.stem] = {
        "sampled_rows": len(sample),
        "columns": list(sample.columns)
    }

    label_columns = [
        col for col in sample.columns
        if col.lower() in {"label", "label_n", "lable_n"}
    ]

    for label_col in label_columns:
        for label, count in sample[label_col].value_counts(
            dropna=False
        ).items():
            network_label_results.append({
                "acquisition": path.stem,
                "label_column": label_col,
                "label_value": str(label),
                "sample_count": int(count)
            })

network_labels_sample = pd.DataFrame(
    network_label_results
)

print("\nNETWORK LABEL DISTRIBUTIONS")
print("(First 10,000 rows per file)")
display(network_labels_sample)

print("\nNETWORK FILE STRUCTURE")
display(pd.DataFrame([
    {
        "acquisition": name,
        "sampled_rows": info["sampled_rows"],
        "columns": len(info["columns"])
    }
    for name, info in network_schema.items()
]))

print("\nRaw physical data loaded successfully.")
print("No preprocessing or fusion performed.")
print("No Qwen3 inference request made.")
print("\nStep 2 complete.")


PHYSICAL LABEL DISTRIBUTIONS


,acquisition,label_column,label_value,count
0,phy_att_1,Label_n,0,1610
1,phy_att_1,Label_n,1,810
2,phy_att_1,Label,normal,1610
3,phy_att_1,Label,MITM,533
4,phy_att_1,Label,physical fault,277
5,phy_att_2,Lable_n,0,1798
6,phy_att_2,Lable_n,1,306
7,phy_att_2,Label,normal,1549
8,phy_att_2,Label,nomal,249
9,phy_att_2,Label,physical fault,123



PHYSICAL DATASET STRUCTURE


,acquisition,rows,columns
0,phy_att_1,2420,43
1,phy_att_2,2104,43
2,phy_att_3,1254,43
3,phy_att_4,1717,43
4,phy_norm,3428,43



NETWORK LABEL DISTRIBUTIONS
(First 10,000 rows per file)


,acquisition,label_column,label_value,sample_count
0,attack_1,label_n,0,10000
1,attack_1,label,normal,10000
2,attack_2,label_n,0,10000
3,attack_2,label,normal,10000
4,attack_3,label_n,0,10000
5,attack_3,label,normal,10000
6,attack_4,label_n,0,10000
7,attack_4,label,normal,10000
8,normal,label_n,0,10000
9,normal,label,normal,10000



NETWORK FILE STRUCTURE


,acquisition,sampled_rows,columns
0,attack_1,10000,16
1,attack_2,10000,16
2,attack_3,10000,16
3,attack_4,10000,16
4,normal,10000,16



Raw physical data loaded successfully.
No preprocessing or fusion performed.
No Qwen3 inference request made.

Step 2 complete.


In [6]:

# STEP 3: Full network multiclass label audit (corrected)

import pandas as pd
from collections import Counter

network_label_results_3 = []
network_label_counts_3 = {}

for path in sorted(network_files):
    name = path.stem

    # Inspect and normalize the actual file header
    header = pd.read_csv(path, nrows=0, encoding="utf-8-sig")
    header_map = {
        str(col).strip().lower(): col
        for col in header.columns
    }

    label_col = header_map.get("label")
    binary_col = header_map.get("label_n")

    if label_col is None or binary_col is None:
        print(f"\n{name}: Cannot locate label columns")
        print("Actual columns:", header.columns.tolist())
        continue

    pair_counts = Counter()
    total_rows = 0

    print(f"\nInspecting {name}...")

    for chunk in pd.read_csv(
        path,
        encoding="utf-8-sig",
        usecols=[binary_col, label_col],
        chunksize=200_000,
        low_memory=False
    ):
        chunk = chunk.rename(columns={
            binary_col: "label_n",
            label_col: "label"
        })

        total_rows += len(chunk)

        counts = chunk.groupby(
            ["label_n", "label"],
            dropna=False
        ).size()

        for pair, count in counts.items():
            pair_counts[pair] += int(count)

    network_label_counts_3[name] = pair_counts

    for (label_n, label), count in pair_counts.items():
        network_label_results_3.append({
            "acquisition": name,
            "label_n": str(label_n),
            "label": str(label),
            "count": int(count),
            "total_packets": total_rows,
            "percentage": round(
                100 * count / total_rows, 3
            )
        })

network_labels_full_3 = pd.DataFrame(network_label_results_3)

print("\nFULL NETWORK LABEL DISTRIBUTIONS")

if not network_labels_full_3.empty:
    display(
        network_labels_full_3.sort_values(
            ["acquisition", "label_n", "label"]
        ).reset_index(drop=True)
    )

    print("\nUNIQUE NETWORK CLASSES")
    print(sorted(network_labels_full_3["label"].unique()))
else:
    print("No network labels were extracted.")

print("\nStep 3 complete.")



Inspecting attack_1...

Inspecting attack_2...

Inspecting attack_3...

Inspecting attack_4...

Inspecting normal...

FULL NETWORK LABEL DISTRIBUTIONS


,acquisition,label_n,label,count,total_packets,percentage
0,attack_1,0,normal,321263,430776,74.578
1,attack_1,1,MITM,109461,430776,25.410
2,attack_1,1,anomaly,51,430776,0.012
3,attack_1,nan,nan,1,430776,0.000
4,attack_2,0,normal,4093168,5159469,79.333
5,attack_2,1,DoS,571875,5159469,11.084
6,attack_2,1,MITM,217009,5159469,4.206
7,attack_2,1,anomaly,105,5159469,0.002
8,attack_2,1,physical fault,277282,5159469,5.374
9,attack_2,1,scan,30,5159469,0.001



UNIQUE NETWORK CLASSES
['DoS', 'MITM', 'anomaly', 'nan', 'normal', 'physical fault', 'scan']

Step 3 complete.


# Agent Implementation


In [7]:

# STEP 4: Qwen3's first independent multiclass engineering assessment

import json
import re
import pandas as pd

# Verified evidence only. No preprocessing decisions imposed.

evidence_4 = {
    "research_goal": (
        "Develop an independent agentic preprocessing pipeline "
        "for multiclass classification of WDT industrial events."
    ),
    "dataset": "Water Distribution Testbed (WDT)",
    "modalities": {
        "physical": {
            "acquisition_structure": physical_schema,
            "label_distribution": physical_labels.to_dict(
                orient="records"
            )
        },
        "network": {
            "schema_samples": network_schema,
            "full_label_distribution": network_labels_full_3.to_dict(
                orient="records"
            )
        }
    },
    "raw_data_notes": [
        "Physical files contain 43 columns, including Time, "
        "Label_n or Lable_n, and Label.",
        "Network files contain 16 columns, including Time, "
        "label_n, and label.",
        "Network label counts were obtained by scanning "
        "the full raw packet CSV files.",
        "Physical and network observations have different "
        "sampling granularities.",
        "Physical and network labels may not agree at "
        "identical or nearby timestamps.",
        "Raw data has not yet been preprocessed or fused."
    ],
    "previous_research_lessons": [
        "The previous binary experiment encountered poor "
        "cross-acquisition generalization.",
        "Network coverage differed substantially between "
        "acquisitions.",
        "A valid-looking model score does not establish "
        "generalization.",
        "Agent decisions should be supported by diagnostics "
        "and falsifiable hypotheses.",
        "The previous binary preprocessing and fusion "
        "decisions are not inherited in this experiment."
    ],
    "constraints": {
        "task": "multiclass classification",
        "downstream_classifier": "XGBoost",
        "agent_authority": (
            "Choose preprocessing decisions independently "
            "based on evidence."
        ),
        "protect_evaluation": (
            "Develop and document an appropriate split "
            "and leakage prevention policy before model selection."
        )
    }
}

prompt_4 = f"""
You are Qwen3-32B, acting as an independent AI preprocessing
engineer for a research project involving the Water Distribution
Testbed dataset.

Your responsibility is to design a technically rigorous
agentic preprocessing workflow for multiclass classification.

You have not inherited the preprocessing methods from our
previous binary classification experiment.

You must make engineering decisions independently, using
dataset evidence rather than predetermined techniques.

Here is the verified raw dataset information:

{json.dumps(evidence_4, indent=2, default=str)}

Your first task is NOT to train a classifier.

Analyze the data and determine what must be established
before reliable multiclass preprocessing can begin.

Pay particular attention to:
- Differences between physical and network label taxonomies.
- Rare and missing classes.
- The relationship between descriptive and binary labels.
- The distinction between physical faults and cyberattacks.
- Potential ambiguity when labels disagree across modalities.
- The risk of acquisition-specific shortcuts and leakage.
- Whether every target class can actually be learned and tested.

You have complete authority to propose diagnostics,
class-definition policies, preprocessing operations,
or experimental safeguards.

Do not assume a particular label mapping or fusion method.
Do not assume classes should be merged or removed.
Do not copy the previous binary pipeline.

Make ONE prioritized engineering decision for our next step.

Respond with:

1. Your interpretation of the raw datasets.
2. The most important unresolved issue.
3. Your independently selected next action.
4. Specific Python-implementable diagnostic or
   preprocessing instructions.
5. What evidence would determine the following decision.
6. Important risks or limitations.

Be precise and verify the supplied counts.
Distinguish observed facts from hypotheses.
Do not execute code.
"""

messages_4 = [
    {
        "role": "system",
        "content": (
            "You are an independent industrial AI preprocessing "
            "research agent. Reason critically from verified "
            "dataset evidence. Prioritize correctness, "
            "reproducibility, temporal integrity, and "
            "generalization over novelty."
        )
    },
    {
        "role": "user",
        "content": prompt_4
    }
]

response_4 = client.chat.completions.create(
    model=MODEL_ID,
    messages=messages_4,
    tool_choice="none",
    temperature=AGENT_CONFIG["temperature"],
    top_p=AGENT_CONFIG["top_p"],
    max_tokens=AGENT_CONFIG["max_tokens"]
)

reply_4 = response_4.choices[0].message
raw_reply_4 = reply_4.content or ""

# Remove a reasoning block from displayed output if present.
# The original response remains available in response_4.

clean_reply_4 = re.sub(
    r"<think>.*?</think>",
    "",
    raw_reply_4,
    flags=re.DOTALL
).strip()

print("QWEN3: INITIAL MULTICLASS ENGINEERING ASSESSMENT\n")
print(clean_reply_4)

print("\nFinish reason:", response_4.choices[0].finish_reason)
print("\nStep 4 complete.")


QWEN3: INITIAL MULTICLASS ENGINEERING ASSESSMENT

1. **Interpretation of Raw Datasets**  
   - **Physical Data**: Contains 5 acquisitions (phy_att_1-4, phy_norm) with 43 columns (42 features + 1 label). Label columns (`Label_n`/`Label`) show inconsistent taxonomy:  
     - `Label_n` uses `0`/`1` for "normal"/anomalies, but `Label` strings vary (e.g., "normal", "MITM", "physical fault", "DoS", "scan", "nomal").  
     - Typos exist (e.g., `Lable_n` in phy_att_2).  
     - Class counts differ significantly across acquisitions (e.g., phy_att_1 has 1,610 "normal" but only 277 "physical fault").  
   - **Network Data**: 5 acquisitions (attack_1-4, normal) with 16 columns. Labels include "normal", "MITM", "DoS", "physical fault", "scan", and "anomaly". Class distributions vary widely (e.g., attack_3 has 56% "DoS" but only 6% "physical fault").  
   - **Critical Discrepancies**:  
     - Physical and network modalities have overlapping but non-identical label sets (e.g., "physical fault" appe

In [8]:

# STEP 5: Unified multiclass taxonomy and label integrity audit

import pandas as pd

CANONICAL_CLASSES = [
    "normal", "MITM", "DoS", "physical fault", "scan"
]

LABEL_NORMALIZATION = {
    "nomal": "normal"
}

# Preserve all original labels
physical_multiclass = {}
physical_audit_rows = []

for name, raw in physical_data.items():
    df = raw.copy()

    binary_col = (
        "Label_n" if "Label_n" in df.columns else "Lable_n"
    )

    df["target_class"] = (
        df["Label"]
        .astype("string")
        .str.strip()
        .replace(LABEL_NORMALIZATION)
    )

    df["target_binary"] = pd.to_numeric(
        df[binary_col], errors="coerce"
    )

    df["label_consistent"] = (
        ((df["target_class"] == "normal") &
         (df["target_binary"] == 0))
        |
        ((df["target_class"].isin(
            CANONICAL_CLASSES[1:]
        )) &
         (df["target_binary"] == 1))
    ).fillna(False)

    df["target_supported"] = (
        df["target_class"].isin(CANONICAL_CLASSES)
    ).fillna(False)

    physical_multiclass[name] = df

    physical_audit_rows.append({
        "acquisition": name,
        "rows": len(df),
        "unsupported_labels": int(
            (~df["target_supported"]).sum()
        ),
        "binary_mismatches": int(
            (~df["label_consistent"]).sum()
        )
    })

physical_audit_5 = pd.DataFrame(physical_audit_rows)

print("PHYSICAL LABEL INTEGRITY")
display(physical_audit_5)

# Network: use the full-file counts already collected
# without rescanning millions of packets.
network_audit_5 = network_labels_full_3.copy()

network_audit_5["canonical_label"] = (
    network_audit_5["label"]
    .astype("string")
    .str.strip()
    .replace(LABEL_NORMALIZATION)
)

network_audit_5["supported"] = (
    network_audit_5["canonical_label"]
    .isin(CANONICAL_CLASSES)
    .fillna(False)
)

print("\nNETWORK LABELS REQUIRING REVIEW")
display(
    network_audit_5.loc[
        ~network_audit_5["supported"],
        ["acquisition", "label_n", "label", "count"]
    ]
)

print("\nPHYSICAL CLASS DISTRIBUTION")
display(
    pd.concat(
        [
            df.assign(acquisition=name)[
                ["acquisition", "target_class"]
            ]
            for name, df in physical_multiclass.items()
        ],
        ignore_index=True
    ).groupby(
        ["acquisition", "target_class"],
        dropna=False
    ).size().unstack(fill_value=0)
)

print("\nStep 5 complete.")


PHYSICAL LABEL INTEGRITY


,acquisition,rows,unsupported_labels,binary_mismatches
0,phy_att_1,2420,0,0
1,phy_att_2,2104,0,0
2,phy_att_3,1254,0,0
3,phy_att_4,1717,0,0
4,phy_norm,3428,0,0



NETWORK LABELS REQUIRING REVIEW


,acquisition,label_n,label,count
1,attack_1,1,anomaly,51
3,attack_1,nan,nan,1
5,attack_2,1,anomaly,105
13,attack_3,1,anomaly,74
15,attack_3,nan,nan,1
19,attack_4,nan,nan,1



PHYSICAL CLASS DISTRIBUTION


target_class,DoS,MITM,normal,physical fault,scan
acquisition,,,,,
phy_att_1,0,533,1610,277,0
phy_att_2,80,96,1798,123,7
phy_att_3,77,114,911,152,0
phy_att_4,153,265,1159,133,7
phy_norm,0,0,3428,0,0



Step 5 complete.


In [22]:
import pandas as pd

for name, df in physical_multiclass.items():
    if not pd.api.types.is_datetime64_any_dtype(df["Time"]):
        df["Time"] = pd.to_datetime(df["Time"], dayfirst=True, errors="raise")

for name in ["phy_norm", "phy_att_1", "phy_att_2", "phy_att_3"]:
    t = physical_multiclass[name]["Time"]
    print(name, t.min(), "->", t.max(), "| sorted:", t.is_monotonic_increasing)

phy_norm 2021-04-09 11:30:50 -> 2021-04-09 12:27:57 | sorted: True
phy_att_1 2021-04-09 18:23:28 -> 2021-04-09 19:03:47 | sorted: True
phy_att_2 2021-04-19 15:37:12 -> 2021-04-19 16:12:16 | sorted: True
phy_att_3 2021-04-09 19:42:12 -> 2021-04-09 20:03:05 | sorted: True


In [9]:

# STEP 6: Qwen3 taxonomy and evaluation-design decision

import json
import re

evidence_6 = {
    "physical_label_integrity": physical_audit_5.to_dict(
        orient="records"
    ),
    "physical_class_distribution": {
        name: (
            df["target_class"]
            .value_counts(dropna=False)
            .to_dict()
        )
        for name, df in physical_multiclass.items()
    },
    "network_unresolved_labels": network_audit_5.loc[
        ~network_audit_5["supported"],
        ["acquisition", "label_n", "label", "count"]
    ].to_dict(orient="records"),
    "confirmed_physical_classes": CANONICAL_CLASSES,
    "important_findings": [
        "All physical descriptive labels agree with binary labels.",
        "Physical label typo nomal was normalized in a new column.",
        "Network anomaly labels are unspecified attack types.",
        "Network scan and physical scan counts are extremely small.",
        "No raw labels have been overwritten or removed.",
        "No feature preprocessing, fusion, or training has occurred."
    ]
}

prompt_6 = f"""
You are the independent AI engineering agent for WDT
multiclass preprocessing.

Review these verified label-integrity findings:

{json.dumps(evidence_6, indent=2, default=str)}

We need a defensible multiclass target definition and
evaluation design before preprocessing.

You have engineering authority to make these decisions.
However, do not invent attack labels or assume binary
and descriptive labels have identical meanings.

Make ONE coherent decision covering:

1. The multiclass taxonomy.
2. Treatment of unspecified network anomaly labels.
3. Handling of rare classes, especially scan.
4. Which modality should provide the target labels
   for a fused physical-network classifier, and why.
5. Acquisition-level train, validation, and test design.
6. Safeguards against temporal and label leakage.
7. Exact next implementation action.

If a reliable five-class experiment is not possible,
explain the limitation rather than forcing it.

Keep the recommendation concise but technically precise.

Do not execute code.
"""

messages_6 = [
    {
        "role": "system",
        "content": (
            "You are an independent preprocessing research "
            "agent. Favor verifiable evidence, defensible "
            "experimental design, and minimal unnecessary work."
        )
    },
    {"role": "user", "content": prompt_6}
]

response_6 = client.chat.completions.create(
    model=MODEL_ID,
    messages=messages_6,
    tool_choice="none",
    temperature=AGENT_CONFIG["temperature"],
    top_p=AGENT_CONFIG["top_p"],
    max_tokens=AGENT_CONFIG["max_tokens"]
)

reply_6 = response_6.choices[0].message

clean_reply_6 = re.sub(
    r"<think>.*?</think>",
    "",
    reply_6.content or "",
    flags=re.DOTALL
).strip()

print("QWEN3: MULTICLASS TAXONOMY AND SPLIT DECISION\n")
print(clean_reply_6)

print("\nFinish reason:", response_6.choices[0].finish_reason)
print("Step 6 complete.")


QWEN3: MULTICLASS TAXONOMY AND SPLIT DECISION

### **Multiclass Preprocessing Recommendation**  
**1. Multiclass Taxonomy**  
- **Confirmed classes**: Use the five confirmed physical classes: `normal`, `MITM`, `DoS`, `physical fault`, and `scan`.  
- **Exclusion of network "anomaly"**: Network labels with value "anomaly" (e.g., label_n=1 in unresolved labels) are excluded due to ambiguity and lack of alignment with physical classes.  

**2. Network Anomaly Labels**  
- **Unspecified network "anomaly"**: Excluded from the target taxonomy to avoid conflating vague labels with physical attack types. Network data may still be used as features but not for defining multiclass targets.  

**3. Rare Classes**  
- **"scan" class**: Retained as a distinct class despite low counts (total ~14 rows across acquisitions). Use stratified sampling and note the risk of overfitting. Consider oversampling or synthetic data augmentation in later stages if needed.  

**4. Target Label Modality**  
- **Physi

In [11]:

# STEP 7: Qwen3 independently reviews the split and class policy

import json
import re

review_7 = {
    "current_proposal": {
        "training": ["phy_att_1", "phy_att_2"],
        "validation": ["phy_att_3"],
        "test": ["phy_att_4", "phy_norm"],
        "classes": [
            "normal", "MITM", "DoS",
            "physical fault", "scan"
        ]
    },
    "verified_physical_class_counts": {
        "phy_norm": {
            "normal": 3428, "MITM": 0,
            "DoS": 0, "physical fault": 0, "scan": 0
        },
        "phy_att_1": {
            "normal": 1610, "MITM": 533,
            "DoS": 0, "physical fault": 277, "scan": 0
        },
        "phy_att_2": {
            "normal": 1798, "MITM": 96,
            "DoS": 80, "physical fault": 123, "scan": 7
        },
        "phy_att_3": {
            "normal": 911, "MITM": 114,
            "DoS": 77, "physical fault": 152, "scan": 0
        },
        "phy_att_4": {
            "normal": 1159, "MITM": 265,
            "DoS": 153, "physical fault": 133, "scan": 7
        }
    },
    "independent_review_concerns": [
        "Testing on phy_norm together with phy_att_4 creates "
        "a heavily normal-dominated test population.",
        "Training on only phy_att_1 and phy_att_2 provides "
        "limited attack-acquisition diversity.",
        "The normal acquisition may contain session-specific "
        "features that differ from normal periods inside attacks.",
        "Class weighting may be preferable to resampling, "
        "but must be fitted from training data only.",
        "Scan has only 14 physical observations across "
        "all acquisitions.",
        "phy_att_3 contains no scan and is insufficient "
        "for assessing all five classes independently.",
        "A suggested alternative is acquisition-held-out "
        "development over phy_att_1, phy_att_2, phy_att_3.",
        "The attack_1 network recording has very low "
        "coverage in the previous binary experiment.",
        "Fusion evaluation should report network availability "
        "and compare models on equivalent observation populations."
    ],
    "candidate_diagnostics": [
        "Compare normal-labeled phy_norm data against normal "
        "observations from attack acquisitions to investigate "
        "session separability.",
        "Calculate network availability by acquisition and "
        "physical target class.",
        "Inspect whether scan observations form independent "
        "events or adjacent samples."
    ],
    "evaluation_protection": {
        "final_test": "phy_att_4",
        "test_policy": "Do not use to select transformations or models",
        "previous_binary_experiment": (
            "phy_att_3 and phy_att_4 were previously examined "
            "for a different binary task; document this."
        )
    }
}

prompt_7 = f"""
You are the independent WDT multiclass preprocessing
research engineer.

Your previous proposed split has been challenged by
an independent methodological review.

Review the following verified counts and concerns:

{json.dumps(review_7, indent=2)}

The review suggests some alternatives, but these are
NOT instructions. You must assess them independently.

Your task:

1. Reconsider the acquisition split and justify your choice.
2. Decide how to handle the normal-only acquisition.
3. Decide whether scan remains a fifth predictive class,
   a separately reported class, or is handled differently.
4. Establish a development evaluation protocol that
   respects acquisition boundaries and class availability.
5. Specify how class imbalance should be handled.
6. Explain how limited network coverage affects the
   planned multimodal evaluation.
7. Select ONE immediate, implementable next action.

Scientific requirements:
- Do not use phy_att_4 to choose preprocessing.
- Do not fabricate class labels.
- Do not conflate packet counts with physical observations.
- Preserve physical and network label provenance.
- Any data-dependent weighting or fitting must use
  training data within each development fold.
- Distinguish known facts from untested assumptions.
- Prioritize a compact, reproducible experiment.

Your decisions should be independent and concise.
Do not execute code.
"""

messages_7 = [
    {
        "role": "system",
        "content": (
            "You are an autonomous preprocessing research "
            "engineer. Critically evaluate methodological "
            "feedback, verify arithmetic, and make clear "
            "decisions grounded in evidence."
        )
    },
    {
        "role": "user",
        "content": prompt_7
    }
]

response_parts_7 = []
reasoning_parts_7 = []

print("QWEN3: REVISED EXPERIMENTAL DESIGN\n")

try:
    stream_7 = client.chat.completions.create(
        model=MODEL_ID,
        messages=messages_7,
        tool_choice="none",
        temperature=AGENT_CONFIG["temperature"],
        top_p=AGENT_CONFIG["top_p"],
        max_tokens=AGENT_CONFIG["max_tokens"],
        stream=True
    )

    for chunk in stream_7:
        if not chunk.choices:
            continue

        delta = chunk.choices[0].delta

        content = getattr(delta, "content", None)
        reasoning = getattr(delta, "reasoning", None)
        reasoning_content = getattr(
            delta, "reasoning_content", None
        )

        if content:
            response_parts_7.append(content)
            print(content, end="", flush=True)

        thought = reasoning or reasoning_content
        if thought:
            reasoning_parts_7.append(thought)

    clean_reply_7 = "".join(response_parts_7).strip()

    print("\n\nResponse received.")
    print("Answer characters:", len(clean_reply_7))
    print("Reasoning characters:", len("".join(reasoning_parts_7)))

except Exception as error:
    clean_reply_7 = "".join(response_parts_7).strip()

    print("\n\nStreaming interrupted:", type(error).__name__)
    print(str(error)[:400])
    print("Answer characters received:", len(clean_reply_7))

print("\nStep 7 complete.")


QWEN3: REVISED EXPERIMENTAL DESIGN



**1. Acquisition Split Reconsideration**  
Adopt **acquisition-held-out development** using `phy_att_1`, `phy_att_2`, and `phy_att_3` for training/validation (e.g., cross-validation within these acquisitions), with `phy_att_4` as the final test set. This addresses test imbalance (phy_att_4 is less normal-dominated than phy_norm) and improves attack diversity in training. Phy_att_3 (validation) remains valid for development but avoids phy_att_4 leakage.  

**2. Handling phy_norm**  
Exclude `phy_norm` from the test set to prevent overrepresentation of normal samples. Instead, treat it as a **normal-only reference set** for post-hoc analysis (e.g., evaluating normal detection in isolation). Report results on `phy_att_4` alone for primary evaluation.  

**3. Scan Class Handling**  
Retain scan as a **separately reported class** due to its low sample count (14 total). Do not exclude it, but explicitly note its limitations in generalizability. Avoid res

In [12]:
# STEP 8: Normal-session separability (pairwise, time-blocked)

import itertools
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score

assert "physical_multiclass" in globals(), "Run Step 5 first."

NAMES = ["phy_norm", "phy_att_1", "phy_att_2", "phy_att_3", "phy_att_4"]
N_BLOCKS = 5     # contiguous time blocks per acquisition
GAP = 10         # rows purged on each side of the held-out block

# Columns that are labels or helper columns, never features
META = {
    "Time", "Label", "Label_n", "Lable_n",
    "target_class", "target_binary",
    "label_consistent", "target_supported",
}

ref = physical_multiclass["phy_norm"]
feature_cols = [
    c for c in ref.columns
    if c not in META and pd.api.types.is_numeric_dtype(ref[c])
]
dropped = [c for c in ref.columns
           if c not in META and c not in feature_cols]

print("Features used:", len(feature_cols))
if dropped:
    print("Non-numeric columns skipped:", dropped)

def normal_rows(name):
    d = physical_multiclass[name]
    return d[d["target_class"] == "normal"].reset_index(drop=True).copy()

print("\nNormal-labeled rows per acquisition")
for n in NAMES:
    print(f"  {n}: {len(normal_rows(n))}")

def pair_auc(a, b):
    """Can a model tell session a from session b on held-out time blocks?"""
    parts = []
    for name, y in ((a, 0), (b, 1)):
        d = normal_rows(name)
        d["y"] = y
        d["pos"] = np.arange(len(d))
        d["blk"] = (d["pos"] * N_BLOCKS) // len(d)
        parts.append(d)

    aucs = []
    for k in range(N_BLOCKS):
        train_parts, test_parts = [], []
        for d in parts:
            held = d[d["blk"] == k]
            lo = held["pos"].min() - GAP
            hi = held["pos"].max() + GAP
            train_parts.append(
                d[(d["blk"] != k) & ~d["pos"].between(lo, hi)]
            )
            test_parts.append(held)

        train = pd.concat(train_parts)
        test = pd.concat(test_parts)

        model = xgb.XGBClassifier(
            n_estimators=200, max_depth=4, learning_rate=0.1,
            tree_method="hist", eval_metric="logloss",
            n_jobs=-1, random_state=0,
        )
        model.fit(train[feature_cols], train["y"])
        prob = model.predict_proba(test[feature_cols])[:, 1]
        aucs.append(roc_auc_score(test["y"], prob))

    return float(np.mean(aucs))

sep = pd.DataFrame(index=NAMES, columns=NAMES, dtype=float)
for a, b in itertools.combinations(NAMES, 2):
    sep.loc[a, b] = sep.loc[b, a] = pair_auc(a, b)

print("\nPAIRWISE SESSION SEPARABILITY (ROC-AUC, normal rows only)")
print("0.5 = indistinguishable, 1.0 = fully separable\n")
display(sep.round(3))

attack_names = NAMES[1:]
norm_vs_attack = [sep.loc["phy_norm", n] for n in attack_names]
attack_vs_attack = [sep.loc[a, b]
                    for a, b in itertools.combinations(attack_names, 2)]

print("phy_norm vs attack sessions: "
      f"mean {np.mean(norm_vs_attack):.3f} "
      f"(range {min(norm_vs_attack):.3f}-{max(norm_vs_attack):.3f})")
print("attack vs attack sessions:   "
      f"mean {np.mean(attack_vs_attack):.3f} "
      f"(range {min(attack_vs_attack):.3f}-{max(attack_vs_attack):.3f})")

try:
    out_path = WORK_DIR / "step8_normal_session_separability.csv"
    sep.to_csv(out_path)
    print("\nSaved:", out_path)
except NameError:
    print("\nWORK_DIR not defined; matrix not saved.")

print("\nStep 8 complete.")

Features used: 40

Normal-labeled rows per acquisition
  phy_norm: 3428
  phy_att_1: 1610
  phy_att_2: 1798
  phy_att_3: 911
  phy_att_4: 1159

PAIRWISE SESSION SEPARABILITY (ROC-AUC, normal rows only)
0.5 = indistinguishable, 1.0 = fully separable



,phy_norm,phy_att_1,phy_att_2,phy_att_3,phy_att_4
phy_norm,NaN,0.829,0.981,0.953,0.827
phy_att_1,0.829,NaN,0.949,0.605,0.725
phy_att_2,0.981,0.949,NaN,0.949,0.938
phy_att_3,0.953,0.605,0.949,NaN,0.893
phy_att_4,0.827,0.725,0.938,0.893,NaN


phy_norm vs attack sessions: mean 0.898 (range 0.827-0.981)
attack vs attack sessions:   mean 0.843 (range 0.605-0.949)

Saved: C:\WDT\agentic_multiclass\step8_normal_session_separability.csv

Step 8 complete.


In [13]:
# STEP 9: Raw-feature leave-one-acquisition-out baseline (physical only, no preprocessing)

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import f1_score, recall_score, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

assert "physical_multiclass" in globals(), "Run Step 5 first."

TARGET_MODE = "three"                      # "three" or "five"
SEEDS = [0, 1, 2]
DEV_FOLDS = ["phy_att_1", "phy_att_2", "phy_att_3"]   # phy_att_4 is never used here
assert "phy_att_4" not in DEV_FOLDS

if TARGET_MODE == "three":
    CLASS_MAP = {
        "normal": "normal",
        "physical fault": "physical_fault",
        "MITM": "cyberattack",
        "DoS": "cyberattack",
        "scan": "cyberattack",
    }
else:
    CLASS_MAP = {
        "normal": "normal",
        "MITM": "MITM",
        "DoS": "DoS",
        "physical fault": "physical_fault",
        "scan": "scan",
    }
ALL_CLASSES = list(dict.fromkeys(CLASS_MAP.values()))

META = {
    "Time", "Label", "Label_n", "Lable_n",
    "target_class", "target_binary",
    "label_consistent", "target_supported",
}
ref = physical_multiclass["phy_norm"]
feature_cols = [
    c for c in ref.columns
    if c not in META and pd.api.types.is_numeric_dtype(ref[c])
]
print("Target mode:", TARGET_MODE, "| classes:", ALL_CLASSES)
print("Features used:", len(feature_cols))

frames = {}
for name, d in physical_multiclass.items():
    d = d.copy()
    d["y"] = d["target_class"].astype(str).map(CLASS_MAP)
    assert d["y"].notna().all(), f"Unmapped labels in {name}"
    frames[name] = d

def fit_predict(train, test, seed):
    classes = sorted(train["y"].unique())
    idx = {c: i for i, c in enumerate(classes)}
    y_tr = train["y"].map(idx).to_numpy()
    weights = compute_sample_weight("balanced", y_tr)   # fitted inside the training fold
    model = xgb.XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        tree_method="hist", n_jobs=-1, random_state=seed,
    )
    model.fit(train[feature_cols], y_tr, sample_weight=weights)
    return np.array(classes)[model.predict(test[feature_cols]).astype(int)]

rows = []
pooled = {True: [], False: []}
for held in DEV_FOLDS:
    train_names = [n for n in DEV_FOLDS if n != held]
    for with_norm in (True, False):
        names = train_names + (["phy_norm"] if with_norm else [])
        train = pd.concat([frames[n] for n in names])
        test = frames[held]
        truth = test["y"].to_numpy()
        present = [c for c in ALL_CLASSES if c in set(truth)]
        for seed in SEEDS:
            pred = fit_predict(train, test, seed)
            row = {
                "fold": held,
                "phy_norm_in_train": with_norm,
                "seed": seed,
                "macro_f1": f1_score(truth, pred, labels=present,
                                     average="macro", zero_division=0),
            }
            rec = recall_score(truth, pred, labels=present,
                               average=None, zero_division=0)
            row.update({f"recall_{c}": r for c, r in zip(present, rec)})
            rows.append(row)
            if seed == SEEDS[0]:
                pooled[with_norm].append((truth, pred))

res = pd.DataFrame(rows)
recall_cols = [c for c in res.columns if c.startswith("recall_")]

summary = res.groupby(["fold", "phy_norm_in_train"]).agg(
    macro_f1_mean=("macro_f1", "mean"),
    macro_f1_std=("macro_f1", "std"),
    **{c: (c, "mean") for c in recall_cols},
).round(3)

print("\nPER-FOLD RESULTS (mean over seeds; macro-F1 over classes present in the held-out fold)")
display(summary)

print("\nOVERALL macro-F1 (mean over folds and seeds)")
display(res.groupby("phy_norm_in_train")["macro_f1"].agg(["mean", "std"]).round(3))

for with_norm in (True, False):
    truth = np.concatenate([t for t, _ in pooled[with_norm]])
    pred = np.concatenate([p for _, p in pooled[with_norm]])
    cm = pd.DataFrame(
        confusion_matrix(truth, pred, labels=ALL_CLASSES),
        index=[f"true_{c}" for c in ALL_CLASSES],
        columns=[f"pred_{c}" for c in ALL_CLASSES],
    )
    print(f"\nPOOLED OUT-OF-FOLD CONFUSION (seed {SEEDS[0]}, phy_norm in train = {with_norm})")
    display(cm)

try:
    out_path = WORK_DIR / "step9_raw_baseline_loao.csv"
    res.to_csv(out_path, index=False)
    print("\nSaved:", out_path)
except NameError:
    print("\nWORK_DIR not defined; results not saved.")

print("\nStep 9 complete. phy_att_4 was not used.")

Target mode: three | classes: ['normal', 'physical_fault', 'cyberattack']
Features used: 40

PER-FOLD RESULTS (mean over seeds; macro-F1 over classes present in the held-out fold)


macro_f1_mean  macro_f1_std  recall_normal  \
fold      phy_norm_in_train                                               
phy_att_1 False                      0.410         0.003          0.822   
          True                       0.343         0.002          0.924   
phy_att_2 False                      0.472         0.020          0.673   
          True                       0.585         0.016          0.911   
phy_att_3 False                      0.373         0.013          0.764   
          True                       0.345         0.040          0.864   

                             recall_physical_fault  recall_cyberattack  
fold      phy_norm_in_train                                             
phy_att_1 False                              0.063               0.346  
          True                               0.008               0.154  
phy_att_2 False                              0.618               0.361  
          True                               0.561               0.321  
phy_att_3 False                              0.215               0.119  
          True                               0.123               0.061


OVERALL macro-F1 (mean over folds and seeds)


,mean,std
phy_norm_in_train,,
False,0.418,0.045
True,0.424,0.123



POOLED OUT-OF-FOLD CONFUSION (seed 0, phy_norm in train = True)


,pred_normal,pred_physical_fault,pred_cyberattack
true_normal,3885,204,230
true_physical_fault,419,102,31
true_cyberattack,729,30,148



POOLED OUT-OF-FOLD CONFUSION (seed 0, phy_norm in train = False)


,pred_normal,pred_physical_fault,pred_cyberattack
true_normal,3236,227,856
true_physical_fault,376,123,53
true_cyberattack,590,40,277



Saved: C:\WDT\agentic_multiclass\step9_raw_baseline_loao.csv

Step 9 complete. phy_att_4 was not used.


In [14]:
# STEP 10: Threshold-free attack detection and subtype recall (physical-only raw baseline)

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.utils.class_weight import compute_sample_weight

for v in ["frames", "feature_cols", "CLASS_MAP", "DEV_FOLDS", "SEEDS", "TARGET_MODE"]:
    assert v in globals(), "Run Step 9 first."
assert "phy_att_4" not in DEV_FOLDS

def fit_proba(train, test, seed):
    classes = sorted(train["y"].unique())
    idx = {c: i for i, c in enumerate(classes)}
    y_tr = train["y"].map(idx).to_numpy()
    model = xgb.XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        tree_method="hist", n_jobs=-1, random_state=seed,
    )
    model.fit(train[feature_cols], y_tr,
              sample_weight=compute_sample_weight("balanced", y_tr))
    return classes, model.predict_proba(test[feature_cols])

auc_rows, sub_rows = [], []
for held in DEV_FOLDS:
    train_names = [n for n in DEV_FOLDS if n != held]
    for with_norm in (True, False):
        names = train_names + (["phy_norm"] if with_norm else [])
        train = pd.concat([frames[n] for n in names])
        test = frames[held]

        y_true = test["y"].to_numpy()
        raw = test["target_class"].astype(str).to_numpy()
        is_attack = y_true != "normal"

        for seed in SEEDS:
            classes, proba = fit_proba(train, test, seed)
            attack_score = 1.0 - proba[:, classes.index("normal")]
            pred = np.array(classes)[proba.argmax(axis=1)]

            auc_rows.append({
                "fold": held, "phy_norm_in_train": with_norm, "seed": seed,
                "base_rate": is_attack.mean(),
                "roc_auc": roc_auc_score(is_attack, attack_score),
                "pr_auc": average_precision_score(is_attack, attack_score),
            })

            for sub in sorted(set(raw) - {"normal"}):
                m = raw == sub
                sub_rows.append({
                    "fold": held, "phy_norm_in_train": with_norm,
                    "seed": seed, "subtype": sub, "n": int(m.sum()),
                    "detected_as_attack": float((pred[m] != "normal").mean()),
                    "correct_type": float((pred[m] == y_true[m]).mean()),
                })

auc_df = pd.DataFrame(auc_rows)
sub_df = pd.DataFrame(sub_rows)

print("ATTACK-vs-NORMAL RANKING (threshold-free; mean over seeds)")
display(
    auc_df.groupby(["fold", "phy_norm_in_train"])[
        ["base_rate", "roc_auc", "pr_auc"]
    ].mean().round(3)
)

print("\nRECALL BY RAW SUBTYPE (argmax decision; mean over seeds)")
print("detected_as_attack = predicted anything except normal")
display(
    sub_df.groupby(["fold", "phy_norm_in_train", "subtype"]).agg(
        n=("n", "first"),
        detected_as_attack=("detected_as_attack", "mean"),
        correct_type=("correct_type", "mean"),
    ).round(3)
)

try:
    auc_df.to_csv(WORK_DIR / "step10_ranking_by_fold.csv", index=False)
    sub_df.to_csv(WORK_DIR / "step10_subtype_recall.csv", index=False)
    print("\nSaved to", WORK_DIR)
except NameError:
    print("\nWORK_DIR not defined; results not saved.")

print("\nStep 10 complete. phy_att_4 was not used.")

ATTACK-vs-NORMAL RANKING (threshold-free; mean over seeds)


base_rate  roc_auc  pr_auc
fold      phy_norm_in_train                            
phy_att_1 False                  0.335    0.621   0.460
          True                   0.335    0.646   0.482
phy_att_2 False                  0.145    0.676   0.317
          True                   0.145    0.725   0.362
phy_att_3 False                  0.274    0.623   0.310
          True                   0.274    0.559   0.290


RECALL BY RAW SUBTYPE (argmax decision; mean over seeds)
detected_as_attack = predicted anything except normal


n  detected_as_attack  \
fold      phy_norm_in_train subtype                                   
phy_att_1 False             MITM            533               0.413   
                            physical fault  277               0.084   
          True              MITM            533               0.214   
                            physical fault  277               0.008   
phy_att_2 False             DoS              80               0.054   
                            MITM             96               0.622   
                            physical fault  123               0.808   
                            scan              7               0.286   
          True              DoS              80               0.000   
                            MITM             96               0.611   
                            physical fault  123               0.631   
                            scan              7               0.000   
phy_att_3 False             DoS              77               0.156   
                            MITM            114               0.094   
                            physical fault  152               0.344   
          True              DoS              77               0.000   
                            MITM            114               0.102   
                            physical fault  152               0.246   

                                            correct_type  
fold      phy_norm_in_train subtype                       
phy_att_1 False             MITM                   0.346  
                            physical fault         0.063  
          True              MITM                   0.154  
                            physical fault         0.008  
phy_att_2 False             DoS                    0.054  
                            MITM                   0.622  
                            physical fault         0.618  
                            scan                   0.286  
          True              DoS                    0.000  
                            MITM                   0.611  
                            physical fault         0.561  
                            scan                   0.000  
phy_att_3 False             DoS                    0.156  
                            MITM                   0.094  
                            physical fault         0.215  
          True              DoS                    0.000  
                            MITM                   0.102  
                            physical fault         0.123


Saved to C:\WDT\agentic_multiclass

Step 10 complete. phy_att_4 was not used.


In [44]:
# STEP 10b: Baseline with MINIMAL network fusion (no agent, no preprocessing beyond packets -> one row per second)
# Development acquisitions only (phy_norm, phy_att_1, phy_att_2, phy_att_3); phy_att_4 and attack_4 are never read.
# Arms (same XGBoost, same folds, same seeds as Step 9):
#   A1  raw physical only                      (Step 9 / Step 10 baseline; also scored on the network-covered rows)
#   A2  raw physical + minimal network         <- this cell: packets per second and mean packet size, nothing else
#   A3  agent pipeline: physical + 14 per-second network features (built later in Step 14a, tested in Steps 14b-21)
# Needs Steps 1, 5 (with the day-first date fix) and 9 to have run. Step 18 is optional (adds the A3 comparison).

import json
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score, roc_curve
from sklearn.utils.class_weight import compute_sample_weight

for v in ["network_files", "physical_multiclass", "frames", "feature_cols", "DEV_FOLDS", "SEEDS", "ALL_CLASSES", "WORK_DIR"]:
    assert v in globals(), f"Missing {v}: run Steps 1, 5 and 9 first."
assert "phy_att_4" not in DEV_FOLDS
assert pd.api.types.is_datetime64_any_dtype(physical_multiclass["phy_att_1"]["Time"]), \
    "Physical Time must be parsed day-first (the date fix at the end of Step 5)."

PAIRS = {"normal": "phy_norm", "attack_1": "phy_att_1", "attack_2": "phy_att_2", "attack_3": "phy_att_3"}
MIN_COLS = ["net_pkts", "net_size_mean"]          # the whole "minimal network" definition, fixed in advance
CHUNK = 500_000
REUSE_STEP14A = True                              # take the same two columns from Step 14a's file if it already exists
EPOCH, ONE_S = pd.Timestamp("1970-01-01"), pd.Timedelta("1s")

# ---------------------------------------------------------------- packets -> one row per second (2 columns)
def scan_minimal(path):
    header = pd.read_csv(path, nrows=0, encoding="utf-8-sig")
    colmap = {str(c).strip().lower(): c for c in header.columns}
    assert "time" in colmap and "size" in colmap, f"{path.name}: needs Time and Size columns"
    parts = []
    for ch in pd.read_csv(path, encoding="utf-8-sig", usecols=[colmap["time"], colmap["size"]],
                          chunksize=CHUNK, low_memory=False):
        ch.columns = ["time", "size"]
        ts = pd.to_datetime(ch["time"], format="ISO8601", errors="coerce")
        ok = ts.notna().to_numpy()
        sec = ((ts[ok].dt.floor("s") - EPOCH) // ONE_S).to_numpy()
        d = pd.DataFrame({"sec": sec, "size": pd.to_numeric(ch["size"][ok], errors="coerce").to_numpy(float)})
        parts.append(d.groupby("sec").agg(n=("size", "size"), size_sum=("size", "sum")))
    g = pd.concat(parts).groupby(level=0).sum()
    return pd.DataFrame({"net_pkts": g["n"], "net_size_mean": g["size_sum"] / g["n"]})

minimal = {}
for stem, phy in PAIRS.items():
    own = WORK_DIR / f"step10b_minimal_network_{phy}.csv"
    cached14a = WORK_DIR / f"step14a_network_features_{phy}.csv"
    if own.exists():
        minimal[phy] = pd.read_csv(own).set_index("sec"); src = "own cache"
    elif REUSE_STEP14A and cached14a.exists():
        minimal[phy] = pd.read_csv(cached14a, usecols=["sec"] + MIN_COLS).set_index("sec"); src = "Step 14a file (same definition)"
        minimal[phy].reset_index().to_csv(own, index=False)
    else:
        path = {p.stem: p for p in network_files}.get(stem)
        assert path is not None, f"network file '{stem}' not found"
        minimal[phy] = scan_minimal(path); src = "scanned packets"
        minimal[phy].reset_index().to_csv(own, index=False)
    print(f"{phy}: {len(minimal[phy]):,} seconds with packets ({src})")

# ---------------------------------------------------------------- join to the physical rows (exact second, no tolerance)
data10 = {}
for name in ["phy_norm"] + list(DEV_FOLDS):
    d = frames[name].copy()
    d["Time"] = physical_multiclass[name]["Time"]
    sec = ((d["Time"].dt.floor("s") - EPOCH) // ONE_S).to_numpy()
    nf = minimal[name].reindex(sec)
    for c in MIN_COLS:
        d[c] = nf[c].to_numpy()
    d["covered"] = d["net_pkts"].notna().to_numpy()
    data10[name] = d
print("Share of physical rows with network data:",
      {n: round(float(d["covered"].mean()), 4) for n, d in data10.items()})

# ---------------------------------------------------------------- arms
ARMS = {  # name: (columns, covered rows only?)
    "A1_physical_all_rows":              (list(feature_cols), False),
    "A1_physical_covered_rows":          (list(feature_cols), True),
    "A2_physical_plus_minimal_network":  (list(feature_cols) + MIN_COLS, True),
}

def recall_at_fpr(y, s, lim=0.05):
    fpr, tpr, _ = roc_curve(y, s)
    return float(tpr[fpr <= lim].max())

rows = []
for arm, (cols, cov_only) in ARMS.items():
    for held in DEV_FOLDS:
        names = [n for n in DEV_FOLDS if n != held] + ["phy_norm"]
        tr, te = pd.concat([data10[n] for n in names]), data10[held]
        if cov_only:
            tr, te = tr[tr["covered"]], te[te["covered"]]
        classes = sorted(set(tr["y"])); idx = {c: i for i, c in enumerate(classes)}
        ytr = tr["y"].map(idx).to_numpy()
        truth = te["y"].to_numpy(); y_att = (truth != "normal").astype(int)
        present = [c for c in ALL_CLASSES if c in set(truth)]
        for seed in SEEDS:
            m = xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1, subsample=0.8,
                                  colsample_bytree=0.8, tree_method="hist", n_jobs=-1, random_state=seed)
            m.fit(tr[cols].astype(float), ytr, sample_weight=compute_sample_weight("balanced", ytr))
            proba = m.predict_proba(te[cols].astype(float))
            pred = np.array(classes)[proba.argmax(1)]
            p_att = 1.0 - proba[:, classes.index("normal")]
            flag = pred != "normal"
            tp, fp = int((flag & (y_att == 1)).sum()), int((flag & (y_att == 0)).sum())
            fn, tn = int((~flag & (y_att == 1)).sum()), int((~flag & (y_att == 0)).sum())
            ok = 0 < y_att.sum() < len(y_att)
            rows.append({"arm": arm, "fold": held, "seed": seed, "n_test": len(te), "attack_share": y_att.mean(),
                         "precision": tp / (tp + fp) if tp + fp else 0.0, "recall": tp / (tp + fn) if tp + fn else np.nan,
                         "specificity": tn / (tn + fp) if tn + fp else np.nan,
                         "roc_auc": roc_auc_score(y_att, p_att) if ok else np.nan,
                         "pr_auc": average_precision_score(y_att, p_att) if ok else np.nan,
                         "recall_at_5pct_fpr": recall_at_fpr(y_att, p_att) if ok else np.nan,
                         "cyber_recall": float(np.mean(pred[truth == "cyberattack"] == "cyberattack")) if (truth == "cyberattack").any() else np.nan,
                         "macro_f1": f1_score(truth, pred, labels=present, average="macro", zero_division=0)})
    print("done:", arm)

res = pd.DataFrame(rows)
res.to_csv(WORK_DIR / "step10b_minimal_network_by_seed.csv", index=False)
metrics = ["n_test", "attack_share", "precision", "recall", "specificity", "roc_auc", "pr_auc", "recall_at_5pct_fpr", "cyber_recall", "macro_f1"]
full = res.groupby(["arm", "fold"])[metrics].mean().reset_index()
full.to_csv(WORK_DIR / "step10b_minimal_network_metrics.csv", index=False)

pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
print("\nA) PER FOLD (mean over seeds; XGBoost argmax rule for precision/recall/specificity)")
display(full.round(3))
print("\nB) AVERAGE OVER THE THREE FOLDS")
display(full.groupby("arm")[metrics[2:]].mean().round(3))

def deltas(arm_hi, arm_lo, src, label):
    out = []
    for f in DEV_FOLDS:
        for metric in ("roc_auc", "recall_at_5pct_fpr"):
            hi = src[(src.arm == arm_hi) & (src.fold == f)][metric]
            lo = src[(src.arm == arm_lo) & (src.fold == f)][metric]
            out.append({"comparison": label, "fold": f, "metric": metric, "delta": round(float(hi.mean() - lo.mean()), 4),
                        "lower_arm_seed_sd": round(float(lo.std()), 4)})
    d = pd.DataFrame(out)
    d["delta_above_sd"] = d["delta"] > d["lower_arm_seed_sd"]
    return d

print("\nC) WHAT THE NETWORK DATA ADDS WITHOUT ANY AGENT (A2 minus A1 on the same covered rows; informational, no verdict)")
display(deltas("A2_physical_plus_minimal_network", "A1_physical_covered_rows", res, "A2 - A1(covered)"))

s18 = WORK_DIR / "step18_all_metrics_by_seed.csv"
if s18.exists():
    r18 = pd.read_csv(s18)
    r18 = r18[(r18.model == "XGBoost") & (r18.variant.isin(["physical_covered_rows", "physical_plus_network"]))]
    if len(r18):
        a1c = res[res.arm == "A1_physical_covered_rows"].groupby("fold")["roc_auc"].mean()
        s1c = r18[r18.variant == "physical_covered_rows"].groupby("fold")["roc_auc"].mean()
        print("\nSanity check: A1(covered) here vs Step 18 'physical_covered_rows' ROC-AUC difference per fold:",
              (a1c - s1c).abs().round(6).to_dict())
        a3 = r18[r18.variant == "physical_plus_network"].rename(columns={"variant": "arm"}).assign(arm="A3_agent_14_features")
        both = pd.concat([res[["arm", "fold", "seed", "roc_auc", "recall_at_5pct_fpr"]],
                          a3[["arm", "fold", "seed", "roc_auc", "recall_at_5pct_fpr"]]])
        print("\nD) WHAT THE AGENT'S 14-FEATURE DESIGN ADDS OVER THE MINIMAL NETWORK BASELINE "
              "(A3 minus A2; informational, no verdict)")
        display(deltas("A3_agent_14_features", "A2_physical_plus_minimal_network", both, "A3 - A2"))
    else:
        print("\n(Step 18 file has no XGBoost fused rows; comparison D skipped.)")
else:
    print("\n(Step 18 results not found: run Step 18 later to add comparison D, or rerun this cell then.)")
print("\nStep 10b complete (XGBoost). LSTM and Qwen versions of the minimal baseline come in the final test. attack_4 was not used.")

phy_norm: 3,425 seconds with packets (Step 14a file (same definition))
phy_att_1: 189 seconds with packets (Step 14a file (same definition))
phy_att_2: 2,096 seconds with packets (Step 14a file (same definition))
phy_att_3: 1,181 seconds with packets (Step 14a file (same definition))
Share of physical rows with network data: {'phy_norm': 0.9991, 'phy_att_1': 0.0781, 'phy_att_2': 0.9962, 'phy_att_3': 0.9418}
done: A1_physical_all_rows
done: A1_physical_covered_rows
done: A2_physical_plus_minimal_network

A) PER FOLD (mean over seeds; XGBoost argmax rule for precision/recall/specificity)


,arm,fold,n_test,attack_share,precision,recall,specificity,roc_auc,pr_auc,recall_at_5pct_fpr,cyber_recall,macro_f1
0,A1_physical_all_rows,phy_att_1,2420.0,0.335,0.486,0.144,0.924,0.646,0.482,0.112,0.154,0.343
1,A1_physical_all_rows,phy_att_2,2104.0,0.145,0.461,0.446,0.911,0.725,0.362,0.367,0.321,0.585
2,A1_physical_all_rows,phy_att_3,1254.0,0.274,0.279,0.143,0.864,0.559,0.290,0.011,0.061,0.345
3,A1_physical_covered_rows,phy_att_1,189.0,0.259,0.000,0.000,1.000,0.540,0.398,0.347,0.000,0.426
4,A1_physical_covered_rows,phy_att_2,2096.0,0.146,0.643,0.356,0.966,0.709,0.402,0.407,0.000,0.529
5,A1_physical_covered_rows,phy_att_3,1181.0,0.282,0.667,0.033,1.000,0.647,0.556,0.305,0.000,0.323
6,A2_physical_plus_minimal_network,phy_att_1,189.0,0.259,0.000,0.000,0.993,0.687,0.430,0.177,0.000,0.424
7,A2_physical_plus_minimal_network,phy_att_2,2096.0,0.146,0.425,0.373,0.914,0.745,0.349,0.277,0.016,0.528
8,A2_physical_plus_minimal_network,phy_att_3,1181.0,0.282,1.000,0.189,1.000,0.769,0.723,0.526,0.144,0.501



B) AVERAGE OVER THE THREE FOLDS


,precision,recall,specificity,roc_auc,pr_auc,recall_at_5pct_fpr,cyber_recall,macro_f1
arm,,,,,,,,
A1_physical_all_rows,0.409,0.244,0.899,0.643,0.378,0.163,0.179,0.424
A1_physical_covered_rows,0.437,0.130,0.989,0.632,0.452,0.353,0.000,0.426
A2_physical_plus_minimal_network,0.475,0.187,0.969,0.734,0.501,0.326,0.053,0.484



C) WHAT THE NETWORK DATA ADDS WITHOUT ANY AGENT (A2 minus A1 on the same covered rows; informational, no verdict)


,comparison,fold,metric,delta,lower_arm_seed_sd,delta_above_sd
0,A2 - A1(covered),phy_att_1,roc_auc,0.1472,0.0160,True
1,A2 - A1(covered),phy_att_1,recall_at_5pct_fpr,-0.1701,0.0000,False
2,A2 - A1(covered),phy_att_2,roc_auc,0.0358,0.0167,True
3,A2 - A1(covered),phy_att_2,recall_at_5pct_fpr,-0.1307,0.0154,False
4,A2 - A1(covered),phy_att_3,roc_auc,0.1225,0.0146,True
5,A2 - A1(covered),phy_att_3,recall_at_5pct_fpr,0.2202,0.0063,True



Sanity check: A1(covered) here vs Step 18 'physical_covered_rows' ROC-AUC difference per fold: {'phy_att_1': 0.0, 'phy_att_2': 0.0, 'phy_att_3': 0.0}

D) WHAT THE AGENT'S 14-FEATURE DESIGN ADDS OVER THE MINIMAL NETWORK BASELINE (A3 minus A2; informational, no verdict)


,comparison,fold,metric,delta,lower_arm_seed_sd,delta_above_sd
0,A3 - A2,phy_att_1,roc_auc,0.0033,0.0187,False
1,A3 - A2,phy_att_1,recall_at_5pct_fpr,0.1769,0.0236,True
2,A3 - A2,phy_att_2,roc_auc,0.1080,0.0193,True
3,A3 - A2,phy_att_2,recall_at_5pct_fpr,0.1536,0.0311,True
4,A3 - A2,phy_att_3,roc_auc,0.0339,0.0041,True
5,A3 - A2,phy_att_3,recall_at_5pct_fpr,0.0460,0.0104,True



Step 10b complete (XGBoost). LSTM and Qwen versions of the minimal baseline come in the final test. attack_4 was not used.


In [ ]:
# STEP 11 (v2): Qwen3 decision with verified thinking, two-stage output, repeated sampling

import json
import re
import numpy as np
import pandas as pd
from collections import Counter

for v in ["sep", "res", "auc_df", "sub_df", "network_labels_full_3",
          "physical_multiclass", "client", "MODEL_ID", "DEV_FOLDS", "WORK_DIR"]:
    assert v in globals(), f"Missing {v}: run the earlier steps first."
assert "phy_att_4" not in DEV_FOLDS

# ---------------- Settings ----------------
N_SAMPLES = 3
ANALYSIS_MAX_TOKENS = 16000
FORMAT_MAX_TOKENS = 3000
TEMPERATURE = 0.6        # recommended sampling for thinking mode
TOP_P = 0.95
PRINT_ANALYSIS = True

ALLOWED_METRICS = ["roc_auc_attack_vs_normal", "pr_auc_attack_vs_normal",
                   "subtype_detection_rate"]
ALLOWED_DECISIONS = ["preprocessing_change", "diagnostic",
                     "evaluation_design", "no_change_justified"]

def to_json(obj):
    return json.dumps(
        obj, indent=2,
        default=lambda o: o.item() if hasattr(o, "item") else str(o)
    )

# ---------------- Evidence ----------------
recall_cols = [c for c in res.columns if c.startswith("recall_")]

baseline = {
    "features": "40 raw physical variables, no preprocessing",
    "argmax_results_by_fold": (
        res.groupby(["fold", "phy_norm_in_train"])[["macro_f1"] + recall_cols]
        .mean().round(3).reset_index().to_dict("records")),
    "ranking_by_fold": (
        auc_df.groupby(["fold", "phy_norm_in_train"])[["base_rate", "roc_auc", "pr_auc"]]
        .mean().round(3).reset_index().to_dict("records")),
    "seed_std_of_ranking": (
        auc_df.groupby(["fold", "phy_norm_in_train"])[["roc_auc", "pr_auc"]]
        .std().round(4).reset_index().to_dict("records")),
    "subtype_detection": (
        sub_df.groupby(["fold", "phy_norm_in_train", "subtype"])
        .agg(n=("n", "first"),
             detected_as_attack=("detected_as_attack", "mean"),
             correct_type=("correct_type", "mean"))
        .round(3).reset_index().to_dict("records")),
}

sep_dict = {
    a: {b: (None if pd.isna(v) else round(float(v), 3)) for b, v in row.items()}
    for a, row in sep.iterrows()
}

net_counts = (
    network_labels_full_3
    .pivot_table(index="acquisition", columns="label", values="count",
                 aggfunc="sum", fill_value=0).astype(int)
)

phys_counts = (
    pd.concat([d.assign(acq=n)[["acq", "target_class"]]
               for n, d in physical_multiclass.items()])
    .groupby(["acq", "target_class"]).size().unstack(fill_value=0)
)

evidence_11 = {
    "fixed_by_researchers": {
        "project_goal": (
            "Design ONE model-agnostic preprocessing plan, then evaluate it with "
            "three downstream models on identical preprocessed data and splits: "
            "XGBoost, an LSTM, and Qwen. The reference baseline is XGBoost on "
            "the raw physical variables with no preprocessing (shown below)."
        ),
        "model_agnostic_rule": (
            "The plan defines a shared canonical representation. Per-model "
            "adapters (flat table for XGBoost, windowed tensor for the LSTM, "
            "text serialization for Qwen) are fixed by the researchers and are "
            "not tuned by the agent. A decision must be justified by data-level "
            "evidence that does not depend on one model's behavior."
        ),
        "downstream_models": {
            "xgboost": "tabular input; class weights fitted inside each training fold",
            "lstm": "sequence input; windows must not cross acquisition boundaries; scaling fitted on training folds only",
            "qwen": "text-serialized input; context length and token cost limit how many observations and features it can see",
        },
        "evidence_limits": "Baseline numbers come from XGBoost only. No LSTM or Qwen results exist yet.",
        "target": ("three classes: normal, physical_fault, cyberattack "
                   "(MITM, DoS and scan merged into cyberattack); raw subtype "
                   "labels are kept for error analysis only"),
        "development_protocol": "leave-one-acquisition-out over phy_att_1, phy_att_2, phy_att_3",
        "final_test": "phy_att_4: not accessible, never used to choose anything",
        "decision_rule": "argmax over class probabilities; thresholds are not tuned",
        "phy_norm_status": "unresolved; baseline is shown with and without it",
    },
    "glossary": {
        "phy_norm": ("the name of the dedicated NORMAL-OPERATION ACQUISITION "
                     "(3428 observations, all normal). It is not a normalization "
                     "or scaling method."),
        "phy_norm_in_train": "whether that normal-operation acquisition was included in training",
        "network coverage": ("whether an aggregated network record exists within the "
                             "alignment tolerance of a physical observation"),
    },
    "network_coverage_by_acquisition": {
        "phy_norm":  {"observations": 3428, "with_network": 3425, "without_network": 3},
        "phy_att_1": {"observations": 2420, "with_network": 199,  "without_network": 2221},
        "phy_att_2": {"observations": 2104, "with_network": 2097, "without_network": 7},
        "phy_att_3": {"observations": 1254, "with_network": 1191, "without_network": 63},
    },
    "network_missingness_as_attack_predictor_roc_auc": {
        "pooled_over_phy_norm_phy_att_1_phy_att_2": 0.7277,
        "within_phy_att_1_only": 0.506,
        "within_phy_att_2_only": 0.498,
        "note": ("computed from the binary-experiment coverage table; the pooled "
                 "value mixes acquisitions with different coverage and attack rates"),
    },
    "evidence_standard": ("Statistics pooled across acquisitions are not evidence of "
                          "generalization. A signal counts only if it holds within "
                          "individual acquisitions."),
    "acceptance_rule": (
        "A change is accepted only if the primary metric improves over the raw "
        "baseline in ALL three development folds by more than the baseline seed "
        "standard deviation for that fold. Seed spread understates split variance."
    ),
    "baseline_raw_physical_only": baseline,
    "normal_row_session_separability_roc_auc": sep_dict,
    "physical_observation_counts_by_class": phys_counts.to_dict("index"),
    "network_packet_label_counts_by_file": net_counts.to_dict("index"),
    "verified_facts": [
        "phy_att_1 has 2420 observations; only 199 have network data within the "
        "alignment tolerance (139 normal, 60 attack); 2221 have none (1471 normal, "
        "750 attack). The attack_1 network recording is genuinely short.",
        "In the binary experiment, across phy_norm, phy_att_1 and phy_att_2, rows "
        "without network data were 33.6% attack (2231 rows) versus 6.4% for rows "
        "with network data (5721 rows). Network missingness alone gave ROC-AUC "
        "0.7277 as an attack predictor.",
        "attack_4 network labels contain no MITM or physical fault packets, although "
        "phy_att_4 has 265 MITM and 133 physical fault observations.",
        "attack_1 network labels contain no physical fault packets, although "
        "phy_att_1 has 277 physical fault observations.",
        "One physical observation corresponds to many network packets; packet "
        "counts and observation counts must not be compared.",
        "Network anomaly packets (230 in total) have unspecified type.",
    ],
}
evidence_text = to_json(evidence_11)

# ---------------- Stage 1: analysis prompt (thinking on) ----------------
prompt_analysis = f"""
You are the independent preprocessing engineering agent for the WDT
physical-network classification research project.

Review the following verified evidence:

{evidence_text}

Work through the following sections in order. Be explicit and quantitative.

A. OBSERVATIONS: the five most decision-relevant facts in the evidence, each
   with its exact numbers. Re-compute any arithmetic you rely on.
B. UNKNOWNS: what the evidence cannot tell you, and which claims would be
   unsupported guesses.
C. CANDIDATES: at least three different next decisions, one of which must be
   "no_change_justified". For each: the problem it addresses, evidence for,
   evidence against, and cost.
D. ELIMINATION: check each candidate against the rules below and eliminate any
   that break one.
E. DECISION: choose ONE surviving candidate. State the decision_type (one of
   {ALLOWED_DECISIONS}), the decision, the hypothesis, the procedure as concrete
   implementable steps, the primary_metric (one of {ALLOWED_METRICS}), the
   folds, a numeric falsification criterion, why it is model-agnostic, and the
   limitations.

Rules:
- Use only the evidence above. Do not invent numbers or facts.
- Do not use or request phy_att_4.
- Folds must be whole acquisitions from the development list.
- Any fitting or weighting must use training folds only.
- Preserve acquisition boundaries and temporal causality.
- Do not repeat the raw baseline.
- Distinguish observed facts from hypotheses.

- The plan must serve XGBoost, an LSTM and Qwen alike; do not tune it to one of them.
- Plain ASCII text only.
"""

messages_analysis = [
    {"role": "system", "content": (
        "You are an independent preprocessing research agent. Base every "
        "claim on the supplied evidence. Check your arithmetic. Prefer a "
        "justified no-change decision over an unsupported one.")},
    {"role": "user", "content": prompt_analysis},
]

def format_prompt(analysis_text):
    return f"""/no_think
Below is a written engineering analysis. Convert ONLY its final decision
(section E) into one JSON object. Do not add, remove or reinterpret anything.
Copy numbers exactly. Plain ASCII only. Return the JSON object and nothing else.

Keys:
  "diagnosis": string,
  "decision_type": one of {ALLOWED_DECISIONS},
  "decision": string,
  "hypothesis": string,
  "procedure": list of strings (empty list only if decision_type is no_change_justified),
  "primary_metric": one of {ALLOWED_METRICS},
  "folds": list of fold names,
  "falsification": string with the numeric criterion,
  "model_agnostic_justification": string,
  "limitations": list of strings

ANALYSIS:
{analysis_text}
"""

# ---------------- Helpers ----------------
def stream_chat(messages, max_tokens, temperature, top_p):
    content_parts, reasoning_parts, finish = [], [], None
    try:
        stream = client.chat.completions.create(
            model=MODEL_ID, messages=messages, tool_choice="none",
            temperature=temperature, top_p=top_p,
            max_tokens=max_tokens, stream=True)
        for chunk in stream:
            if not chunk.choices:
                continue
            choice = chunk.choices[0]
            delta = choice.delta
            c = getattr(delta, "content", None)
            r = (getattr(delta, "reasoning", None)
                 or getattr(delta, "reasoning_content", None))
            if c:
                content_parts.append(c)
            if r:
                reasoning_parts.append(r)
            if getattr(choice, "finish_reason", None):
                finish = choice.finish_reason
    except Exception as e:
        finish = f"error:{type(e).__name__}"
        print("Stream interrupted:", str(e)[:200])
    return "".join(content_parts), "".join(reasoning_parts), finish

def parse_json(text):
    s, e = text.find("{"), text.rfind("}")
    if s == -1 or e <= s:
        return None
    try:
        return json.loads(text[s:e + 1])
    except json.JSONDecodeError:
        return None

def validate(p, raw):
    problems, warnings = [], []
    required = ["diagnosis", "decision_type", "decision", "hypothesis",
                "procedure", "primary_metric", "folds", "falsification",
                "model_agnostic_justification", "limitations"]
    missing = [k for k in required if k not in p]
    if missing:
        problems.append(f"missing keys: {missing}")
    if not raw.replace("\u2019", "'").replace("\u2018", "'").isascii():
        problems.append("non-ASCII characters in reply")
    if p.get("decision_type") not in ALLOWED_DECISIONS:
        problems.append("decision_type not in allowed list")
    if p.get("primary_metric") not in ALLOWED_METRICS:
        problems.append("primary_metric not in allowed list")
    folds = p.get("folds")
    if not isinstance(folds, list) or not folds or not set(folds) <= set(DEV_FOLDS):
        problems.append("folds must be a non-empty subset of the development folds")
    proc = p.get("procedure")
    if p.get("decision_type") != "no_change_justified":
        if (not isinstance(proc, list) or not proc
                or any(len(str(s)) < 25 for s in proc)):
            problems.append("procedure must be specific, non-empty steps")
    if not re.search(r"\d", str(p.get("falsification", ""))):
        problems.append("falsification contains no numeric criterion")

    blob = json.dumps(p).lower()
    if "phy_att_4" in blob:
        warnings.append("mentions phy_att_4: confirm it appears only as a prohibition")
    if "threshold" in blob:
        warnings.append("mentions thresholds: the decision rule is fixed at argmax")
    if any(w in blob for w in ["smote", "oversampl", "synthetic"]):
        warnings.append("resampling or synthetic data: check for leakage")
    if any(w in blob for w in ["k-fold", "kfold", "stratif", "random split"]):
        warnings.append("row-level splitting terms: folds must be whole acquisitions")
    if any(w in blob for w in ["n_estimators", "max_depth", "learning_rate",
                               "epochs", "hidden_size"]):
        warnings.append("model-specific hyperparameters: the plan must stay model-agnostic")

    claims = f"{p.get('diagnosis', '')} {p.get('hypothesis', '')}"
    odd = sorted({n for n in re.findall(r"\d+\.\d+|\d{3,}", claims)
                  if n not in evidence_text})
    if odd:
        warnings.append(f"numbers in diagnosis/hypothesis not found verbatim in evidence: {odd}")
    return problems, warnings

# ---------------- Run samples ----------------
(WORK_DIR / "step11_evidence.json").write_text(evidence_text, encoding="utf-8")
(WORK_DIR / "step11_prompt.txt").write_text(prompt_analysis, encoding="utf-8")

samples = []
for i in range(1, N_SAMPLES + 1):
    print("\n" + "=" * 60)
    print(f"SAMPLE {i} of {N_SAMPLES}: analysis (thinking on)")
    print("=" * 60)

    content, reasoning, finish = stream_chat(
        messages_analysis, ANALYSIS_MAX_TOKENS, TEMPERATURE, TOP_P)
    inline = re.findall(r"<think>(.*?)</think>", content, flags=re.DOTALL)
    analysis = re.sub(r"<think>.*?</think>", "", content, flags=re.DOTALL).strip()
    reasoning_chars = len(reasoning) + sum(len(x) for x in inline)

    print("Finish reason:", finish)
    print("Reasoning characters:", reasoning_chars,
          "(0 means thinking was NOT active)")
    print("Analysis characters:", len(analysis))
    if PRINT_ANALYSIS:
        print("\n" + analysis)

    (WORK_DIR / f"step11_sample{i}_analysis.txt").write_text(analysis, encoding="utf-8")
    (WORK_DIR / f"step11_sample{i}_reasoning.txt").write_text(
        reasoning + "\n" + "\n".join(inline), encoding="utf-8")

    proposal, problems, warnings = None, [], []
    if finish != "stop" or not analysis:
        problems.append(f"analysis incomplete (finish={finish}); raise ANALYSIS_MAX_TOKENS and rerun")
    else:
        fmt_messages = [
            {"role": "system", "content": (
                "You convert a written engineering decision into JSON. "
                "You never add, remove or change content.")},
            {"role": "user", "content": format_prompt(analysis)},
        ]
        fmt_content, _, fmt_finish = stream_chat(
            fmt_messages, FORMAT_MAX_TOKENS, 0.0, 1.0)
        fmt_clean = re.sub(r"<think>.*?</think>", "", fmt_content,
                           flags=re.DOTALL).strip()
        proposal = parse_json(fmt_clean)
        if proposal is None:
            problems.append("no parseable JSON from the formatting call")
        else:
            problems, warnings = validate(proposal, fmt_clean)
            (WORK_DIR / f"step11_sample{i}_proposal.json").write_text(
                to_json(proposal), encoding="utf-8")

    (WORK_DIR / f"step11_sample{i}_validation.json").write_text(
        to_json({"problems": problems, "warnings": warnings}), encoding="utf-8")

    print("\nVALIDATION  problems:", problems if problems else "none")
    print("VALIDATION  warnings:", warnings if warnings else "none")

    samples.append({"i": i, "finish": finish, "reasoning_chars": reasoning_chars,
                    "proposal": proposal, "problems": problems,
                    "warnings": warnings})

# ---------------- Comparison ----------------
rows = []
for s in samples:
    p = s["proposal"] or {}
    rows.append({
        "sample": s["i"],
        "finish": s["finish"],
        "reasoning_chars": s["reasoning_chars"],
        "decision_type": p.get("decision_type"),
        "primary_metric": p.get("primary_metric"),
        "problems": len(s["problems"]),
        "warnings": len(s["warnings"]),
        "decision": str(p.get("decision", ""))[:110],
    })

print("\n" + "=" * 60)
print("COMPARISON ACROSS SAMPLES")
print("=" * 60)
display(pd.DataFrame(rows))
print("decision_type counts:",
      dict(Counter(r["decision_type"] for r in rows if r["decision_type"])))
print("\nIf the samples disagree, the evidence does not settle the question.")
print("Human review is required before executing anything.")
print("\nStep 11 complete. Everything saved to", WORK_DIR)


SAMPLE 1 of 3: analysis (thinking on)
Finish reason: stop
Reasoning characters: 5405 (0 means thinking was NOT active)
Analysis characters: 5028

A. **OBSERVATIONS**  
1. **Network missingness as an attack predictor**: Pooled ROC-AUC across phy_norm, phy_att_1, and phy_att_2 is **0.7277**, but within phy_att_1 it is **0.506** (chance level) and within phy_att_2 it is **0.498** (chance level). This indicates the signal is not generalizable within individual acquisitions.  
2. **Normal vs. attack separability**: For phy_att_1, the ROC-AUC between phy_norm and phy_att_1 is **0.829**, and between phy_att_2 and phy_att_3 it is **0.949–0.981**. This suggests physical observations in normal vs. attack acquisitions are highly separable.  
3. **Baseline performance inconsistency**: Including phy_norm in training improves macro F1 in phy_att_2 (from 0.472 to 0.585) but reduces it in phy_att_1 (from 0.41 to 0.343). This inconsistency violates the acceptance rule requiring improvement in *all* fo

,sample,finish,reasoning_chars,decision_type,primary_metric,problems,warnings,decision
0,1,stop,5405,no_change_justified,roc_auc_attack_vs_normal,1,0,Retain raw physical variables as the canonical...
1,2,stop,6926,no_change_justified,roc_auc_attack_vs_normal,1,0,Retain raw physical variables with no preproce...
2,3,stop,4774,no_change_justified,roc_auc_attack_vs_normal,1,0,Maintain raw physical variables as the canonic...


decision_type counts: {'no_change_justified': 3}

If the samples disagree, the evidence does not settle the question.
Human review is required before executing anything.

Step 11 complete. Everything saved to C:\WDT\agentic_multiclass


In [23]:
# STEP 12: Data-level profile of the 40 physical variables (development data only)

import itertools, json, warnings
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
for v in ["physical_multiclass", "feature_cols", "WORK_DIR"]:
    assert v in globals(), f"Missing {v}: run Steps 5 and 9 first."

PROFILE_ACQ = ["phy_norm", "phy_att_1", "phy_att_2", "phy_att_3"]   # phy_att_4 excluded on purpose
ATTACK_ACQ = ["phy_att_1", "phy_att_2", "phy_att_3"]
assert "phy_att_4" not in PROFILE_ACQ

all_dev = pd.concat(
    [physical_multiclass[n].assign(acq=n) for n in PROFILE_ACQ], ignore_index=True)
X = all_dev[feature_cols]

# 1) Time structure
time_rows = []
for n in PROFILE_ACQ:
    t = pd.to_datetime(physical_multiclass[n]["Time"], errors="coerce")
    dt = t.diff().dt.total_seconds().dropna()
    med = float(dt.median()) if len(dt) else np.nan
    time_rows.append({
        "acquisition": n, "rows": int(len(t)),
        "unparsed_times": int(t.isna().sum()),
        "monotonic": bool(t.dropna().is_monotonic_increasing),
        "median_step_seconds": med,
        "gaps_over_2x_median": int((dt > 2 * med).sum()) if len(dt) else 0,
        "duplicate_timestamps": int(t.duplicated().sum()),
    })
print("TIME STRUCTURE")
display(pd.DataFrame(time_rows))

# 2) Per-feature profile
prof = pd.DataFrame(index=feature_cols)
prof["missing"] = X.isna().sum()
prof["n_unique"] = X.nunique()
prof["std"] = X.std()
prof["min"] = X.min()
prof["max"] = X.max()
prof["kind"] = np.select(
    [prof["n_unique"] <= 1, prof["n_unique"] == 2, prof["n_unique"] <= 10],
    ["constant", "binary", "discrete"], "continuous")

def outlier_share(s):
    s = s.dropna()
    med = s.median()
    mad = (s - med).abs().median()
    return float(((s - med).abs() > 5 * 1.4826 * mad).mean()) if mad > 0 else np.nan
prof["outlier_share_5mad"] = [outlier_share(X[c]) for c in feature_cols]

def lag1(s):
    s = s.dropna()
    return float(s.autocorr(lag=1)) if len(s) > 3 and s.std() > 0 else np.nan
prof["lag1_autocorr"] = [
    np.nanmean([lag1(physical_multiclass[n][c]) for n in PROFILE_ACQ])
    for c in feature_cols]

# 3) Session shift on normal rows only
normal = {n: physical_multiclass[n][
    physical_multiclass[n]["target_class"].astype(str) == "normal"]
    for n in PROFILE_ACQ}

def max_shift(c):
    stats = [(normal[n][c].mean(), normal[n][c].std()) for n in PROFILE_ACQ]
    best = 0.0
    for (m1, s1), (m2, s2) in itertools.combinations(stats, 2):
        pooled = np.sqrt((s1 ** 2 + s2 ** 2) / 2)
        if pooled > 0:
            best = max(best, abs(m1 - m2) / pooled)
    return best
prof["max_session_shift_sd"] = [max_shift(c) for c in feature_cols]

# 4) Within-acquisition single-feature attack-vs-normal AUC
for n in ATTACK_ACQ:
    d = physical_multiclass[n]
    y = (d["target_class"].astype(str) != "normal").to_numpy()
    aucs = []
    for c in feature_cols:
        v = d[c].to_numpy(dtype=float)
        ok = ~np.isnan(v)
        aucs.append(roc_auc_score(y[ok], v[ok])
                    if len(np.unique(v[ok])) > 1 and len(np.unique(y[ok])) > 1
                    else np.nan)
    prof[f"auc_{n}"] = aucs

auc_cols = [f"auc_{n}" for n in ATTACK_ACQ]
prof["auc_same_direction"] = (
    (prof[auc_cols] > 0.5).all(axis=1) | (prof[auc_cols] < 0.5).all(axis=1))
prof["auc_min_strength"] = (prof[auc_cols] - 0.5).abs().min(axis=1)

# 5) Redundancy
corr = X.corr().abs()
pairs = [(a, b, round(float(corr.loc[a, b]), 3))
         for a, b in itertools.combinations(feature_cols, 2)
         if corr.loc[a, b] > 0.98]

# ---- Summary
print("\nFEATURE KINDS:", prof["kind"].value_counts().to_dict())
print("Features with missing values:", int((prof["missing"] > 0).sum()))
print("Highly correlated pairs (|r| > 0.98):", len(pairs))

print("\nSTRONGEST CONSISTENT WITHIN-ACQUISITION SIGNALS (same direction in all 3 attack acquisitions)")
display(prof[prof["auc_same_direction"]]
        .sort_values("auc_min_strength", ascending=False)
        [auc_cols + ["auc_min_strength", "kind"]].head(10).round(3))

print("\nLARGEST SESSION SHIFT (normal rows, in pooled SDs)")
display(prof.sort_values("max_session_shift_sd", ascending=False)
        [["max_session_shift_sd", "kind", "lag1_autocorr"]].head(10).round(3))

print("\nHIGHEST OUTLIER SHARE")
display(prof.sort_values("outlier_share_5mad", ascending=False)
        [["outlier_share_5mad", "kind", "min", "max"]].head(8).round(4))

records = json.loads(prof.round(3).rename_axis("feature").reset_index().to_json(orient="records"))
profile_evidence = {
    "scope": "development acquisitions only: phy_norm, phy_att_1, phy_att_2, phy_att_3",
    "time_structure": time_rows,
    "feature_kinds": prof["kind"].value_counts().to_dict(),
    "per_feature": records,
    "highly_correlated_pairs_abs_r_over_0.98": pairs,
    "column_notes": {
        "max_session_shift_sd": "largest difference in mean between any two acquisitions' NORMAL rows, in pooled standard deviations",
        "auc_phy_att_k": "single-feature ROC-AUC, attack vs normal, within that acquisition only",
        "auc_same_direction": "feature points the same way in all three attack acquisitions",
    },
}
(WORK_DIR / "step12_data_profile.json").write_text(
    json.dumps(profile_evidence, indent=2), encoding="utf-8")
print("\nSaved:", WORK_DIR / "step12_data_profile.json")
print("Step 12 complete. phy_att_4 was not used.")

TIME STRUCTURE


,acquisition,rows,unparsed_times,monotonic,median_step_seconds,gaps_over_2x_median,duplicate_timestamps
0,phy_norm,3428,0,True,1.0,0,0
1,phy_att_1,2420,0,True,1.0,0,0
2,phy_att_2,2104,0,True,1.0,0,0
3,phy_att_3,1254,0,True,1.0,0,0



FEATURE KINDS: {'binary': 16, 'constant': 14, 'continuous': 9, 'discrete': 1}
Features with missing values: 0
Highly correlated pairs (|r| > 0.98): 7

STRONGEST CONSISTENT WITHIN-ACQUISITION SIGNALS (same direction in all 3 attack acquisitions)


,auc_phy_att_1,auc_phy_att_2,auc_phy_att_3,auc_min_strength,kind
Tank_1,0.553,0.611,0.601,0.053,continuous
Tank_2,0.665,0.536,0.650,0.036,continuous
Valv_18,0.542,0.532,0.610,0.032,binary
Pump_1,0.605,0.537,0.519,0.019,binary
Valv_17,0.515,0.528,0.594,0.015,binary
Valv_20,0.473,0.420,0.487,0.013,binary



LARGEST SESSION SHIFT (normal rows, in pooled SDs)


,max_session_shift_sd,kind,lag1_autocorr
Tank_2,2.066,continuous,0.999
Valv_17,1.503,binary,0.989
Valv_14,1.229,binary,0.980
Valv_13,1.227,binary,0.980
Valv_15,1.227,binary,0.980
Tank_4,1.177,continuous,0.999
Tank_3,1.053,continuous,0.999
Valv_10,0.974,binary,0.979
Valv_11,0.973,binary,0.979
Valv_12,0.971,binary,0.979



HIGHEST OUTLIER SHARE


,outlier_share_5mad,kind,min,max
Tank_4,0.3796,continuous,0,1143
Tank_3,0.0075,continuous,0,3427
Tank_2,0.0000,continuous,0,2012
Tank_1,0.0000,continuous,0,2021
Tank_5,0.0000,continuous,0,838
Tank_6,0.0000,continuous,0,800
Tank_7,0.0000,continuous,0,812
Tank_8,0.0000,continuous,0,507



Saved: C:\WDT\agentic_multiclass\step12_data_profile.json
Step 12 complete. phy_att_4 was not used.


In [24]:
# STEP 12b: Network packet profile (development acquisitions only; attack_4 excluded)

import json
import time
from collections import Counter

import numpy as np
import pandas as pd

for v in ["network_files", "physical_multiclass", "WORK_DIR"]:
    assert v in globals(), f"Missing {v}: run Steps 1 and 5 first."

# network file stem -> matching physical acquisition (attack_4 is excluded on purpose)
PAIRS = {"normal": "phy_norm", "attack_1": "phy_att_1",
         "attack_2": "phy_att_2", "attack_3": "phy_att_3"}
assert "attack_4" not in PAIRS

CHUNK = 500_000
MAX_CHUNKS_PER_FILE = None        # set to 2 for a quick dry run (results are then partial)
TOLERANCES = [0, 1, 2, 5, 10]     # seconds

EPOCH = pd.Timestamp("1970-01-01")
ONE_S = pd.Timedelta("1s")
rng = np.random.default_rng(0)

def to_epoch_seconds(ts):
    return ((ts.dt.floor("s") - EPOCH) // ONE_S).to_numpy()

def top(counter, k=8):
    total = sum(counter.values()) or 1
    return [{"value": str(v), "count": int(c), "share": round(c / total, 4)}
            for v, c in counter.most_common(k)]

by_stem = {p.stem: p for p in network_files}
profiles = {}

for stem, phys_name in PAIRS.items():
    path = by_stem.get(stem)
    if path is None:
        print(f"{stem}: file not found, skipped")
        continue

    header = pd.read_csv(path, nrows=0, encoding="utf-8-sig")
    colmap = {str(c).strip().lower(): c for c in header.columns}
    need = ["time", "ip_s", "ip_d", "sport", "dport", "proto", "flags", "size",
            "modbus_fn", "n_pkt_src", "n_pkt_dst", "modbus_response", "label"]
    optional = ["mac_s", "mac_d"]
    missing = [c for c in need if c not in colmap]
    if missing:
        print(f"{stem}: missing columns {missing}; actual: {list(header.columns)}")
        continue
    use = [colmap[c] for c in need + [o for o in optional if o in colmap]]

    t0 = time.time()
    rows = bad_time = 0
    t_min = t_max = None
    sec_counts, sec_label = Counter(), Counter()
    cats = {k: Counter() for k in ["proto", "flags", "modbus_fn", "sport", "dport",
                                   "ip_s", "ip_d", "mac_s", "mac_d", "resp"]}
    num = {k: {"n": 0, "sum": 0.0, "sumsq": 0.0, "min": np.inf, "max": -np.inf,
               "zero": 0, "sample": []} for k in ["size", "n_pkt_src", "n_pkt_dst"]}
    resp_null = 0

    reader = pd.read_csv(path, encoding="utf-8-sig", usecols=use,
                         chunksize=CHUNK, low_memory=False)
    for ci, chunk in enumerate(reader):
        if MAX_CHUNKS_PER_FILE is not None and ci >= MAX_CHUNKS_PER_FILE:
            break
        chunk.columns = [str(c).strip().lower() for c in chunk.columns]
        rows += len(chunk)

        ts = pd.to_datetime(chunk["time"], format="ISO8601", errors="coerce")
        ok = ts.notna()
        bad_time += int((~ok).sum())
        chunk, ts = chunk[ok], ts[ok]
        if chunk.empty:
            continue
        t_min = ts.min() if t_min is None else min(t_min, ts.min())
        t_max = ts.max() if t_max is None else max(t_max, ts.max())

        sec = to_epoch_seconds(ts)
        u, c = np.unique(sec, return_counts=True)
        for s_, n_ in zip(u, c):
            sec_counts[int(s_)] += int(n_)
        labels = chunk["label"].astype(str).to_numpy()
        g = pd.DataFrame({"sec": sec, "label": labels}).groupby(["sec", "label"]).size()
        for (s_, l_), n_ in g.items():
            sec_label[(int(s_), l_)] += int(n_)

        for k in ["proto", "flags", "modbus_fn", "sport", "dport", "ip_s", "ip_d"]:
            cats[k].update(chunk[k].astype(str).value_counts().to_dict())
        for k in ["mac_s", "mac_d"]:
            if k in chunk.columns:
                cats[k].update(chunk[k].astype(str).value_counts().to_dict())
        resp = chunk["modbus_response"]
        resp_null += int(resp.isna().sum())
        cats["resp"].update(resp.dropna().astype(str).value_counts().head(50).to_dict())

        for k in num:
            v = pd.to_numeric(chunk[k], errors="coerce").dropna().to_numpy(dtype=float)
            if len(v) == 0:
                continue
            d = num[k]
            d["n"] += len(v)
            d["sum"] += float(v.sum())
            d["sumsq"] += float((v ** 2).sum())
            d["min"] = min(d["min"], float(v.min()))
            d["max"] = max(d["max"], float(v.max()))
            d["zero"] += int((v == 0).sum())
            d["sample"].append(rng.choice(v, size=min(2000, len(v)), replace=False))

    # ---- per-second summaries
    secs = np.array(sorted(sec_counts))
    counts = np.array([sec_counts[s] for s in secs])
    span = int(secs.max() - secs.min() + 1) if len(secs) else 0
    per_sec_labels = {}
    for (s_, l_), n_ in sec_label.items():
        per_sec_labels.setdefault(s_, Counter())[l_] += n_
    dominant, mixed_normal_attack, multi_attack = Counter(), 0, 0
    for s_, cnt in per_sec_labels.items():
        dominant[cnt.most_common(1)[0][0]] += 1
        non_normal = [l for l in cnt if l != "normal"]
        if non_normal and "normal" in cnt:
            mixed_normal_attack += 1
        if len(non_normal) > 1:
            multi_attack += 1

    # ---- numeric summaries
    numsum = {}
    for k, d in num.items():
        if d["n"] == 0:
            continue
        mean = d["sum"] / d["n"]
        var = max(d["sumsq"] / d["n"] - mean ** 2, 0.0)
        smp = np.concatenate(d["sample"])
        numsum[k] = {"mean": round(mean, 3), "std": round(var ** 0.5, 3),
                     "min": d["min"], "p50": float(np.percentile(smp, 50)),
                     "p95": float(np.percentile(smp, 95)), "max": d["max"],
                     "share_zero": round(d["zero"] / d["n"], 4)}

    # ---- alignment against the matching physical acquisition
    pt = pd.to_datetime(physical_multiclass[phys_name]["Time"], errors="coerce")
    p_sec = to_epoch_seconds(pt)
    align = {"physical_rows": int(len(p_sec)),
             "physical_start": str(pt.min()), "physical_end": str(pt.max())}
    if len(secs):
        idx = np.searchsorted(secs, p_sec)
        left = np.where(idx > 0, p_sec - secs[np.clip(idx - 1, 0, len(secs) - 1)], np.inf)
        right = np.where(idx < len(secs), secs[np.clip(idx, 0, len(secs) - 1)] - p_sec, np.inf)
        dist = np.minimum(left, right)
        align["nearest_network_second_coverage"] = {
            f"within_{k}s": round(float((dist <= k).mean()), 4) for k in TOLERANCES}
        align["network_start_minus_physical_start_seconds"] = int(secs.min() - p_sec.min())
        align["network_end_minus_physical_end_seconds"] = int(secs.max() - p_sec.max())

    profiles[stem] = {
        "matching_physical_acquisition": phys_name,
        "packets": int(rows),
        "unparsed_times": int(bad_time),
        "network_start": str(t_min), "network_end": str(t_max),
        "seconds_with_packets": int(len(secs)),
        "empty_seconds_inside_span": int(span - len(secs)),
        "packets_per_second": ({"median": float(np.median(counts)),
                                "p05": float(np.percentile(counts, 5)),
                                "p95": float(np.percentile(counts, 95)),
                                "max": int(counts.max())} if len(counts) else {}),
        "label_by_second": {
            "dominant_label_seconds": {k: int(v) for k, v in dominant.items()},
            "seconds_mixing_normal_and_attack": int(mixed_normal_attack),
            "seconds_with_more_than_one_attack_label": int(multi_attack)},
        "numeric_fields": numsum,
        "modbus_response_null_share": round(resp_null / max(rows, 1), 4),
        "categorical_fields": {
            "proto": top(cats["proto"]), "flags": top(cats["flags"]),
            "modbus_fn": top(cats["modbus_fn"], 10),
            "modbus_response_top_values_approx": top(cats["resp"], 6),
            "distinct": {k: len(cats[k]) for k in
                         ["sport", "dport", "ip_s", "ip_d", "mac_s", "mac_d"]},
            "top_dport": top(cats["dport"], 5), "top_sport": top(cats["sport"], 5)},
        "alignment_with_physical": align,
    }
    flag = " (PARTIAL: dry run)" if MAX_CHUNKS_PER_FILE else ""
    print(f"{stem}: {rows:,} packets scanned in {time.time() - t0:.0f}s{flag}")

# ---------------- Summary ----------------
ov = pd.DataFrame([{
    "file": s, "packets": p["packets"], "start": p["network_start"][:19], "end": p["network_end"][:19],
    "sec_with_pkts": p["seconds_with_packets"], "empty_sec": p["empty_seconds_inside_span"],
    "pps_median": p["packets_per_second"].get("median"), "pps_p95": p["packets_per_second"].get("p95"),
    "pps_max": p["packets_per_second"].get("max")} for s, p in profiles.items()])
print("\nOVERVIEW"); display(ov)

al = pd.DataFrame([{
    "file": s, "phys_rows": p["alignment_with_physical"]["physical_rows"],
    **p["alignment_with_physical"].get("nearest_network_second_coverage", {}),
    "net_start_minus_phys_start_s": p["alignment_with_physical"].get("network_start_minus_physical_start_seconds"),
    "net_end_minus_phys_end_s": p["alignment_with_physical"].get("network_end_minus_physical_end_seconds")}
    for s, p in profiles.items()])
print("\nALIGNMENT WITH PHYSICAL TIMELINE (share of physical rows with a network second within N s)")
display(al)

lb = pd.DataFrame([{
    "file": s, "mixed_normal_attack_sec": p["label_by_second"]["seconds_mixing_normal_and_attack"],
    "multi_attack_label_sec": p["label_by_second"]["seconds_with_more_than_one_attack_label"],
    **{f"dom_{k}": v for k, v in p["label_by_second"]["dominant_label_seconds"].items()}}
    for s, p in profiles.items()]).fillna(0)
print("\nLABELS PER SECOND (dominant label counts; mixed = both normal and attack packets in one second)")
display(lb)

fs = pd.DataFrame([{
    "file": s, **{f"size_{k}": v for k, v in p["numeric_fields"].get("size", {}).items()
                  if k in ("mean", "std", "p95", "max")},
    "n_pkt_src_max": p["numeric_fields"].get("n_pkt_src", {}).get("max"),
    "n_pkt_dst_max": p["numeric_fields"].get("n_pkt_dst", {}).get("max"),
    "resp_null": p["modbus_response_null_share"],
    **{f"distinct_{k}": v for k, v in p["categorical_fields"]["distinct"].items()}}
    for s, p in profiles.items()])
print("\nFIELD SUMMARY"); display(fs)

first = next(iter(profiles.values()))
print("\nTOP modbus_fn (first file):", [(d["value"], d["share"]) for d in first["categorical_fields"]["modbus_fn"][:6]])
print("TOP proto (first file):", [(d["value"], d["share"]) for d in first["categorical_fields"]["proto"][:4]])

out = WORK_DIR / "step12b_network_profile.json"
out.write_text(json.dumps({
    "scope": "development network files only: normal, attack_1, attack_2, attack_3",
    "per_file": profiles,
    "column_notes": {
        "alignment_with_physical": "share of physical observations whose nearest network second is within N seconds",
        "label_by_second": "network labels are per packet; this shows how labels look when packets are grouped by second",
        "modbus_response_top_values_approx": "approximate: top values are taken per chunk",
    }}, indent=2, default=str), encoding="utf-8")
print("\nSaved:", out)
print("Step 12b complete. attack_4 was not read.")

normal: 7,757,289 packets scanned in 23s
attack_1: 430,776 packets scanned in 1s
attack_2: 5,159,469 packets scanned in 14s
attack_3: 5,701,973 packets scanned in 13s

OVERVIEW


,file,packets,start,end,sec_with_pkts,empty_sec,pps_median,pps_p95,pps_max
0,normal,7757289,2021-04-09 11:30:52,2021-04-09 12:27:56,3425,0,2268.0,2293.0,2308
1,attack_1,430776,2021-04-09 18:23:28,2021-04-09 18:26:36,189,0,2290.0,2342.6,2414
2,attack_2,5159469,2021-04-19 15:37:19,2021-04-19 16:12:14,2096,0,2277.0,2323.0,12209
3,attack_3,5701973,2021-04-09 19:42:13,2021-04-09 20:01:53,1181,0,2276.0,7868.0,70014



ALIGNMENT WITH PHYSICAL TIMELINE (share of physical rows with a network second within N s)


,file,phys_rows,within_0s,within_1s,within_2s,within_5s,within_10s,net_start_minus_phys_start_s,net_end_minus_phys_end_s
0,normal,3428,0.9991,0.9997,1.0000,1.0000,1.0000,2,-1
1,attack_1,2420,0.0781,0.0785,0.0789,0.0802,0.0822,0,-2231
2,attack_2,2104,0.9962,0.9971,0.9981,0.9990,1.0000,7,-2
3,attack_3,1254,0.9418,0.9434,0.9442,0.9466,0.9506,1,-72



LABELS PER SECOND (dominant label counts; mixed = both normal and attack packets in one second)


,file,mixed_normal_attack_sec,multi_attack_label_sec,dom_normal,dom_MITM,dom_DoS,dom_physical fault
0,normal,0,0,3425,0.0,0.0,0.0
1,attack_1,7,1,141,48.0,0.0,0.0
2,attack_2,32,6,1800,95.0,79.0,122.0
3,attack_3,21,5,854,101.0,76.0,150.0



FIELD SUMMARY


,file,size_mean,size_std,size_p95,size_max,n_pkt_src_max,n_pkt_dst_max,resp_null,distinct_sport,distinct_dport,distinct_ip_s,distinct_ip_d,distinct_mac_s,distinct_mac_d
0,normal,65.326,1.390,66.0,78.0,51.0,52.0,0.5135,10261,10261,7,7,7,7
1,attack_1,65.329,1.422,66.0,78.0,51.0,52.0,0.5145,1020,1020,7,7,8,9
2,attack_2,64.970,1.948,66.0,82.0,97.0,94.0,0.5523,25046,10765,8,8,8,9
3,attack_3,795.439,723.384,1514.0,1514.0,100.0,100.0,0.7753,5405,5397,8,8,9,9



TOP modbus_fn (first file): [('Read Holding Registers Response', 0.2659), ('Read Holding Registers', 0.2658), ('Read Coils Request', 0.2342), ('Read Coils Response', 0.2342)]
TOP proto (first file): [('Modbus', 0.9729), ('TCP', 0.0271)]

Saved: C:\WDT\agentic_multiclass\step12b_network_profile.json
Step 12b complete. attack_4 was not read.


In [21]:
import pandas as pd

for name in ["phy_norm", "phy_att_1", "phy_att_2", "phy_att_3"]:
    raw = physical_multiclass[name]["Time"]
    print(f"\n=== {name} ===")
    print("raw head:", raw.head(2).tolist())
    print("raw tail:", raw.tail(2).tolist())
    for label, kw in [("month-first", dict(dayfirst=False)), ("day-first", dict(dayfirst=True))]:
        t = pd.to_datetime(raw, errors="coerce", **kw)
        print(f"{label:12s} {t.min()}  ->  {t.max()}   NaT={t.isna().sum()}")


=== phy_norm ===
raw head: ['09/04/2021 11:30:50', '09/04/2021 11:30:51']
raw tail: ['09/04/2021 12:27:56', '09/04/2021 12:27:57']
month-first  2021-09-04 11:30:50  ->  2021-09-04 12:27:57   NaT=0
day-first    2021-04-09 11:30:50  ->  2021-04-09 12:27:57   NaT=0

=== phy_att_1 ===
raw head: ['09/04/2021 18:23:28', '09/04/2021 18:23:29']
raw tail: ['09/04/2021 19:03:46', '09/04/2021 19:03:47']
month-first  2021-09-04 18:23:28  ->  2021-09-04 19:03:47   NaT=0
day-first    2021-04-09 18:23:28  ->  2021-04-09 19:03:47   NaT=0

=== phy_att_2 ===
raw head: ['19/04/2021 15:37:12', '19/04/2021 15:37:13']
raw tail: ['19/04/2021 16:12:15', '19/04/2021 16:12:16']
month-first  2021-04-19 15:37:12  ->  2021-04-19 16:12:16   NaT=0
day-first    2021-04-19 15:37:12  ->  2021-04-19 16:12:16   NaT=0

=== phy_att_3 ===
raw head: ['09/04/2021 19:42:12', '09/04/2021 19:42:13']
raw tail: ['09/04/2021 20:03:04', '09/04/2021 20:03:05']
month-first  2021-09-04 19:42:12  ->  2021-09-04 20:03:05   NaT=0
day-fir

In [19]:
pd.read_csv(network_files[0], nrows=5, encoding="utf-8-sig").T

,0,1,2,3,4
Time,2021-04-09 18:23:28.385003,2021-04-09 18:23:28.385005,2021-04-09 18:23:28.385006,2021-04-09 18:23:28.385484,2021-04-09 18:23:28.385486
mac_s,74:46:a0:bd:a7:1b,74:46:a0:bd:a7:1b,74:46:a0:bd:a7:1b,0a:fe:ec:47:74:fb,fa:00:bc:90:d7:fa
mac_d,0a:fe:ec:47:74:fb,e6:3f:ac:c9:a8:8c,fa:00:bc:90:d7:fa,74:46:a0:bd:a7:1b,74:46:a0:bd:a7:1b
ip_s,84.3.251.20,84.3.251.20,84.3.251.20,84.3.251.102,84.3.251.103
ip_d,84.3.251.102,84.3.251.101,84.3.251.103,84.3.251.20,84.3.251.20
sport,56667,56666,56668,502,502
dport,502,502,502,56667,56668
proto,Modbus,Modbus,Modbus,Modbus,Modbus
flags,11000,11000,11000,11000,11000
size,66,66,66,64,64


In [25]:
# STEP 13 (v2): Qwen3 designs a model-agnostic preprocessing PLAN from data-level evidence
# (analysis -> critic -> revision -> JSON plan -> validation)

import json
import re
import pandas as pd
from collections import Counter

for v in ["client", "MODEL_ID", "DEV_FOLDS", "WORK_DIR"]:
    assert v in globals(), f"Missing {v}: run the earlier steps first."
assert "phy_att_4" not in DEV_FOLDS

# ---------------- Settings ----------------
N_SAMPLES = 2
USE_CRITIC = True
ANALYSIS_MAX_TOKENS = 16000
CRITIC_MAX_TOKENS = 12000
FORMAT_MAX_TOKENS = 6000
TEMPERATURE = 0.6
TOP_P = 0.95
PRINT_TEXT = True

COMPONENTS = ["cleaning", "alignment_and_fusion", "scaling", "windowing",
              "feature_construction", "class_handling", "normal_session_handling"]
ALLOWED_METRICS = ["roc_auc_attack_vs_normal", "pr_auc_attack_vs_normal",
                   "subtype_detection_rate", "macro_f1_argmax"]

# ---------------- Helpers ----------------
def to_json(obj):
    return json.dumps(
        obj, indent=2,
        default=lambda o: o.item() if hasattr(o, "item") else str(o))

def ascii_fix(s):
    for a, b in {"\u2019": "'", "\u2018": "'", "\u201c": '"', "\u201d": '"',
                 "\u2013": "-", "\u2014": "-", "\u00d7": "x", "\u2265": ">=",
                 "\u2264": "<=", "\u2192": "->", "\u00a0": " "}.items():
        s = s.replace(a, b)
    return s

def stream_chat(messages, max_tokens, temperature, top_p):
    content_parts, reasoning_parts, finish = [], [], None
    try:
        stream = client.chat.completions.create(
            model=MODEL_ID, messages=messages, tool_choice="none",
            temperature=temperature, top_p=top_p,
            max_tokens=max_tokens, stream=True)
        for chunk in stream:
            if not chunk.choices:
                continue
            choice = chunk.choices[0]
            delta = choice.delta
            c = getattr(delta, "content", None)
            r = (getattr(delta, "reasoning", None)
                 or getattr(delta, "reasoning_content", None))
            if c:
                content_parts.append(c)
            if r:
                reasoning_parts.append(r)
            if getattr(choice, "finish_reason", None):
                finish = choice.finish_reason
    except Exception as e:
        finish = f"error:{type(e).__name__}"
        print("Stream interrupted:", str(e)[:200])
    content = "".join(content_parts)
    inline = re.findall(r"<think>(.*?)</think>", content, flags=re.DOTALL)
    clean = re.sub(r"<think>.*?</think>", "", content, flags=re.DOTALL).strip()
    reasoning = "".join(reasoning_parts) + "\n".join(inline)
    return ascii_fix(clean), reasoning, finish

def parse_json(text):
    s, e = text.find("{"), text.rfind("}")
    if s == -1 or e <= s:
        return None
    try:
        return json.loads(text[s:e + 1])
    except json.JSONDecodeError:
        return None

# ---------------- Evidence ----------------
evid_path = WORK_DIR / "step11_evidence.json"
prof_path = WORK_DIR / "step12_data_profile.json"
net_path = WORK_DIR / "step12b_network_profile.json"
assert evid_path.exists() and prof_path.exists() and net_path.exists(), \
    "Run Steps 11, 12 and 12b first."

# Guard: physical Time must have been re-parsed day-first (patch cell) before Steps 12 and 12b.
assert pd.api.types.is_datetime64_any_dtype(physical_multiclass["phy_att_1"]["Time"]), \
    "Run the dayfirst patch cell, then rerun Steps 12 and 12b."

SEALED = re.compile(r"att(ack)?_?4", re.IGNORECASE)

def scrub_sealed(obj, path="", log=None):
    """Remove every key/list item/string that refers to the sealed final-test acquisition."""
    log = [] if log is None else log
    if isinstance(obj, dict):
        out = {}
        for k, v in obj.items():
            if SEALED.search(str(k)):
                log.append(f"key {path}/{k}")
                continue
            if isinstance(v, str) and SEALED.search(v):
                log.append(f"value {path}/{k}")
                continue
            out[k] = scrub_sealed(v, f"{path}/{k}", log)[0]
        return out, log
    if isinstance(obj, list):
        out = []
        for j, v in enumerate(obj):
            if isinstance(v, str) and SEALED.search(v):
                log.append(f"item {path}[{j}]")
                continue
            out.append(scrub_sealed(v, f"{path}[{j}]", log)[0])
        return out, log
    return obj, log

evidence_13, scrub_log = scrub_sealed(
    json.loads(evid_path.read_text(encoding="utf-8")))
profile = json.loads(prof_path.read_text(encoding="utf-8"))
net_profile = json.loads(net_path.read_text(encoding="utf-8"))
assert "phy_att_4" not in profile["scope"]
print("Removed from the Step 11 evidence (sealed test acquisition):")
for s in scrub_log:
    print("  -", s)
if not scrub_log:
    print("  (nothing found)")

by_kind = {}
for r in profile["per_feature"]:
    by_kind.setdefault(r["kind"], []).append(r["feature"])
profile["features_by_kind"] = by_kind

evidence_13.pop("acceptance_rule", None)
evidence_13["fixed_by_researchers"]["plan_deliverable"] = (
    "ONE plan with an explicit decision (apply or no_change) for each of seven "
    "components: " + ", ".join(COMPONENTS) + ".")
evidence_13["how_results_will_be_judged"] = (
    "After a plan is implemented and measured, a component is kept only if it "
    "improves the primary metric over the raw XGBoost baseline in ALL three "
    "development folds by more than that fold's baseline seed standard deviation. "
    "This rule applies AFTER testing. It is NOT a criterion for choosing what to "
    "propose: a component is proposed for testing when data-level evidence "
    "suggests it could help.")
evidence_13["data_profile_physical_variables"] = profile
evidence_13["network_packet_profile_development_acquisitions"] = net_profile
evidence_13["timestamp_alignment_physical_vs_network"] = {
    "note": ("Physical Time is DD/MM/YYYY and was parsed day-first. Network time is ISO. "
             "Physical rows are 1 per second. Share of physical rows having a network "
             "second within N seconds, measured in Step 12b."),
    "by_acquisition": {
        "phy_norm":  {"physical_rows": 3428, "within_0s": 0.9991, "within_2s": 1.0000},
        "phy_att_1": {"physical_rows": 2420, "within_0s": 0.0781, "within_2s": 0.0789,
                      "network_seconds_with_packets": 189,
                      "network_end_minus_physical_end_s": -2231},
        "phy_att_2": {"physical_rows": 2104, "within_0s": 0.9962, "within_2s": 0.9981},
        "phy_att_3": {"physical_rows": 1254, "within_0s": 0.9418, "within_2s": 0.9442,
                      "network_end_minus_physical_end_s": -72},
    },
    "mixed_label_seconds": {"phy_norm": 0, "phy_att_1": 7, "phy_att_2": 32, "phy_att_3": 21},
}
evidence_text = to_json(evidence_13)
leaks = [m for m in re.findall(r"[^\n]*att(?:ack)?_?4[^\n]*", evidence_text, flags=re.I)]
assert not leaks, f"Sealed acquisition still referenced in evidence: {leaks[:3]}"
ev_numbers = set(re.findall(r"\d+\.\d+|\d{3,}", evidence_text))
(WORK_DIR / "step13_evidence.json").write_text(evidence_text, encoding="utf-8")
print(f"Evidence size: {len(evidence_text):,} characters (about {len(evidence_text)//3:,} tokens)")

# ---------------- Prompts ----------------
prompt_analysis = f"""
You are the independent preprocessing engineering agent for the WDT
physical-network classification research project.

Review the following verified evidence:

{evidence_text}

Design ONE model-agnostic preprocessing plan. Work through these sections in
order. Be explicit and quantitative.

A. OBSERVATIONS: the 6 to 8 most decision-relevant data-level facts, each with
   exact numbers and the evidence key they come from.
B. UNKNOWNS: what the evidence cannot tell you.
C. COMPONENTS: for EACH of {COMPONENTS}: the options you considered (always
   including no_change), evidence for and against each, and your choice.
D. CROSS-CHECK: (1) does each chosen step work for XGBoost, an LSTM and Qwen?
   (2) leakage: is everything fitted on training folds only, causal, and
   inside acquisition boundaries? (3) do any components conflict?
E. FINAL PLAN: for each component: decision (apply or no_change), a concrete
   implementable specification, the evidence with exact numbers, and for each
   apply: the metric (one of {ALLOWED_METRICS}), the folds, and a numeric
   falsification criterion. A no_change component must still say what evidence
   would justify a change.

Rules:
- Use only the evidence above. Do not invent numbers or facts. Re-check arithmetic.
- Do not use or request phy_att_4.
- Folds are whole acquisitions from the development list.
- Anything fitted (scalers, weights) uses training folds only.
- Preserve acquisition boundaries and temporal causality: windows never cross
  acquisitions and use only past observations.
- Never use acquisition or session identity as a feature.
- Statistics pooled across acquisitions are not evidence of generalization.
- Network features are different from the network-availability flag. Judge
  them separately.
- The network data are individual packets; the physical data are one row per
  second. Any use of network data needs a rule that turns packets into
  per-second observations (alignment_and_fusion). Use the network profile.
- The three development acquisitions do not contain the same attack types or
  network behaviour. Check the network profile per acquisition before you
  assume a pattern learned from two of them will hold in the third.
- Where network data do not cover a physical row, say how that row is handled
  and state which acquisitions are affected, using the alignment evidence.
- "phy_norm" is the dedicated normal-operation acquisition, not a method.
- The judging rule is applied after testing. Do not use it to decide what to propose.
- The plan must serve XGBoost, an LSTM and Qwen alike. Per-model adapters are
  fixed by the researchers and are not part of the plan.
- Plain ASCII text only.
"""

messages_analysis = [
    {"role": "system", "content": (
        "You are an independent preprocessing research agent. Base every claim on "
        "the supplied evidence and check your arithmetic.")},
    {"role": "user", "content": prompt_analysis},
]

def critic_prompt(analysis):
    return f"""
You are a strict reviewer. Below are verified evidence and an engineering
analysis written by another agent.

List every claim in the analysis that is (a) contradicted by the evidence,
(b) not supported by the evidence, or (c) a misapplied rule. Examples of (c):
using the post-testing judging rule to decide what to propose; treating a
pooled statistic as within-acquisition evidence; confusing the
network-availability flag with network features; reading phy_norm as a
normalization method; using acquisition identity as a feature.
Check every number in the analysis against the evidence. Do not judge whether
the plan is good, only whether it is factually and logically consistent.

Return one JSON object with keys "verdict" ("ok" or "revise") and "issues"
(a list; each item is an object with keys "claim", "problem", "evidence_says").
Plain ASCII only.

EVIDENCE:
{evidence_text}

ANALYSIS:
{analysis}
"""

def revision_prompt(analysis, issues):
    return f"""
Below are verified evidence, your earlier analysis, and a reviewer's list of
problems. Rewrite the COMPLETE analysis (sections A to E) fixing exactly those
problems: correct wrong numbers, remove unsupported claims, and redo any
decision that depended on them. Keep the same rules as before. Plain ASCII only.

EVIDENCE:
{evidence_text}

EARLIER ANALYSIS:
{analysis}

REVIEWER PROBLEMS:
{to_json(issues)}
"""

def format_prompt(analysis):
    return f"""/no_think
Below is a written engineering analysis. Convert ONLY its section E (FINAL PLAN)
into one JSON object. Do not add, remove or reinterpret anything. Copy numbers
exactly. Plain ASCII only. Return the JSON object and nothing else.

The object has keys "summary" (string), "components" (list) and "risks" (list of strings).
Each item of "components" is an object with keys:
  "component": one of {COMPONENTS},
  "decision": "apply" or "no_change",
  "specification": string,
  "evidence": list of strings that each cite exact numbers,
  "test": an object with keys "metric" (one of {ALLOWED_METRICS}), "folds"
          (list of fold names) and "falsification" (string with a numeric
          criterion); use null for a no_change component,

Formatting rules:
- "folds" must list fold names exactly as in {DEV_FOLDS}. Never write "all".
- For a no_change component, "specification" must be one or more full sentences
  saying what stays unchanged and why, and "evidence" must cite at least one
  number from the analysis.
  "model_agnostic_note": string.

ANALYSIS:
{analysis}
"""

# ---------------- Validation ----------------
def validate_plan(p, raw):
    problems, warnings = [], []
    for k in ["summary", "components", "risks"]:
        if k not in p:
            problems.append(f"missing key: {k}")
    if not ascii_fix(raw).isascii():
        problems.append("non-ASCII characters in reply")
    comps = p.get("components")
    if not isinstance(comps, list) or not comps:
        problems.append("components must be a non-empty list")
        comps = []
    names = [c.get("component") for c in comps if isinstance(c, dict)]
    missing = [n for n in COMPONENTS if n not in names]
    extra = [n for n in names if n not in COMPONENTS]
    dup = [n for n, k in Counter(names).items() if k > 1]
    if missing:
        problems.append(f"components missing: {missing}")
    if extra:
        problems.append(f"unknown components: {extra}")
    if dup:
        problems.append(f"duplicate components: {dup}")

    n_apply, odd_numbers = 0, set()
    for c in comps:
        if not isinstance(c, dict):
            problems.append("component entry is not an object")
            continue
        name = c.get("component", "?")
        dec = c.get("decision")
        if dec not in ("apply", "no_change"):
            problems.append(f"{name}: decision must be apply or no_change")
            continue
        ev = c.get("evidence")
        need_num = dec == "apply"
        if (not isinstance(ev, list) or not ev
                or (need_num and any(not re.search(r"\d", str(e)) for e in ev))):
            problems.append(f"{name}: evidence must be a non-empty list"
                            + (" citing numbers" if need_num else ""))
        else:
            for e in ev:
                odd_numbers |= {n for n in re.findall(r"\d+\.\d+|\d{3,}", str(e))
                                if n not in ev_numbers}
        if len(str(c.get("specification", ""))) < (40 if dec == "apply" else 25):
            problems.append(f"{name}: specification too short to implement")
        if dec == "apply":
            n_apply += 1
            t = c.get("test")
            if not isinstance(t, dict):
                problems.append(f"{name}: apply needs a test object")
            else:
                if t.get("metric") not in ALLOWED_METRICS:
                    problems.append(f"{name}: metric not in allowed list")
                f = t.get("folds")
                if isinstance(f, list) and any(
                        re.fullmatch(r"(?i)all( development)?( folds)?", str(x).strip())
                        for x in f):
                    f = list(DEV_FOLDS)
                    t["folds"] = f
                    warnings.append(f"{name}: 'all folds' expanded to the development folds")
                if not isinstance(f, list) or not f or not set(f) <= set(DEV_FOLDS):
                    problems.append(f"{name}: folds must be a subset of the development folds")
                if not re.search(r"\d", str(t.get("falsification", ""))):
                    problems.append(f"{name}: falsification has no numeric criterion")

    if n_apply == 0:
        warnings.append("every component is no_change: the plan equals the raw baseline")
    blob = json.dumps(p).lower()
    if "phy_att_4" in blob:
        warnings.append("mentions phy_att_4: confirm it appears only as a prohibition")
    if any(w in blob for w in ["smote", "oversampl", "synthetic"]):
        warnings.append("resampling or synthetic data: check for leakage")
    if any(w in blob for w in ["k-fold", "kfold", "stratif", "random split"]):
        warnings.append("row-level splitting terms: folds must be whole acquisitions")
    if any(w in blob for w in ["n_estimators", "max_depth", "learning_rate",
                               "epochs", "hidden_size"]):
        warnings.append("model-specific hyperparameters: keep the plan model-agnostic")
    if any(w in blob for w in ["acquisition id", "session id", "session_id",
                               "session-based", "acquisition_id"]):
        warnings.append("acquisition or session identity as a feature")
    if re.search(r"has_network|network[_ ](data[_ ])?(present|presence|missing|availab)", blob):
        warnings.append("mentions a network-availability flag (confirmed coverage shortcut)")
    if odd_numbers:
        warnings.append("numbers in evidence items not found verbatim in the evidence: "
                        + str(sorted(odd_numbers)[:8]))
    return problems, warnings, n_apply

# ---------------- Run ----------------
(WORK_DIR / "step13_prompt.txt").write_text(prompt_analysis, encoding="utf-8")

results = []
for i in range(1, N_SAMPLES + 1):
    print("\n" + "=" * 60)
    print(f"SAMPLE {i} of {N_SAMPLES}")
    print("=" * 60)

    # 1) analysis
    analysis, reasoning, finish = stream_chat(
        messages_analysis, ANALYSIS_MAX_TOKENS, TEMPERATURE, TOP_P)
    print(f"[analysis] finish={finish} reasoning_chars={len(reasoning)} text_chars={len(analysis)}")
    (WORK_DIR / f"step13_sample{i}_analysis.txt").write_text(analysis, encoding="utf-8")
    (WORK_DIR / f"step13_sample{i}_reasoning.txt").write_text(reasoning, encoding="utf-8")

    problems, warnings, n_apply, issues, plan = [], [], 0, [], None
    final_analysis = analysis
    if finish != "stop" or not analysis:
        problems.append(f"analysis incomplete (finish={finish}); raise ANALYSIS_MAX_TOKENS and rerun")
    else:
        # 2) critic and revision
        if USE_CRITIC:
            critic_text, _, cfinish = stream_chat(
                [{"role": "system", "content": "You are a strict, literal fact-checker."},
                 {"role": "user", "content": critic_prompt(analysis)}],
                CRITIC_MAX_TOKENS, TEMPERATURE, TOP_P)
            critic = parse_json(critic_text) or {}
            issues = critic.get("issues", []) if isinstance(critic, dict) else []
            print(f"[critic] finish={cfinish} verdict={critic.get('verdict')} issues={len(issues)}")
            (WORK_DIR / f"step13_sample{i}_critic.json").write_text(
                to_json(critic), encoding="utf-8")
            if issues:
                if PRINT_TEXT:
                    for it in issues:
                        print("  - ", str(it)[:300])
                revised, _, rfinish = stream_chat(
                    [{"role": "system", "content": (
                        "You are an independent preprocessing research agent. "
                        "Base every claim on the supplied evidence.")},
                     {"role": "user", "content": revision_prompt(analysis, issues)}],
                    ANALYSIS_MAX_TOKENS, TEMPERATURE, TOP_P)
                print(f"[revision] finish={rfinish} text_chars={len(revised)}")
                if rfinish == "stop" and revised:
                    final_analysis = revised
                    (WORK_DIR / f"step13_sample{i}_revised.txt").write_text(
                        revised, encoding="utf-8")
                else:
                    problems.append("revision incomplete; the unrevised analysis was used")

        if PRINT_TEXT:
            print("\n" + final_analysis)

        # 3) JSON plan
        fmt_text, _, ffinish = stream_chat(
            [{"role": "system", "content": (
                "You convert a written engineering plan into JSON. You never add, "
                "remove or change content.")},
             {"role": "user", "content": format_prompt(final_analysis)}],
            FORMAT_MAX_TOKENS, 0.0, 1.0)
        plan = parse_json(fmt_text)
        if plan is None:
            problems.append(f"no parseable JSON from the formatting call (finish={ffinish})")
        else:
            p2, w2, n_apply = validate_plan(plan, fmt_text)
            problems += p2
            warnings += w2
            (WORK_DIR / f"step13_sample{i}_plan.json").write_text(to_json(plan), encoding="utf-8")

    (WORK_DIR / f"step13_sample{i}_validation.json").write_text(
        to_json({"problems": problems, "warnings": warnings}), encoding="utf-8")
    print("\nVALIDATION problems:", problems if problems else "none")
    print("VALIDATION warnings:", warnings if warnings else "none")
    results.append({"i": i, "plan": plan, "problems": problems,
                    "warnings": warnings, "n_apply": n_apply, "issues": len(issues)})

# ---------------- Comparison ----------------
rows = []
for r in results:
    comps = (r["plan"] or {}).get("components", []) if r["plan"] else []
    applied = [c.get("component") for c in comps
               if isinstance(c, dict) and c.get("decision") == "apply"]
    rows.append({"sample": r["i"], "critic_issues": r["issues"],
                 "components_applied": ", ".join(applied) or "none",
                 "problems": len(r["problems"]), "warnings": len(r["warnings"])})

print("\n" + "=" * 60)
print("COMPARISON ACROSS SAMPLES")
print("=" * 60)
display(pd.DataFrame(rows))
print("Disagreement between samples shows where the evidence does not settle the question.")
print("Human review is required before executing anything.")
print("\nStep 13 complete. Everything saved to", WORK_DIR)

Removed from the Step 11 evidence (sealed test acquisition):
  - value /fixed_by_researchers/final_test
  - key /normal_row_session_separability_roc_auc/phy_norm/phy_att_4
  - key /normal_row_session_separability_roc_auc/phy_att_1/phy_att_4
  - key /normal_row_session_separability_roc_auc/phy_att_2/phy_att_4
  - key /normal_row_session_separability_roc_auc/phy_att_3/phy_att_4
  - key /normal_row_session_separability_roc_auc/phy_att_4
  - key /physical_observation_counts_by_class/phy_att_4
  - key /network_packet_label_counts_by_file/attack_4
  - item /verified_facts[2]
Evidence size: 61,139 characters (about 20,379 tokens)

SAMPLE 1 of 2
[analysis] finish=stop reasoning_chars=16027 text_chars=6646
[critic] finish=stop verdict=ok issues=0

A. OBSERVATIONS:  
1. **Network Coverage Disparity**: phy_att_1 has only 199 (8%) physical observations with network data within alignment tolerance, while phy_att_2/3 have 99.6% and 94.2% respectively (from `network_coverage_by_acquisition` and `time

,sample,critic_issues,components_applied,problems,warnings
0,1,0,"alignment_and_fusion, windowing, feature_const...",1,2
1,2,6,"cleaning, alignment_and_fusion, scaling, windo...",5,1


Disagreement between samples shows where the evidence does not settle the question.
Human review is required before executing anything.

Step 13 complete. Everything saved to C:\WDT\agentic_multiclass


In [26]:
# STEP 14a: Freeze the tested plan + build per-second network features
# (development network files only: normal, attack_1, attack_2, attack_3; attack_4 never read; no labels used)

import json
import re
import time
from datetime import datetime

import numpy as np
import pandas as pd

for v in ["network_files", "physical_multiclass", "WORK_DIR", "DEV_FOLDS"]:
    assert v in globals(), f"Missing {v}: run the earlier steps first."
assert pd.api.types.is_datetime64_any_dtype(physical_multiclass["phy_att_1"]["Time"]), \
    "Run the dayfirst patch cell first."
assert "phy_att_4" not in DEV_FOLDS

# ---------------------------------------------------------------- frozen plan
FROZEN_PLAN = f"""# Step 14 frozen test plan
Written {datetime.now().isoformat(timespec='seconds')} BEFORE any component result was computed.

## Source
Qwen3-32B proposals: step13_sample1_plan.json / step13_sample2_plan.json (unedited, kept as they are).

## Human adjudication (each change tied to a rule)
| Qwen proposed | Tested instead | Rule |
|---|---|---|
| network_available / network_coverage flag | not used; no flag | evidence showed a coverage shortcut (within-acquisition AUC ~0.50) |
| network features only for phy_att_2/3 | one rule for every acquisition; compared on network-covered rows only | acquisition identity cannot be a rule; missing values would act as an implicit flag |
| 10-second non-overlapping windows | causal sliding window ending at each row (past 10 s only): rolling mean, rolling std, 1-step difference | keeps rows one-to-one with the baseline; no leakage across acquisitions |
| falsify if metric 'drops by x' on one fold | kept only if better than baseline by more than the baseline seed SD in ALL three folds | rule fixed by the researchers |
| cleaning: 14/16 constant + correlated pairs (arithmetic inconsistent) | drop features constant in the training fold; drop features with |r|>0.98 to an earlier kept feature (training fold) | arithmetic could not be reconciled; rule is computed, not copied |
| scaling: min-max (sample 2) / no change (sample 1) | min-max fitted on the training fold, tested alone | samples disagreed; test settles it |
| class_handling, normal_session_handling | no change (class weights fitted in training fold; phy_norm in training) | both samples agree |

## Fixed test design
- Folds: leave-one-acquisition-out over phy_att_1, phy_att_2, phy_att_3; phy_norm always in training; phy_att_4 untouched.
- Model: the same XGBoost as Step 9, seeds 0, 1, 2, class weights fitted inside the training fold.
- Primary metric: ROC-AUC attack vs normal (attack = physical_fault or cyberattack). Secondary: macro-F1 at argmax, PR-AUC.
- Verdict per component: KEEP only if (mean variant - mean baseline) > baseline seed SD and > 0 in all three folds. Otherwise DROP.
- Network components are compared against a baseline trained and tested on exactly the same network-covered rows.
- Network feature vocabulary (top Modbus functions) is chosen from pooled development packet counts, no labels.
"""
(WORK_DIR / "step14_frozen_plan.md").write_text(FROZEN_PLAN, encoding="utf-8")
print("Frozen plan saved:", WORK_DIR / "step14_frozen_plan.md")

# ---------------------------------------------------------------- feature builder
PAIRS = {"normal": "phy_norm", "attack_1": "phy_att_1",
         "attack_2": "phy_att_2", "attack_3": "phy_att_3"}
assert "attack_4" not in PAIRS
CHUNK = 500_000
MAX_CHUNKS_PER_FILE = None          # 2 for a dry run
TOP_FN = 8
EPOCH = pd.Timestamp("1970-01-01")
ONE_S = pd.Timedelta("1s")

def to_epoch_seconds(ts):
    return ((ts.dt.floor("s") - EPOCH) // ONE_S).to_numpy()

def clean_name(s):
    return re.sub(r"[^0-9a-zA-Z]+", "_", str(s)).strip("_").lower()

def scan(path):
    header = pd.read_csv(path, nrows=0, encoding="utf-8-sig")
    colmap = {str(c).strip().lower(): c for c in header.columns}
    need = ["time", "size", "modbus_fn", "modbus_response", "sport", "dport", "ip_s", "ip_d"]
    miss = [c for c in need if c not in colmap]
    assert not miss, f"{path.name}: missing columns {miss}"
    use = [colmap[c] for c in need]
    base, fn_counts, pairs = [], [], {k: [] for k in ["sport", "dport", "ip_s", "ip_d"]}
    rows = 0
    reader = pd.read_csv(path, encoding="utf-8-sig", usecols=use, chunksize=CHUNK, low_memory=False)
    for ci, ch in enumerate(reader):
        if MAX_CHUNKS_PER_FILE is not None and ci >= MAX_CHUNKS_PER_FILE:
            break
        ch.columns = [str(c).strip().lower() for c in ch.columns]
        ts = pd.to_datetime(ch["time"], format="ISO8601", errors="coerce")
        ok = ts.notna().to_numpy()
        ch, ts = ch[ok], ts[ok]
        if ch.empty:
            continue
        rows += len(ch)
        sec = to_epoch_seconds(ts)
        size = pd.to_numeric(ch["size"], errors="coerce").to_numpy(dtype=float)
        d = pd.DataFrame({"sec": sec, "size": size,
                          "resp_null": ch["modbus_response"].isna().to_numpy().astype(float)})
        d["size2"] = d["size"] ** 2
        g = d.groupby("sec").agg(n=("size", "size"), size_sum=("size", "sum"),
                                 size_sq=("size2", "sum"), size_max=("size", "max"),
                                 resp_null=("resp_null", "sum"))
        base.append(g)
        fn = ch["modbus_fn"].astype(str).to_numpy()
        fn_counts.append(pd.crosstab(sec, fn))
        for k in pairs:
            pairs[k].append(pd.DataFrame({"sec": sec, "v": ch[k].astype(str).to_numpy()}).drop_duplicates())
    b = pd.concat(base).groupby(level=0).agg(
        n=("n", "sum"), size_sum=("size_sum", "sum"), size_sq=("size_sq", "sum"),
        size_max=("size_max", "max"), resp_null=("resp_null", "sum"))
    f = pd.concat(fn_counts).groupby(level=0).sum().fillna(0)
    dist = {k: pd.concat(v).drop_duplicates().groupby("sec").size() for k, v in pairs.items()}
    return b, f, dist, rows

raw = {}
for stem, phys in PAIRS.items():
    path = {p.stem: p for p in network_files}.get(stem)
    assert path is not None, f"network file {stem} not found"
    t0 = time.time()
    raw[stem] = scan(path)
    print(f"{stem}: {raw[stem][3]:,} packets -> {len(raw[stem][0]):,} seconds in {time.time() - t0:.0f}s")

pooled_fn = sum((r[1].sum(axis=0) for r in raw.values()), start=pd.Series(dtype=float)).fillna(0)
top_fn = list(pooled_fn.sort_values(ascending=False).index[:TOP_FN])
print("Top Modbus functions kept:", top_fn)

NET_COLS = None
for stem, (b, f, dist, rows) in raw.items():
    out = pd.DataFrame(index=b.index)
    out["net_pkts"] = b["n"]
    mean = b["size_sum"] / b["n"]
    out["net_size_mean"] = mean
    out["net_size_std"] = np.sqrt(np.maximum(b["size_sq"] / b["n"] - mean ** 2, 0))
    out["net_size_max"] = b["size_max"]
    out["net_resp_null_share"] = b["resp_null"] / b["n"]
    for k, s in dist.items():
        out[f"net_distinct_{k}"] = s.reindex(out.index).fillna(0)
    f = f.reindex(out.index).fillna(0)
    tot = f.sum(axis=1).replace(0, np.nan)
    for fn_name in top_fn:
        out[f"net_fn_{clean_name(fn_name)}"] = (f[fn_name] / tot) if fn_name in f.columns else 0.0
    out["net_fn_other"] = 1.0 - sum(out[f"net_fn_{clean_name(x)}"] for x in top_fn)
    out.index.name = "sec"
    NET_COLS = list(out.columns)
    out.reset_index().to_csv(WORK_DIR / f"step14a_network_features_{PAIRS[stem]}.csv", index=False)

meta = {"top_modbus_fn": top_fn, "net_cols": NET_COLS, "scope": "normal, attack_1, attack_2, attack_3"}
(WORK_DIR / "step14a_network_features_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
print(f"\nSaved {len(NET_COLS)} per-second network features per acquisition:", NET_COLS)
print("Step 14a complete. attack_4 was not read; no labels were used.")

Frozen plan saved: C:\WDT\agentic_multiclass\step14_frozen_plan.md
normal: 7,757,289 packets -> 3,425 seconds in 26s
attack_1: 430,776 packets -> 189 seconds in 1s
attack_2: 5,159,469 packets -> 2,096 seconds in 11s
attack_3: 5,701,973 packets -> 1,181 seconds in 11s
Top Modbus functions kept: ['Read Coils Request', 'Read Coils Response', 'Read Holding Registers', 'Read Holding Registers Response']

Saved 14 per-second network features per acquisition: ['net_pkts', 'net_size_mean', 'net_size_std', 'net_size_max', 'net_resp_null_share', 'net_distinct_sport', 'net_distinct_dport', 'net_distinct_ip_s', 'net_distinct_ip_d', 'net_fn_read_coils_request', 'net_fn_read_coils_response', 'net_fn_read_holding_registers', 'net_fn_read_holding_registers_response', 'net_fn_other']
Step 14a complete. attack_4 was not read; no labels were used.


In [28]:
# STEP 14b: Test each plan component against the raw XGBoost baseline (leave-one-acquisition-out)

import json
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import average_precision_score, f1_score, recall_score, roc_auc_score
from sklearn.utils.class_weight import compute_sample_weight

for v in ["frames", "physical_multiclass", "feature_cols", "DEV_FOLDS", "SEEDS", "ALL_CLASSES", "WORK_DIR"]:
    assert v in globals(), f"Missing {v}: run Step 9 first."
assert (WORK_DIR / "step14_frozen_plan.md").exists(), "Run Step 14a first (it freezes the plan)."
assert "phy_att_4" not in DEV_FOLDS

EPOCH = pd.Timestamp("1970-01-01")
ONE_S = pd.Timedelta("1s")
WINDOW = 10
CORR_LIMIT = 0.98
PRIMARY = "roc_auc"

meta = json.loads((WORK_DIR / "step14a_network_features_meta.json").read_text(encoding="utf-8"))
NET_COLS = meta["net_cols"]

# ---------------------------------------------------------------- data preparation
data = {}
for name in ["phy_norm"] + DEV_FOLDS:
    d = frames[name].copy()
    d["Time"] = physical_multiclass[name]["Time"]      # day-first parsed (patch cell)
    assert pd.api.types.is_datetime64_any_dtype(d["Time"]), "Run the dayfirst patch cell."
    assert d["Time"].is_monotonic_increasing, f"{name}: rows not in time order"
    sec = ((d["Time"].dt.floor("s") - EPOCH) // ONE_S).to_numpy()
    net = pd.read_csv(WORK_DIR / f"step14a_network_features_{name}.csv").set_index("sec")
    nf = net.reindex(sec)
    nf.index = d.index
    # causal window features (past WINDOW rows only, inside this acquisition)
    X = d[feature_cols].astype(float)
    roll = X.rolling(WINDOW, min_periods=1)
    win = pd.concat([roll.mean().add_suffix("_rmean"),
                     roll.std().add_suffix("_rstd"),
                     X.diff().add_suffix("_diff1")], axis=1)
    d = pd.concat([d, nf, win], axis=1)
    d["covered"] = nf["net_pkts"].notna().to_numpy()
    data[name] = d
WIN_COLS = [c for c in data["phy_norm"].columns if c.endswith(("_rmean", "_rstd", "_diff1"))]
print("Covered share by acquisition:",
      {n: round(float(d["covered"].mean()), 4) for n, d in data.items()})

# ---------------------------------------------------------------- component logic (fitted on training rows only)
def cleaning_cols(train, cols):
    keep = [c for c in cols if train[c].nunique(dropna=True) > 1]
    corr = train[keep].astype(float).corr().abs().fillna(0)
    kept = []
    for c in keep:
        if all(corr.loc[c, k] <= CORR_LIMIT for k in kept):
            kept.append(c)
    return kept

def minmax(train, test, cols):
    tr, te = train[cols].astype(float), test[cols].astype(float)   # bool columns -> 0/1
    lo, hi = tr.min(), tr.max()
    span = (hi - lo).replace(0, 1.0)
    return (tr - lo) / span, (te - lo) / span

def run_variant(variant, held, seed):
    names = [n for n in DEV_FOLDS if n != held] + ["phy_norm"]
    train = pd.concat([data[n] for n in names])
    test = data[held]
    covered_only = variant in ("net_base", "net_feat", "plan_base", "plan_all")
    if covered_only:
        train, test = train[train["covered"]], test[test["covered"]]
    cols = list(feature_cols)
    if variant in ("cleaning", "plan_all"):
        cols = cleaning_cols(train, cols)
    if variant in ("window", "plan_all"):
        cols += WIN_COLS
    if variant in ("net_feat", "plan_all"):
        cols += NET_COLS
    Xtr, Xte = train[cols].astype(float), test[cols].astype(float)
    if variant == "scaling":
        Xtr, Xte = minmax(train, test, cols)
    y_tr_lab = train["y"].to_numpy()
    classes = sorted(set(y_tr_lab))
    idx = {c: i for i, c in enumerate(classes)}
    y_tr = np.array([idx[c] for c in y_tr_lab])
    model = xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                              subsample=0.8, colsample_bytree=0.8, tree_method="hist",
                              n_jobs=-1, random_state=seed)
    model.fit(Xtr, y_tr, sample_weight=compute_sample_weight("balanced", y_tr))
    proba = model.predict_proba(Xte)
    pred = np.array(classes)[proba.argmax(axis=1)]
    truth = test["y"].to_numpy()
    is_att = (truth != "normal").astype(int)
    p_att = 1.0 - proba[:, classes.index("normal")]
    present = [c for c in ALL_CLASSES if c in set(truth)]
    row = {"variant": variant, "fold": held, "seed": seed, "n_test": len(test),
           "n_features": len(cols),
           "macro_f1": f1_score(truth, pred, labels=present, average="macro", zero_division=0)}
    if 0 < is_att.sum() < len(is_att):
        row["roc_auc"] = roc_auc_score(is_att, p_att)
        row["pr_auc"] = average_precision_score(is_att, p_att)
    else:
        row["roc_auc"] = row["pr_auc"] = np.nan
    return row

VARIANTS = ["baseline", "cleaning", "scaling", "window",
            "net_base", "net_feat", "plan_base", "plan_all"]
rows = []
for variant in VARIANTS:
    for held in DEV_FOLDS:
        for seed in SEEDS:
            rows.append(run_variant(variant, held, seed))
    print("done:", variant)
res = pd.DataFrame(rows)
res.to_csv(WORK_DIR / "step14_results.csv", index=False)

# ---------------------------------------------------------------- verdicts
COMPONENTS = [("cleaning", "baseline"), ("scaling", "baseline"), ("window", "baseline"),
              ("net_feat", "net_base"), ("plan_all", "plan_base")]

def fold_table(metric, variant):
    return res[res.variant == variant].groupby("fold")[metric].agg(["mean", "std"])

verdict_rows = []
for comp, base in COMPONENTS:
    ok_all, detail = True, {}
    for metric in (PRIMARY, "macro_f1"):
        b, v = fold_table(metric, base), fold_table(metric, comp)
        for fold in DEV_FOLDS:
            delta = v.loc[fold, "mean"] - b.loc[fold, "mean"]
            sd = b.loc[fold, "std"]
            detail[f"{metric}_{fold}_delta"] = round(float(delta), 4)
            detail[f"{metric}_{fold}_base_sd"] = round(float(sd), 4)
            if metric == PRIMARY and not (np.isfinite(delta) and np.isfinite(sd)
                                          and delta > 0 and delta > sd):
                ok_all = False
    verdict_rows.append({"component": comp, "compared_with": base,
                         "verdict_on_" + PRIMARY: "KEEP" if ok_all else "DROP", **detail})
verdicts = pd.DataFrame(verdict_rows)
verdicts.to_csv(WORK_DIR / "step14_verdicts.csv", index=False)

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
print("\nPER-FOLD MEANS (over seeds)")
display(res.groupby(["variant", "fold"])[["n_test", "n_features", "roc_auc", "pr_auc", "macro_f1"]]
        .mean().round(4))
print("\nVERDICTS (rule: better than baseline by more than the baseline seed SD in ALL three folds, primary = ROC-AUC)")
display(verdicts[["component", "compared_with", "verdict_on_" + PRIMARY]])
print("\nDELTAS vs paired baseline (and baseline seed SD)")
display(verdicts.drop(columns=["verdict_on_" + PRIMARY]).set_index("component").T)
print("\nStep 14b complete. phy_att_4 was not used.")

Covered share by acquisition: {'phy_norm': 0.9991, 'phy_att_1': 0.0781, 'phy_att_2': 0.9962, 'phy_att_3': 0.9418}
done: baseline
done: cleaning
done: scaling
done: window
done: net_base
done: net_feat
done: plan_base
done: plan_all

PER-FOLD MEANS (over seeds)


n_test  n_features  roc_auc  pr_auc  macro_f1
variant   fold                                                    
baseline  phy_att_1  2420.0        40.0   0.6464  0.4822    0.3427
          phy_att_2  2104.0        40.0   0.7252  0.3624    0.5854
          phy_att_3  1254.0        40.0   0.5586  0.2900    0.3447
cleaning  phy_att_1  2420.0        20.0   0.6400  0.4778    0.3430
          phy_att_2  2104.0        21.0   0.7215  0.3581    0.5852
          phy_att_3  1254.0        21.0   0.5709  0.2942    0.3437
net_base  phy_att_1   189.0        40.0   0.5400  0.3982    0.4255
          phy_att_2  2096.0        40.0   0.7090  0.4016    0.5293
          phy_att_3  1181.0        40.0   0.6468  0.5562    0.3232
net_feat  phy_att_1   189.0        54.0   0.6905  0.5369    0.4876
          phy_att_2  2096.0        54.0   0.8528  0.4808    0.6555
          phy_att_3  1181.0        54.0   0.8032  0.7429    0.4753
plan_all  phy_att_1   189.0       154.0   0.6719  0.4160    0.4425
          phy_att_2  2096.0       154.0   0.8224  0.5310    0.7074
          phy_att_3  1181.0       154.0   0.8679  0.7368    0.4479
plan_base phy_att_1   189.0        40.0   0.5400  0.3982    0.4255
          phy_att_2  2096.0        40.0   0.7090  0.4016    0.5293
          phy_att_3  1181.0        40.0   0.6468  0.5562    0.3232
scaling   phy_att_1  2420.0        40.0   0.6464  0.4822    0.3427
          phy_att_2  2104.0        40.0   0.7252  0.3624    0.5854
          phy_att_3  1254.0        40.0   0.5586  0.2900    0.3447
window    phy_att_1  2420.0       160.0   0.6474  0.5507    0.4434
          phy_att_2  2104.0       160.0   0.8192  0.5637    0.7172
          phy_att_3  1254.0       160.0   0.6391  0.4698    0.4645


VERDICTS (rule: better than baseline by more than the baseline seed SD in ALL three folds, primary = ROC-AUC)


,component,compared_with,verdict_on_roc_auc
0,cleaning,baseline,DROP
1,scaling,baseline,DROP
2,window,baseline,DROP
3,net_feat,net_base,KEEP
4,plan_all,plan_base,KEEP



DELTAS vs paired baseline (and baseline seed SD)


component,cleaning,scaling,window,net_feat,plan_all
compared_with,baseline,baseline,baseline,net_base,plan_base
roc_auc_phy_att_1_delta,-0.0064,0.0,0.001,0.1505,0.1319
roc_auc_phy_att_1_base_sd,0.0059,0.0059,0.0059,0.016,0.016
roc_auc_phy_att_2_delta,-0.0037,0.0,0.094,0.1438,0.1135
roc_auc_phy_att_2_base_sd,0.0083,0.0083,0.0083,0.0167,0.0167
roc_auc_phy_att_3_delta,0.0123,0.0,0.0805,0.1564,0.2211
roc_auc_phy_att_3_base_sd,0.008,0.008,0.008,0.0146,0.0146
macro_f1_phy_att_1_delta,0.0002,0.0,0.1007,0.0621,0.017
macro_f1_phy_att_1_base_sd,0.0022,0.0022,0.0022,0.0,0.0
macro_f1_phy_att_2_delta,-0.0002,0.0,0.1317,0.1262,0.1782



Step 14b complete. phy_att_4 was not used.


In [29]:
# STEP 15: Diagnostics for the network-feature result (no new verdicts; development folds only; attack_4 untouched)
# Needs Step 14b to have run in this kernel (uses its `data`, NET_COLS, feature_cols, DEV_FOLDS, SEEDS).

import itertools
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.utils.class_weight import compute_sample_weight

for v in ["data", "NET_COLS", "feature_cols", "DEV_FOLDS", "SEEDS", "WORK_DIR"]:
    assert v in globals(), f"Missing {v}: run Step 14b first."
assert "phy_att_4" not in DEV_FOLDS

def fit_proba(train, test, cols, seed, target="y"):
    y_lab = train[target].to_numpy()
    classes = sorted(set(y_lab))
    idx = {c: i for i, c in enumerate(classes)}
    y = np.array([idx[c] for c in y_lab])
    m = xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1, subsample=0.8,
                          colsample_bytree=0.8, tree_method="hist", n_jobs=-1, random_state=seed)
    m.fit(train[cols].astype(float), y, sample_weight=compute_sample_weight("balanced", y))
    return classes, m.predict_proba(test[cols].astype(float))

def covered_split(held):
    names = [n for n in DEV_FOLDS if n != held] + ["phy_norm"]
    tr = pd.concat([data[n] for n in names]); te = data[held]
    return tr[tr["covered"]], te[te["covered"]]

# =========================================================== 1) detection by attack subtype
print("=" * 70, "\n1) DETECTION RATE BY SUBTYPE (argmax is not 'normal'), network-covered rows\n" + "=" * 70)
rows = []
for held in DEV_FOLDS:
    tr, te = covered_split(held)
    for label, cols in [("physical_only", list(feature_cols)),
                        ("physical+network", list(feature_cols) + NET_COLS)]:
        hits = {}
        for seed in SEEDS:
            classes, proba = fit_proba(tr, te, cols, seed)
            detected = np.array(classes)[proba.argmax(1)] != "normal"
            df = pd.DataFrame({"sub": te["target_class"].astype(str).to_numpy(), "det": detected})
            for sub, g in df.groupby("sub"):
                hits.setdefault(sub, []).append(g["det"].mean())
        for sub, v in hits.items():
            n = int((te["target_class"].astype(str) == sub).sum())
            rows.append({"fold": held, "subtype": sub, "n_rows": n, "model": label,
                         "detected_share": round(float(np.mean(v)), 3)})
sub_tbl = pd.DataFrame(rows).pivot_table(index=["fold", "subtype", "n_rows"],
                                         columns="model", values="detected_share").reset_index()
display(sub_tbl)
print("For 'normal' rows the number is the false-alarm rate (lower is better); for attacks it is detection (higher is better).")
sub_tbl.to_csv(WORK_DIR / "step15_subtype_detection.csv", index=False)

# =========================================================== 2) feature-group ablation
print("\n" + "=" * 70, "\n2) WHICH NETWORK FEATURES DRIVE THE GAIN (ROC-AUC attack vs normal, covered rows)\n" + "=" * 70)
groups = {
    "volume": [c for c in NET_COLS if c == "net_pkts"],
    "size": [c for c in NET_COLS if c.startswith("net_size")],
    "endpoints": [c for c in NET_COLS if c.startswith("net_distinct")],
    "modbus_mix": [c for c in NET_COLS if c.startswith("net_fn") or c == "net_resp_null_share"],
}
assert sorted(sum(groups.values(), [])) == sorted(NET_COLS), "a network feature is missing from the groups"

def auc_for(cols, held):
    tr, te = covered_split(held)
    truth = (te["y"] != "normal").astype(int).to_numpy()
    out = []
    for seed in SEEDS:
        classes, proba = fit_proba(tr, te, cols, seed)
        out.append(roc_auc_score(truth, 1 - proba[:, classes.index("normal")]))
    return float(np.mean(out))

rows = []
settings = [("physical only", [])] + [(f"physical + {g} only", c) for g, c in groups.items()] \
         + [(f"all network minus {g}", [x for x in NET_COLS if x not in c]) for g, c in groups.items()] \
         + [("all network", list(NET_COLS))]
for label, extra in settings:
    r = {"setting": label}
    for held in DEV_FOLDS:
        r[held] = round(auc_for(list(feature_cols) + extra, held), 4)
    r["mean"] = round(np.mean([r[h] for h in DEV_FOLDS]), 4)
    rows.append(r)
abl = pd.DataFrame(rows)
display(abl)
abl.to_csv(WORK_DIR / "step15_group_ablation.csv", index=False)

# =========================================================== 3) session fingerprinting
print("\n" + "=" * 70, "\n3) CAN NORMAL ROWS BE TOLD APART BY SESSION? (time-blocked, normal rows with network data)\n" + "=" * 70)
MIN_ROWS = 40
rows = []
sessions = ["phy_norm"] + DEV_FOLDS
for a, b in itertools.combinations(sessions, 2):
    da = data[a][(data[a]["y"] == "normal") & data[a]["covered"]]
    db = data[b][(data[b]["y"] == "normal") & data[b]["covered"]]
    if min(len(da), len(db)) < MIN_ROWS:
        rows.append({"pair": f"{a} vs {b}", "n_a": len(da), "n_b": len(db),
                     "physical_auc": np.nan, "network_auc": np.nan, "note": "too few rows"})
        continue
    ha, hb = len(da) // 2, len(db) // 2           # first half train, second half test (time-blocked)
    tr = pd.concat([da.iloc[:ha].assign(s=a), db.iloc[:hb].assign(s=b)])
    te = pd.concat([da.iloc[ha:].assign(s=a), db.iloc[hb:].assign(s=b)])
    res = {}
    for label, cols in [("physical_auc", list(feature_cols)), ("network_auc", list(NET_COLS))]:
        classes, proba = fit_proba(tr, te, cols, SEEDS[0], target="s")
        res[label] = round(roc_auc_score((te["s"] == classes[1]).astype(int), proba[:, 1]), 3)
    rows.append({"pair": f"{a} vs {b}", "n_a": len(da), "n_b": len(db), **res, "note": ""})
fp = pd.DataFrame(rows)
display(fp)
fp.to_csv(WORK_DIR / "step15_session_fingerprint.csv", index=False)
print("AUC near 0.5 = sessions cannot be told apart; near 1.0 = the features identify the session.")
print("\nStep 15 complete. No verdicts changed. phy_att_4 was not used.")

1) DETECTION RATE BY SUBTYPE (argmax is not 'normal'), network-covered rows


model,fold,subtype,n_rows,physical+network,physical_only
0,phy_att_1,MITM,49,0.068,0.000
1,phy_att_1,normal,140,0.014,0.000
2,phy_att_2,DoS,80,0.658,0.000
3,phy_att_2,MITM,96,0.198,0.135
4,phy_att_2,normal,1790,0.086,0.034
5,phy_att_2,physical fault,123,0.799,0.780
6,phy_att_2,scan,7,0.286,0.000
7,phy_att_3,DoS,77,0.918,0.000
8,phy_att_3,MITM,104,0.000,0.000
9,phy_att_3,normal,848,0.001,0.000


For 'normal' rows the number is the false-alarm rate (lower is better); for attacks it is detection (higher is better).

2) WHICH NETWORK FEATURES DRIVE THE GAIN (ROC-AUC attack vs normal, covered rows)


,setting,phy_att_1,phy_att_2,phy_att_3,mean
0,physical only,0.5400,0.7090,0.6468,0.6319
1,physical + volume only,0.6678,0.8264,0.7684,0.7542
2,physical + size only,0.7179,0.8303,0.7632,0.7705
3,physical + endpoints only,0.7417,0.8132,0.7636,0.7728
4,physical + modbus_mix only,0.5692,0.8678,0.7540,0.7303
5,all network minus volume,0.6750,0.8643,0.7935,0.7776
6,all network minus size,0.6800,0.8792,0.7859,0.7817
7,all network minus endpoints,0.6546,0.8196,0.7676,0.7473
8,all network minus modbus_mix,0.7164,0.8483,0.8180,0.7942
9,all network,0.6905,0.8528,0.8032,0.7822



3) CAN NORMAL ROWS BE TOLD APART BY SESSION? (time-blocked, normal rows with network data)


,pair,n_a,n_b,physical_auc,network_auc,note
0,phy_norm vs phy_att_1,3425,140,0.494,0.837,
1,phy_norm vs phy_att_2,3425,1790,0.999,1.000,
2,phy_norm vs phy_att_3,3425,848,0.985,0.681,
3,phy_att_1 vs phy_att_2,140,1790,0.746,1.000,
4,phy_att_1 vs phy_att_3,140,848,0.876,0.662,
5,phy_att_2 vs phy_att_3,1790,848,1.000,1.000,


AUC near 0.5 = sessions cannot be told apart; near 1.0 = the features identify the session.

Step 15 complete. No verdicts changed. phy_att_4 was not used.


In [30]:
# STEP 15b: Look at the actual preprocessing (preview only; no models, no verdicts; attack_4 untouched)
# Needs Step 14b to have run in this kernel.

import importlib.util
import numpy as np
import pandas as pd

for v in ["data", "NET_COLS", "feature_cols", "WIN_COLS", "DEV_FOLDS"]:
    assert v in globals(), f"Missing {v}: run Step 14b first."

HELD = "phy_att_3"                       # example held-out fold
names = [n for n in DEV_FOLDS if n != HELD] + ["phy_norm"]
train = pd.concat([data[n] for n in names])
test = data[HELD]
SHOW = 6
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)

# ---- A. physical: raw
phys_show = [c for c in feature_cols][:8]
print("A. PHYSICAL, RAW (first rows of", HELD, "; 8 of", len(feature_cols), "features)")
display(test[["Time", "target_class"] + phys_show].head(SHOW))

# ---- B. physical: what each candidate step does (fitted on the training folds only)
const = [c for c in feature_cols if train[c].astype(float).nunique() <= 1]
corr = train[[c for c in feature_cols if c not in const]].astype(float).corr().abs().fillna(0)
kept = []
for c in corr.columns:
    if all(corr.loc[c, k] <= 0.98 for k in kept):
        kept.append(c)
dup = [c for c in corr.columns if c not in kept]
print(f"\nB. CLEANING (fitted on training folds for held-out {HELD})")
print(f"  constant in training : {len(const)} -> {const}")
print(f"  |r| > 0.98 duplicates: {len(dup)} -> {dup}")
print(f"  features kept        : {len(kept)} of {len(feature_cols)}")

tr_f = train[feature_cols].astype(float)
lo, hi = tr_f.min(), tr_f.max()
span = (hi - lo).replace(0, 1.0)
scaled = (test[feature_cols].astype(float) - lo) / span
print("\n   SCALING (min-max from training folds): same rows, same 8 features")
display(scaled[phys_show].head(SHOW).round(3))
outside = ((scaled < 0) | (scaled > 1)).any(axis=1).mean()
print(f"   share of {HELD} rows with at least one feature outside the training range: {outside:.3f}")

wcols = [c for c in WIN_COLS if c.startswith(phys_show[0])]
print("\n   CAUSAL WINDOW (past 10 s only) for feature", phys_show[0])
display(test[[phys_show[0]] + wcols].iloc[8:8 + SHOW].round(3))

# ---- C. network: packets -> per-second row -> joined to physical row
print("\nC. NETWORK per-second features joined to physical rows (same", HELD, "rows)")
display(pd.concat([test[["Time"]], test[NET_COLS[:8]].round(3)], axis=1).head(SHOW))
cov = test["covered"]
print(f"   rows with network data: {int(cov.sum())} of {len(test)}  (missing rows stay NaN; no flag)")

# ---- D. what each model family would receive
print("\nD. WHAT EACH MODEL RECEIVES")
print(f"  XGBoost : one row = {len(feature_cols)} physical + {len(NET_COLS)} network values (NaN allowed)")
print(f"  LSTM    : one sample = 10 consecutive rows x {len(feature_cols) + len(NET_COLS)} scaled values;")
print("            NaN not allowed, so only network-covered rows can be used")
row = test[cov].iloc[len(test[cov]) // 2]
text = "; ".join(f"{c}={row[c]:.4g}" for c in feature_cols + NET_COLS)
print(f"  Qwen    : one row as text is about {len(text)} characters (~{len(text)//3} tokens); example start:")
print("   ", text[:300], "...")
print(f"            a 10-row window would be about {len(text) * 10 // 3:,} tokens per prediction")

# ---- E. what is installed for the next steps
print("\nE. AVAILABLE LIBRARIES")
for lib in ["torch", "tensorflow", "keras", "transformers", "huggingface_hub"]:
    print(f"  {lib:16s}", "yes" if importlib.util.find_spec(lib) else "no")
print("\nStep 15b complete. Nothing was trained; attack_4 was not used.")

A. PHYSICAL, RAW (first rows of phy_att_3 ; 8 of 40 features)


,Time,target_class,Tank_1,Tank_2,Tank_3,Tank_4,Tank_5,Tank_6,Tank_7,Tank_8
0,2021-04-09 19:42:12,normal,0,0,0,0,0,0,0,0
1,2021-04-09 19:42:13,normal,0,0,0,0,0,0,0,0
2,2021-04-09 19:42:14,normal,0,0,0,0,0,0,0,0
3,2021-04-09 19:42:15,normal,0,0,0,0,0,0,0,0
4,2021-04-09 19:42:16,normal,78,0,0,0,0,0,0,0
5,2021-04-09 19:42:17,normal,81,0,0,0,0,0,0,0



B. CLEANING (fitted on training folds for held-out phy_att_3)
  constant in training : 14 -> ['Pump_3', 'Flow_sensor_3', 'Valv_1', 'Valv_2', 'Valv_3', 'Valv_4', 'Valv_5', 'Valv_6', 'Valv_7', 'Valv_8', 'Valv_9', 'Valv_16', 'Valv_19', 'Valv_21']
  |r| > 0.98 duplicates: 5 -> ['Flow_sensor_2', 'Valv_11', 'Valv_12', 'Valv_14', 'Valv_15']
  features kept        : 21 of 40

   SCALING (min-max from training folds): same rows, same 8 features


,Tank_1,Tank_2,Tank_3,Tank_4,Tank_5,Tank_6,Tank_7,Tank_8
0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.039,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.040,0.0,0.0,0.0,0.0,0.0,0.0,0.0


   share of phy_att_3 rows with at least one feature outside the training range: 0.000

   CAUSAL WINDOW (past 10 s only) for feature Tank_1


,Tank_1,Tank_1_rmean,Tank_1_rstd,Tank_1_diff1
8,284,90.778,107.128,60.0
9,331,114.800,126.380,47.0
10,354,150.200,139.544,23.0
11,398,190.000,148.421,44.0
12,449,234.900,152.417,51.0
13,460,280.900,142.755,11.0



C. NETWORK per-second features joined to physical rows (same phy_att_3 rows)


,Time,net_pkts,net_size_mean,net_size_std,net_size_max,net_resp_null_share,net_distinct_sport,net_distinct_dport,net_distinct_ip_s
0,2021-04-09 19:42:12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-04-09 19:42:13,684.0,65.341,1.922,78.0,0.535,9.0,9.0,5.0
2,2021-04-09 19:42:14,2233.0,65.352,1.458,78.0,0.515,11.0,11.0,7.0
3,2021-04-09 19:42:15,2322.0,65.290,1.478,78.0,0.518,12.0,12.0,7.0
4,2021-04-09 19:42:16,2273.0,65.365,1.384,78.0,0.513,10.0,10.0,7.0
5,2021-04-09 19:42:17,2291.0,65.331,1.511,78.0,0.517,12.0,12.0,7.0


   rows with network data: 1181 of 1254  (missing rows stay NaN; no flag)

D. WHAT EACH MODEL RECEIVES
  XGBoost : one row = 40 physical + 14 network values (NaN allowed)
  LSTM    : one sample = 10 consecutive rows x 54 scaled values;
            NaN not allowed, so only network-covered rows can be used
  Qwen    : one row as text is about 804 characters (~268 tokens); example start:
    Tank_1=3; Tank_2=4; Tank_3=712; Tank_4=426; Tank_5=502; Tank_6=318; Tank_7=0; Tank_8=0; Pump_1=0; Pump_2=0; Pump_3=0; Pump_4=0; Pump_5=0; Pump_6=0; Flow_sensor_1=0; Flow_sensor_2=0; Flow_sensor_3=0; Flow_sensor_4=0; Valv_1=0; Valv_2=0; Valv_3=0; Valv_4=0; Valv_5=0; Valv_6=0; Valv_7=0; Valv_8=0; Valv ...
            a 10-row window would be about 2,680 tokens per prediction

E. AVAILABLE LIBRARIES
  torch            yes
  tensorflow       no
  keras            no
  transformers     no
  huggingface_hub  yes

Step 15b complete. Nothing was trained; attack_4 was not used.


In [31]:
# STEP 16: The same component tests for an LSTM (development folds only; attack_4 untouched)
# Needs Step 14b to have run in this kernel (uses its `data`, NET_COLS, feature_cols, cleaning_cols).

from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.utils.class_weight import compute_sample_weight

for v in ["data", "NET_COLS", "feature_cols", "DEV_FOLDS", "SEEDS", "ALL_CLASSES",
          "cleaning_cols", "WORK_DIR"]:
    assert v in globals(), f"Missing {v}: run Step 14b first."
assert "phy_att_4" not in DEV_FOLDS

SEQ = 10          # rows per sample (past 9 rows + the current row), inside one acquisition
HIDDEN = 64
EPOCHS = 15       # fixed in advance; no early stopping on the test fold
BATCH = 256
LR = 1e-3
PRIMARY = "roc_auc"

# ---------------------------------------------------------------- freeze the design first
PLAN16 = f"""# Step 16 frozen LSTM test design
Written {datetime.now().isoformat(timespec='seconds')} BEFORE any LSTM result was computed.
- Model: 1-layer LSTM (hidden {HIDDEN}) + linear head, class-weighted cross-entropy (weights from the training fold),
  {EPOCHS} epochs, batch {BATCH}, Adam lr {LR}, seeds {SEEDS}. Same folds as Step 14b; phy_norm always in training.
- Sample = {SEQ} consecutive rows of one acquisition, label = class of the last row. Sequences never cross acquisitions.
- Variants: raw (unscaled physical), scaled, cleaned_scaled (all rows with {SEQ} rows of history);
  phys_scaled_cc and net_scaled_cc (only samples whose {SEQ} rows all have network data; no availability flag).
- Components: scaling = scaled vs raw; cleaning = cleaned_scaled vs scaled; network = net_scaled_cc vs phys_scaled_cc.
- Rule (unchanged): KEEP only if ROC-AUC attack-vs-normal improves in ALL three folds by more than the baseline seed SD.
- Windowing is part of how an LSTM reads data, so it is not tested as a component here.
"""
(WORK_DIR / "step16_frozen_plan.md").write_text(PLAN16, encoding="utf-8")

# ---------------------------------------------------------------- sample building (numpy only)
CLASS_ORDER = sorted(ALL_CLASSES)

def acq_arrays(name):
    d = data[name]
    return {"phys": d[feature_cols].astype(float).to_numpy(),
            "net": d[NET_COLS].astype(float).to_numpy(),
            "y": d["y"].to_numpy(),
            "covered": d["covered"].to_numpy()}

def make_samples(arr, cols_phys, use_net, covered_only):
    X = arr["phys"][:, cols_phys]
    if use_net:
        X = np.concatenate([X, arr["net"]], axis=1)
    n = len(X)
    if n < SEQ:
        return np.empty((0, SEQ, X.shape[1])), np.empty(0, dtype=object)
    idx = np.arange(SEQ - 1, n)
    keep = np.ones(len(idx), dtype=bool)
    if covered_only:
        cov = arr["covered"].astype(int)
        csum = np.concatenate([[0], np.cumsum(cov)])
        keep = (csum[idx + 1] - csum[idx + 1 - SEQ]) == SEQ
    idx = idx[keep]
    win = np.stack([X[i - SEQ + 1:i + 1] for i in idx]) if len(idx) else np.empty((0, SEQ, X.shape[1]))
    return win, arr["y"][idx]

def build(variant, held):
    train_names = [n for n in DEV_FOLDS if n != held] + ["phy_norm"]
    arrs = {n: acq_arrays(n) for n in train_names + [held]}
    cols = list(range(len(feature_cols)))
    if variant == "cleaned_scaled":
        train_df = pd.concat([data[n] for n in train_names])
        kept = cleaning_cols(train_df, list(feature_cols))
        cols = [feature_cols.index(c) for c in kept]
    use_net = variant == "net_scaled_cc"
    covered_only = variant in ("phys_scaled_cc", "net_scaled_cc")
    Xtr_l, ytr_l = [], []
    for n in train_names:
        w, y = make_samples(arrs[n], cols, use_net, covered_only)
        Xtr_l.append(w); ytr_l.append(y)
    Xtr, ytr = np.concatenate(Xtr_l), np.concatenate(ytr_l)
    Xte, yte = make_samples(arrs[held], cols, use_net, covered_only)
    if variant != "raw":              # min-max fitted on the training samples only
        flat = Xtr.reshape(-1, Xtr.shape[-1])
        lo, hi = np.nanmin(flat, axis=0), np.nanmax(flat, axis=0)
        span = np.where(hi - lo == 0, 1.0, hi - lo)
        Xtr = (Xtr - lo) / span
        Xte = (Xte - lo) / span
    assert not np.isnan(Xtr).any() and not np.isnan(Xte).any(), "NaN reached the LSTM"
    return Xtr.astype(np.float32), ytr, Xte.astype(np.float32), yte

# ---------------------------------------------------------------- the LSTM (torch)
def train_predict_torch(Xtr, ytr, Xte, seed):
    import torch
    from torch import nn
    torch.manual_seed(seed)
    np.random.seed(seed)
    classes = sorted(set(ytr))
    idx = {c: i for i, c in enumerate(classes)}
    y = np.array([idx[c] for c in ytr])
    w = compute_sample_weight("balanced", y)
    cls_w = np.array([w[y == i][0] for i in range(len(classes))], dtype=np.float32)

    class Net(nn.Module):
        def __init__(self, d, k):
            super().__init__()
            self.lstm = nn.LSTM(d, HIDDEN, batch_first=True)
            self.out = nn.Linear(HIDDEN, k)
        def forward(self, x):
            h, _ = self.lstm(x)
            return self.out(h[:, -1])

    net = Net(Xtr.shape[-1], len(classes))
    opt = torch.optim.Adam(net.parameters(), lr=LR)
    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(cls_w))
    Xt, yt = torch.tensor(Xtr), torch.tensor(y)
    n = len(Xt)
    for _ in range(EPOCHS):
        net.train()
        perm = torch.randperm(n)
        for i in range(0, n, BATCH):
            b = perm[i:i + BATCH]
            opt.zero_grad()
            loss = loss_fn(net(Xt[b]), yt[b])
            loss.backward()
            opt.step()
    net.eval()
    with torch.no_grad():
        proba = torch.softmax(net(torch.tensor(Xte)), dim=1).numpy()
    return classes, proba

train_predict = globals().get("TEST_TRAIN_PREDICT", train_predict_torch)

# ---------------------------------------------------------------- run
VARIANTS = ["raw", "scaled", "cleaned_scaled", "phys_scaled_cc", "net_scaled_cc"]
rows = []
for variant in VARIANTS:
    for held in DEV_FOLDS:
        Xtr, ytr, Xte, yte = build(variant, held)
        for seed in SEEDS:
            classes, proba = train_predict(Xtr, ytr, Xte, seed)
            pred = np.array(classes)[proba.argmax(1)]
            is_att = (yte != "normal").astype(int)
            p_att = 1.0 - proba[:, classes.index("normal")]
            present = [c for c in CLASS_ORDER if c in set(yte)]
            r = {"variant": variant, "fold": held, "seed": seed, "n_train": len(ytr), "n_test": len(yte),
                 "n_features": Xtr.shape[-1],
                 "macro_f1": f1_score(yte, pred, labels=present, average="macro", zero_division=0)}
            if 0 < is_att.sum() < len(is_att) and np.isfinite(p_att).all():
                r["roc_auc"] = roc_auc_score(is_att, p_att)
                r["pr_auc"] = average_precision_score(is_att, p_att)
            else:
                r["roc_auc"] = r["pr_auc"] = np.nan
            rows.append(r)
    print("done:", variant)
res = pd.DataFrame(rows)
res.to_csv(WORK_DIR / "step16_lstm_results.csv", index=False)

# ---------------------------------------------------------------- verdicts
COMPONENTS = [("scaling", "scaled", "raw"), ("cleaning", "cleaned_scaled", "scaled"),
              ("network", "net_scaled_cc", "phys_scaled_cc")]
def fold_table(metric, variant):
    return res[res.variant == variant].groupby("fold")[metric].agg(["mean", "std"])

vrows = []
for comp, var, base in COMPONENTS:
    ok, detail = True, {}
    for metric in (PRIMARY, "macro_f1"):
        b, v = fold_table(metric, base), fold_table(metric, var)
        for fold in DEV_FOLDS:
            delta, sd = v.loc[fold, "mean"] - b.loc[fold, "mean"], b.loc[fold, "std"]
            detail[f"{metric}_{fold}_delta"] = round(float(delta), 4)
            detail[f"{metric}_{fold}_base_sd"] = round(float(sd), 4)
            if metric == PRIMARY and not (np.isfinite(delta) and np.isfinite(sd) and delta > 0 and delta > sd):
                ok = False
    vrows.append({"component": comp, "variant": var, "compared_with": base,
                  "verdict_on_" + PRIMARY: "KEEP" if ok else "DROP", **detail})
verdicts = pd.DataFrame(vrows)
verdicts.to_csv(WORK_DIR / "step16_lstm_verdicts.csv", index=False)

pd.set_option("display.width", 250)
print("\nLSTM PER-FOLD MEANS (over seeds)")
display(res.groupby(["variant", "fold"])[["n_train", "n_test", "n_features", "roc_auc", "pr_auc", "macro_f1"]]
        .mean().round(4))
print("\nLSTM VERDICTS (same rule as XGBoost)")
display(verdicts[["component", "variant", "compared_with", "verdict_on_" + PRIMARY]])
print("\nDELTAS vs paired baseline (and baseline seed SD)")
display(verdicts.drop(columns=["variant", "compared_with", "verdict_on_" + PRIMARY]).set_index("component").T)
print("\nStep 16 complete. phy_att_4 was not used.")

done: raw
done: scaled
done: cleaned_scaled
done: phys_scaled_cc
done: net_scaled_cc

LSTM PER-FOLD MEANS (over seeds)


n_train  n_test  n_features  roc_auc  pr_auc  macro_f1
variant        fold                                                             
cleaned_scaled phy_att_1   6759.0  2411.0        20.0   0.6228  0.4226    0.4098
               phy_att_2   7075.0  2095.0        21.0   0.6436  0.2147    0.4370
               phy_att_3   7925.0  1245.0        21.0   0.6384  0.3216    0.4006
net_scaled_cc  phy_att_1   6675.0   180.0        54.0   0.4467  0.3550    0.5625
               phy_att_2   4768.0  2087.0        54.0   0.5812  0.1584    0.3258
               phy_att_3   5683.0  1172.0        54.0   0.7148  0.5359    0.3624
phys_scaled_cc phy_att_1   6675.0   180.0        40.0   0.2853  0.1924    0.4010
               phy_att_2   4768.0  2087.0        40.0   0.5275  0.1746    0.3115
               phy_att_3   5683.0  1172.0        40.0   0.5924  0.3375    0.3284
raw            phy_att_1   6759.0  2411.0        40.0   0.5779  0.4200    0.3966
               phy_att_2   7075.0  2095.0        40.0   0.4995  0.1368    0.2964
               phy_att_3   7925.0  1245.0        40.0   0.4660  0.2512    0.3618
scaled         phy_att_1   6759.0  2411.0        40.0   0.5859  0.4039    0.3807
               phy_att_2   7075.0  2095.0        40.0   0.6186  0.2763    0.3960
               phy_att_3   7925.0  1245.0        40.0   0.6296  0.3205    0.4087


LSTM VERDICTS (same rule as XGBoost)


,component,variant,compared_with,verdict_on_roc_auc
0,scaling,scaled,raw,DROP
1,cleaning,cleaned_scaled,scaled,DROP
2,network,net_scaled_cc,phys_scaled_cc,KEEP



DELTAS vs paired baseline (and baseline seed SD)


component,scaling,cleaning,network
roc_auc_phy_att_1_delta,0.0080,0.0369,0.1614
roc_auc_phy_att_1_base_sd,0.0187,0.0181,0.0405
roc_auc_phy_att_2_delta,0.1191,0.0250,0.0537
roc_auc_phy_att_2_base_sd,0.0674,0.0446,0.0017
roc_auc_phy_att_3_delta,0.1637,0.0087,0.1224
roc_auc_phy_att_3_base_sd,0.0100,0.0272,0.0319
macro_f1_phy_att_1_delta,-0.0159,0.0291,0.1615
macro_f1_phy_att_1_base_sd,0.0341,0.0291,0.0218
macro_f1_phy_att_2_delta,0.0995,0.0410,0.0142
macro_f1_phy_att_2_base_sd,0.0446,0.0817,0.0048



Step 16 complete. phy_att_4 was not used.


In [36]:
# STEP 17: The same component tests for Qwen3-32B reading one second of data as text
# (development folds only; attack_4 untouched). Needs Step 14b to have run in this kernel.

import json
import re
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

for v in ["data", "NET_COLS", "feature_cols", "DEV_FOLDS", "client", "MODEL_ID", "WORK_DIR"]:
    assert v in globals(), f"Missing {v}: run Steps 1 and 14b first."
assert "phy_att_4" not in DEV_FOLDS

# ------------------------------------------------------------------ settings you may change
MODE = "run"          # "estimate" = no API calls | "probe" = 24 real calls | "run" = everything
ATT_PER_FOLD = 75          # attack rows per fold (fewer if the fold has fewer covered attack rows)
NORM_PER_FOLD = 75         # normal rows per fold
CAP_USD = 2.00             # the run stops when the estimated spend reaches this
PRICE_IN_PER_M = 1.00      # ASSUMED ceiling price, dollars per million input tokens (provider unknown)
PRICE_OUT_PER_M = 3.00     # ASSUMED ceiling price, dollars per million output tokens
WORKERS = 4
MAX_OUT_TOKENS = 60
BOOTSTRAPS = 300
SAMPLE_SEED = 0
MAX_OUTSIDE = 12           # at most this many "outside the normal range" items per row

assert MODE in ("estimate", "probe", "run")

class StopExecution(Exception):
    """Stops the cell quietly (no traceback in Jupyter)."""
    def _render_traceback_(self):
        return []

# ------------------------------------------------------------------ frozen design
PLAN17 = f"""# Step 17 frozen Qwen test design
Written {datetime.now().isoformat(timespec='seconds')} BEFORE any Qwen result was computed.
- Model: {MODEL_ID}, thinking off (/no_think), temperature 0, JSON reply {{"class", "attack_score" 0-100}}.
- Unit: ONE second (one row), no window. Same sampled rows for every variant: network-covered rows only,
  up to {ATT_PER_FOLD} attack and {NORM_PER_FOLD} normal rows per fold, random seed {SAMPLE_SEED}.
- Variants: Q_raw = task text + raw physical values (no preprocessing);
  Q_ref = Q_raw + summary of normal operation from the training folds + list of values outside the normal range;
  Q_ref_net = Q_ref + the per-second network features (values, normal summary, outside-range list).
- Components: reference = Q_ref vs Q_raw; network = Q_ref_net vs Q_ref.
- Primary metric: ROC-AUC of attack_score, attack vs normal. Unparsable replies get score 50 and are counted.
- Rule: KEEP only if the ROC-AUC gain is positive and larger than the baseline's bootstrap SD ({BOOTSTRAPS} resamples
  of test rows) in ALL three folds. (One deterministic pass per row, so there is no seed SD; the bootstrap measures row sampling noise.)
- The normal summary and the outside-range lists are an adapter built by the researchers; Qwen does not choose them.
"""
(WORK_DIR / "step17_frozen_plan.md").write_text(PLAN17, encoding="utf-8")

# ------------------------------------------------------------------ text building
def fmt(v):
    if pd.isna(v):
        return "NA"
    v = float(v)
    return str(int(v)) if v.is_integer() else f"{v:.4g}"

def build_reference(held, cols):
    names = [n for n in DEV_FOLDS if n != held] + ["phy_norm"]
    tr = pd.concat([data[n] for n in names])
    tr = tr[(tr["y"] == "normal") & tr["covered"]]
    ref = {}
    for c in cols:
        v = tr[c].astype(float)
        ref[c] = (float(v.median()), float(v.quantile(0.05)), float(v.quantile(0.95)))
    return ref

def reference_text(ref):
    const = [f"{c}={fmt(m)}" for c, (m, lo, hi) in ref.items() if lo == hi]
    var = [f"{c}: median {fmt(m)}, usual range {fmt(lo)} to {fmt(hi)}"
           for c, (m, lo, hi) in ref.items() if lo != hi]
    return ("Normal operation (from training data):\n"
            "Constant in normal operation: " + ", ".join(const) + "\n" + "\n".join(var))

def outside_text(row, ref):
    items = []
    for c, (m, lo, hi) in ref.items():
        v = float(row[c])
        if np.isnan(v):
            continue
        if lo == hi:
            if v != lo:
                items.append((abs(v - lo) / (abs(lo) + 1.0), f"{c}={fmt(v)} (normally {fmt(lo)})"))
        elif v < lo or v > hi:
            items.append((abs(v - m) / (hi - lo), f"{c}={fmt(v)} (normal {fmt(lo)} to {fmt(hi)})"))
    items.sort(key=lambda t: -t[0])
    if not items:
        return "Outside the normal range: none"
    return "Outside the normal range: " + "; ".join(t for _, t in items[:MAX_OUTSIDE])

def row_text(row, cols):
    return "; ".join(f"{c}={fmt(row[c])}" for c in cols)

TASK = ("You are an industrial control system security analyst for a water distribution testbed. "
        "Below is one second of data. Classify it as normal, physical_fault or cyberattack. "
        'Reply with JSON only: {"class": "...", "attack_score": N} where attack_score is an integer from 0 '
        "(certainly normal operation) to 100 (certainly not normal operation).")

def make_prompt(variant, row, ref, ref_text_cache):
    phys, net = list(feature_cols), list(NET_COLS)
    if variant == "Q_raw":
        body = "Observation:\n" + row_text(row, phys)
    elif variant == "Q_ref":
        body = (ref_text_cache["phys"] + "\n\nObservation:\n" + row_text(row, phys) + "\n"
                + outside_text(row, {c: ref[c] for c in phys}))
    else:
        body = (ref_text_cache["both"] + "\n\nObservation:\n" + row_text(row, phys + net) + "\n"
                + outside_text(row, ref))
    return TASK + "\n\n" + body + "\n/no_think"

# ------------------------------------------------------------------ sample + prompts
rng = np.random.default_rng(SAMPLE_SEED)
VARIANTS = ["Q_raw", "Q_ref", "Q_ref_net"]
items = []                                   # one dict per (variant, fold, row)
for held in DEV_FOLDS:
    te = data[held]
    te = te[te["covered"]]
    att = np.where(te["y"].to_numpy() != "normal")[0]
    nor = np.where(te["y"].to_numpy() == "normal")[0]
    pick = np.concatenate([rng.choice(att, min(ATT_PER_FOLD, len(att)), replace=False),
                           rng.choice(nor, min(NORM_PER_FOLD, len(nor)), replace=False)])
    ref = build_reference(held, list(feature_cols) + list(NET_COLS))
    cache = {"phys": reference_text({c: ref[c] for c in feature_cols}), "both": reference_text(ref)}
    for pos in pick:
        row = te.iloc[pos].copy()
        row[list(feature_cols)] = row[list(feature_cols)].astype(float)
        for var in VARIANTS:
            items.append({"variant": var, "fold": held, "row": int(pos),
                          "truth": str(row["y"]), "subtype": str(row["target_class"]),
                          "prompt": make_prompt(var, row, ref, cache)})
print(f"{len(items):,} prompts ({len(items)//len(VARIANTS)} rows x {len(VARIANTS)} variants)")
chars = pd.Series([len(i["prompt"]) for i in items]).groupby([i["variant"] for i in items]).mean()
print("Average prompt characters by variant:", chars.round(0).to_dict())

est_in = sum(len(i["prompt"]) for i in items) / 3.0          # about 3 characters per token (rough)
est_out = len(items) * 25
est_cost = est_in / 1e6 * PRICE_IN_PER_M + est_out / 1e6 * PRICE_OUT_PER_M
print(f"Rough estimate: {est_in/1e6:.2f}M input tokens, {est_out/1e3:.0f}k output tokens "
      f"-> ${est_cost:.2f} at the assumed ceiling prices (${PRICE_IN_PER_M}/M in, ${PRICE_OUT_PER_M}/M out)")
print("Example prompt (Q_ref_net, first 1,800 characters):\n")
print(next(i for i in items if i["variant"] == "Q_ref_net")["prompt"][:1800])

# ------------------------------------------------------------------ calling the model
def call_qwen(prompt):
    last = None
    for attempt in range(3):
        try:
            r = client.chat.completions.create(
                model=MODEL_ID, messages=[{"role": "user", "content": prompt}],
                max_tokens=MAX_OUT_TOKENS, temperature=0.0, stream=False)
            text = r.choices[0].message.content or ""
            text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
            u = getattr(r, "usage", None)
            return text, getattr(u, "prompt_tokens", None), getattr(u, "completion_tokens", None)
        except Exception as e:
            last = e
            time.sleep(2 * (attempt + 1))
    return f"ERROR {type(last).__name__}: {str(last)[:120]}", None, None

def parse_reply(text):
    m = re.search(r"\{.*?\}", text, flags=re.DOTALL)
    if not m:
        return None, None
    try:
        d = json.loads(m.group(0))
        score = float(d.get("attack_score"))
        return str(d.get("class")), min(max(score, 0.0), 100.0)
    except Exception:
        return None, None

def run_items(batch, cap_usd):
    spent = {"in": 0, "out": 0}
    stop = {"flag": False}
    def work(it):
        if stop["flag"]:
            return it, None
        text, tin, tout = call_qwen(it["prompt"])
        spent["in"] += tin or len(it["prompt"]) // 3
        spent["out"] += tout or 25
        cost = spent["in"] / 1e6 * PRICE_IN_PER_M + spent["out"] / 1e6 * PRICE_OUT_PER_M
        if cost >= cap_usd:
            stop["flag"] = True
        return it, (text, tin, tout)
    out = []
    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        for it, res in ex.map(work, batch):
            out.append((it, res))
    cost = spent["in"] / 1e6 * PRICE_IN_PER_M + spent["out"] / 1e6 * PRICE_OUT_PER_M
    return out, spent, cost, stop["flag"]

if MODE == "estimate":
    print("\nMODE = 'estimate': no calls were made. Set MODE = 'probe' to test 24 real calls.")
    raise StopExecution

# probe: 24 calls spread evenly over all folds, subtypes and variants (not just the first rows)
batch = items[::max(1, len(items) // 24)][:24] if MODE == "probe" else items
results, spent, cost, stopped = run_items(batch, CAP_USD)
print(f"\nCalls: {len(results)} | measured tokens in/out: {spent['in']:,}/{spent['out']:,} "
      f"| estimated spend at assumed ceiling prices: ${cost:.3f}" + (" | STOPPED AT CAP" if stopped else ""))

rows = []
for it, res in results:
    if res is None:
        continue
    text, tin, tout = res
    cls, score = parse_reply(text)
    rows.append({"variant": it["variant"], "fold": it["fold"], "row": it["row"], "truth": it["truth"],
                 "subtype": it["subtype"], "reply": text[:200], "cls": cls, "score": score,
                 "tokens_in": tin, "tokens_out": tout})
out = pd.DataFrame(rows)
out.to_csv(WORK_DIR / f"step17_qwen_{MODE}.csv", index=False)

if MODE == "probe":
    display(out[["variant", "fold", "subtype", "score", "tokens_in", "tokens_out"]])
    print("Distinct scores seen:", sorted(out["score"].dropna().unique().tolist()))
    ti, to = out["tokens_in"].dropna().mean(), out["tokens_out"].dropna().mean()
    if not np.isnan(ti):
        full = len(items) * (ti / 1e6 * PRICE_IN_PER_M + to / 1e6 * PRICE_OUT_PER_M)
        print(f"\nMeasured average {ti:.0f} input / {to:.0f} output tokens per call "
              f"-> full run about ${full:.2f} at the assumed ceiling prices.")
    print("Check that the replies parse (score is not empty). Then set MODE = 'run'.")
    raise StopExecution

# ------------------------------------------------------------------ metrics and verdicts
out["score_f"] = out["score"].fillna(50.0)
fails = out.groupby("variant")["score"].apply(lambda s: int(s.isna().sum()))
print("Unparsable replies by variant:", fails.to_dict())

def auc(df):
    y = (df["truth"] != "normal").astype(int).to_numpy()
    return roc_auc_score(y, df["score_f"].to_numpy()) if 0 < y.sum() < len(y) else np.nan

def boot_sd(df, B=BOOTSTRAPS):
    r = np.random.default_rng(1)
    vals = []
    y = (df["truth"] != "normal").astype(int).to_numpy()
    s = df["score_f"].to_numpy()
    for _ in range(B):
        i = r.integers(0, len(df), len(df))
        if 0 < y[i].sum() < len(i):
            vals.append(roc_auc_score(y[i], s[i]))
    return float(np.std(vals)) if vals else np.nan

summ = []
for var in VARIANTS:
    for fold in DEV_FOLDS:
        d = out[(out.variant == var) & (out.fold == fold)]
        if d.empty:
            continue
        a = d["truth"] != "normal"
        summ.append({"variant": var, "fold": fold, "n": len(d), "roc_auc": round(auc(d), 4),
                     "boot_sd": round(boot_sd(d), 4),
                     "attack_detected_at_50": round(float((d.loc[a, "score_f"] >= 50).mean()), 3),
                     "normal_false_alarm_at_50": round(float((d.loc[~a, "score_f"] >= 50).mean()), 3)})
summ = pd.DataFrame(summ)
pd.set_option("display.width", 250)
print("\nQWEN PER-FOLD RESULTS (balanced subsample; detection and false alarm use a fixed cutoff of 50)")
display(summ)

vrows = []
for comp, var, base in [("reference", "Q_ref", "Q_raw"), ("network", "Q_ref_net", "Q_ref")]:
    ok, det = True, {}
    for fold in DEV_FOLDS:
        b = summ[(summ.variant == base) & (summ.fold == fold)].iloc[0]
        v = summ[(summ.variant == var) & (summ.fold == fold)].iloc[0]
        delta = v["roc_auc"] - b["roc_auc"]
        det[f"{fold}_delta"] = round(float(delta), 4)
        det[f"{fold}_base_boot_sd"] = b["boot_sd"]
        if not (np.isfinite(delta) and delta > 0 and delta > b["boot_sd"]):
            ok = False
    vrows.append({"component": comp, "variant": var, "compared_with": base,
                  "verdict_on_roc_auc": "KEEP" if ok else "DROP", **det})
verdicts = pd.DataFrame(vrows)
print("\nQWEN VERDICTS (same rule; noise = bootstrap SD over test rows)")
display(verdicts)
sub = out[out.truth != "normal"].assign(det=lambda d: d["score_f"] >= 50)
print("\nDETECTION AT CUTOFF 50 BY SUBTYPE")
display(sub.pivot_table(index=["fold", "subtype"], columns="variant", values="det", aggfunc="mean").round(3))
out.to_csv(WORK_DIR / "step17_qwen_results.csv", index=False)
verdicts.to_csv(WORK_DIR / "step17_qwen_verdicts.csv", index=False)
print("\nStep 17 complete. phy_att_4 was not used.")

1,272 prompts (424 rows x 3 variants)
Average prompt characters by variant: {'Q_raw': 815.0, 'Q_ref': 2120.0, 'Q_ref_net': 3404.0}
Rough estimate: 0.90M input tokens, 32k output tokens -> $0.99 at the assumed ceiling prices ($1.0/M in, $3.0/M out)
Example prompt (Q_ref_net, first 1,800 characters):

You are an industrial control system security analyst for a water distribution testbed. Below is one second of data. Classify it as normal, physical_fault or cyberattack. Reply with JSON only: {"class": "...", "attack_score": N} where attack_score is an integer from 0 (certainly normal operation) to 100 (certainly not normal operation).

Normal operation (from training data):
Constant in normal operation: Pump_3=0, Flow_sensor_3=0, Valv_1=0, Valv_2=0, Valv_3=0, Valv_4=0, Valv_5=0, Valv_6=0, Valv_7=0, Valv_8=0, Valv_9=0, Valv_16=0, Valv_19=0, Valv_21=0, net_size_max=78, net_fn_other=0
Tank_1: median 611, usual range 1 to 1856
Tank_2: median 1116, usual range 3 to 1877
Tank_3: median 366, usu

,variant,fold,n,roc_auc,boot_sd,attack_detected_at_50,normal_false_alarm_at_50
0,Q_raw,phy_att_1,124,0.5035,0.0204,0.000,0.013
1,Q_raw,phy_att_2,150,0.5612,0.0370,0.160,0.040
2,Q_raw,phy_att_3,150,0.5213,0.0312,0.027,0.053
3,Q_ref,phy_att_1,124,0.2012,0.0371,0.000,0.107
4,Q_ref,phy_att_2,150,0.6226,0.0444,0.427,0.253
5,Q_ref,phy_att_3,150,0.5643,0.0441,0.240,0.200
6,Q_ref_net,phy_att_1,124,0.4331,0.0569,0.265,0.240
7,Q_ref_net,phy_att_2,150,0.6196,0.0372,1.000,1.000
8,Q_ref_net,phy_att_3,150,0.7362,0.0408,0.600,0.280



QWEN VERDICTS (same rule; noise = bootstrap SD over test rows)


,component,variant,compared_with,verdict_on_roc_auc,phy_att_1_delta,phy_att_1_base_boot_sd,phy_att_2_delta,phy_att_2_base_boot_sd,phy_att_3_delta,phy_att_3_base_boot_sd
0,reference,Q_ref,Q_raw,DROP,-0.3023,0.0204,0.0614,0.0370,0.0430,0.0312
1,network,Q_ref_net,Q_ref,DROP,0.2319,0.0371,-0.0030,0.0444,0.1719,0.0441



DETECTION AT CUTOFF 50 BY SUBTYPE


variant                   Q_raw  Q_ref  Q_ref_net
fold      subtype                                
phy_att_1 MITM            0.000  0.000      0.265
phy_att_2 DoS             0.273  0.136      1.000
          MITM            0.000  0.417      1.000
          physical fault  0.214  0.643      1.000
          scan            0.000  1.000      1.000
phy_att_3 DoS             0.000  0.118      1.000
          MITM            0.000  0.154      0.346
          physical fault  0.062  0.375      0.594


Step 17 complete. phy_att_4 was not used.


In [37]:
# STEP 18: Full metrics report (accuracy, precision, recall, F1, ROC-AUC, PR-AUC ...) for every model and variant
# Development folds only; attack_4 untouched. Needs Step 14b (and Step 16 for the LSTM, Step 17 for Qwen) run in this kernel.

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                             confusion_matrix, f1_score, precision_score, recall_score,
                             roc_auc_score, roc_curve)
from sklearn.utils.class_weight import compute_sample_weight

for v in ["data", "NET_COLS", "feature_cols", "DEV_FOLDS", "SEEDS", "ALL_CLASSES", "WORK_DIR"]:
    assert v in globals(), f"Missing {v}: run Step 14b first."
assert "phy_att_4" not in DEV_FOLDS

RUN_LSTM = True            # about 10-15 minutes on CPU; needs Step 16 to have run in this kernel
QWEN_CUTOFF = 50           # fixed in advance, the same cutoff used in Step 17
CLASSES3 = sorted(ALL_CLASSES)

# ------------------------------------------------------------------ metrics
def recall_at_fpr(y_att, score, fpr_limit=0.05):
    if y_att.sum() == 0 or y_att.sum() == len(y_att):
        return np.nan
    fpr, tpr, _ = roc_curve(y_att, score)
    return float(tpr[fpr <= fpr_limit].max())

def all_metrics(truth3, pred3, p_att, pred_att):
    """truth3/pred3: 3-class labels; p_att: attack score; pred_att: boolean decision (attack = True)."""
    truth3, pred3 = np.asarray(truth3), np.asarray(pred3)
    y = (truth3 != "normal").astype(int)
    d = np.asarray(pred_att).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, d, labels=[0, 1]).ravel()
    out = {"n": len(y), "attack_share": y.mean(),
           "majority_acc": max(y.mean(), 1 - y.mean()),
           "accuracy": accuracy_score(y, d), "balanced_acc": balanced_accuracy_score(y, d),
           "precision": precision_score(y, d, zero_division=0), "recall": recall_score(y, d, zero_division=0),
           "specificity": tn / (tn + fp) if tn + fp else np.nan,
           "f1": f1_score(y, d, zero_division=0)}
    ok = 0 < y.sum() < len(y)
    out["roc_auc"] = roc_auc_score(y, p_att) if ok else np.nan
    out["pr_auc"] = average_precision_score(y, p_att) if ok else np.nan
    out["recall_at_5pct_fpr"] = recall_at_fpr(y, p_att)
    present = [c for c in CLASSES3 if c in set(truth3)]
    out["macro_precision"] = precision_score(truth3, pred3, labels=present, average="macro", zero_division=0)
    out["macro_recall"] = recall_score(truth3, pred3, labels=present, average="macro", zero_division=0)
    out["macro_f1"] = f1_score(truth3, pred3, labels=present, average="macro", zero_division=0)
    for c in CLASSES3:
        if c in present:
            out[f"recall_{c}"] = recall_score(truth3, pred3, labels=[c], average="macro", zero_division=0)
            out[f"precision_{c}"] = precision_score(truth3, pred3, labels=[c], average="macro", zero_division=0)
    return out

rows, pooled = [], {}
def add(model, variant, fold, seed, truth3, classes, proba):
    classes = list(classes)
    pred3 = np.array(classes)[np.asarray(proba).argmax(1)]
    p_att = 1.0 - np.asarray(proba)[:, classes.index("normal")]
    m = all_metrics(truth3, pred3, p_att, pred3 != "normal")
    rows.append({"model": model, "variant": variant, "fold": fold, "seed": seed, **m})
    if seed == SEEDS[0]:
        pooled.setdefault((model, variant), []).append((np.asarray(truth3), pred3))

# ------------------------------------------------------------------ XGBoost (baseline = raw physical, all rows)
XGB_VARIANTS = {"baseline_all_rows": (False, False), "physical_covered_rows": (True, False),
                "physical_plus_network": (True, True)}
for variant, (cov_only, use_net) in XGB_VARIANTS.items():
    for held in DEV_FOLDS:
        names = [n for n in DEV_FOLDS if n != held] + ["phy_norm"]
        tr, te = pd.concat([data[n] for n in names]), data[held]
        if cov_only:
            tr, te = tr[tr["covered"]], te[te["covered"]]
        cols = list(feature_cols) + (list(NET_COLS) if use_net else [])
        classes = sorted(set(tr["y"]))
        idx = {c: i for i, c in enumerate(classes)}
        ytr = tr["y"].map(idx).to_numpy()
        for seed in SEEDS:
            m = xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1, subsample=0.8,
                                  colsample_bytree=0.8, tree_method="hist", n_jobs=-1, random_state=seed)
            m.fit(tr[cols].astype(float), ytr, sample_weight=compute_sample_weight("balanced", ytr))
            add("XGBoost", variant, held, seed, te["y"].to_numpy(), classes, m.predict_proba(te[cols].astype(float)))
    print("done: XGBoost", variant)

# ------------------------------------------------------------------ LSTM (re-trained so predictions are available)
if RUN_LSTM and "build" in globals() and "train_predict" in globals():
    for variant in ["raw", "scaled", "phys_scaled_cc", "net_scaled_cc"]:
        for held in DEV_FOLDS:
            Xtr, ytr, Xte, yte = build(variant, held)
            for seed in SEEDS:
                classes, proba = train_predict(Xtr, ytr, Xte, seed)
                add("LSTM", variant, held, seed, yte, classes, proba)
        print("done: LSTM", variant)
else:
    print("LSTM skipped (RUN_LSTM is False or Step 16 has not been run in this kernel).")

# ------------------------------------------------------------------ Qwen (from the saved Step 17 replies; no new calls)
qpath = WORK_DIR / "step17_qwen_results.csv"
if qpath.exists():
    q = pd.read_csv(qpath)
    for (variant, fold), d in q.groupby(["variant", "fold"]):
        truth3 = d["truth"].to_numpy()
        pred3 = np.array([c if c in CLASSES3 else "normal" for c in d["cls"].fillna("normal")])
        score = d["score"].fillna(50.0).to_numpy()
        m = all_metrics(truth3, pred3, score / 100.0, score >= QWEN_CUTOFF)
        rows.append({"model": "Qwen3-32B", "variant": variant, "fold": fold, "seed": 0, **m})
        pooled.setdefault(("Qwen3-32B", variant), []).append((truth3, np.where(score >= QWEN_CUTOFF, pred3, "normal")))
    print("done: Qwen (from saved replies; binary decision = score >= %d; 3-class uses Qwen's own class)" % QWEN_CUTOFF)
else:
    print("Qwen skipped: step17_qwen_results.csv not found.")

res = pd.DataFrame(rows)
res.to_csv(WORK_DIR / "step18_all_metrics_by_seed.csv", index=False)
num = [c for c in res.columns if c not in ("model", "variant", "fold", "seed")]
mean = res.groupby(["model", "variant", "fold"])[num].mean()
sd = res.groupby(["model", "variant", "fold"])[["f1", "recall", "precision", "roc_auc"]].std().add_suffix("_sd")
full = mean.join(sd).reset_index()
full.to_csv(WORK_DIR / "step18_all_metrics.csv", index=False)

pd.set_option("display.width", 260)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
A = ["model", "variant", "fold", "n", "attack_share", "majority_acc", "accuracy", "balanced_acc",
     "precision", "recall", "specificity", "f1", "roc_auc", "pr_auc", "recall_at_5pct_fpr"]
print("\nA) ATTACK vs NORMAL at each model's own decision rule "
      "(XGBoost/LSTM: argmax class is not normal; Qwen: score >= %d). Means over seeds." % QWEN_CUTOFF)
print("   Compare accuracy with majority_acc (always predicting the larger class): accuracy alone can mislead.")
display(full[A].round(3))
B = ["model", "variant", "fold", "macro_precision", "macro_recall", "macro_f1"] + \
    [c for c in full.columns if c.startswith(("recall_", "precision_")) and c != "recall_at_5pct_fpr"]
print("\nB) THREE-CLASS (normal / physical_fault / cyberattack): macro and per-class precision and recall")
display(full[B].round(3))
print("\nC) AVERAGE OVER THE THREE FOLDS (simple mean of the fold means)")
display(full.groupby(["model", "variant"])[["accuracy", "balanced_acc", "precision", "recall", "specificity",
                                           "f1", "roc_auc", "pr_auc", "recall_at_5pct_fpr", "macro_f1"]]
        .mean().round(3))
print("\nD) POOLED CONFUSION MATRICES (attack vs normal, first seed, all three folds together)")
for (model, variant), parts in pooled.items():
    t = np.concatenate([p[0] for p in parts]); p_ = np.concatenate([p[1] for p in parts])
    cm = confusion_matrix(t != "normal", p_ != "normal", labels=[False, True])
    print(f"  {model} / {variant}: TN={cm[0,0]} FP={cm[0,1]} FN={cm[1,0]} TP={cm[1,1]}")
print("\nStep 18 complete. No verdicts changed. phy_att_4 was not used.")

done: XGBoost baseline_all_rows
done: XGBoost physical_covered_rows
done: XGBoost physical_plus_network
done: LSTM raw
done: LSTM scaled
done: LSTM phys_scaled_cc
done: LSTM net_scaled_cc
done: Qwen (from saved replies; binary decision = score >= 50; 3-class uses Qwen's own class)

A) ATTACK vs NORMAL at each model's own decision rule (XGBoost/LSTM: argmax class is not normal; Qwen: score >= 50). Means over seeds.
   Compare accuracy with majority_acc (always predicting the larger class): accuracy alone can mislead.


,model,variant,fold,n,attack_share,majority_acc,accuracy,balanced_acc,precision,recall,specificity,f1,roc_auc,pr_auc,recall_at_5pct_fpr
0,LSTM,net_scaled_cc,phy_att_1,180.0,0.272,0.728,0.639,0.522,0.327,0.265,0.779,0.281,0.447,0.355,0.116
1,LSTM,net_scaled_cc,phy_att_2,2087.0,0.147,0.853,0.593,0.523,0.159,0.424,0.622,0.231,0.581,0.158,0.027
2,LSTM,net_scaled_cc,phy_att_3,1172.0,0.284,0.716,0.724,0.545,0.549,0.131,0.959,0.211,0.715,0.536,0.150
3,LSTM,phys_scaled_cc,phy_att_1,180.0,0.272,0.728,0.635,0.441,0.032,0.014,0.868,0.019,0.285,0.192,0.000
4,LSTM,phys_scaled_cc,phy_att_2,2087.0,0.147,0.853,0.600,0.495,0.142,0.346,0.643,0.201,0.528,0.175,0.115
5,LSTM,phys_scaled_cc,phy_att_3,1172.0,0.284,0.716,0.707,0.510,0.380,0.052,0.967,0.091,0.592,0.338,0.072
6,LSTM,raw,phy_att_1,2411.0,0.336,0.664,0.607,0.539,0.397,0.330,0.747,0.360,0.578,0.420,0.086
7,LSTM,raw,phy_att_2,2095.0,0.146,0.854,0.459,0.483,0.139,0.517,0.449,0.218,0.500,0.137,0.001
8,LSTM,raw,phy_att_3,1245.0,0.276,0.724,0.574,0.465,0.225,0.224,0.707,0.224,0.466,0.251,0.031
9,LSTM,scaled,phy_att_1,2411.0,0.336,0.664,0.604,0.522,0.376,0.275,0.770,0.317,0.586,0.404,0.066



B) THREE-CLASS (normal / physical_fault / cyberattack): macro and per-class precision and recall


,model,variant,fold,macro_precision,macro_recall,macro_f1,recall_cyberattack,precision_cyberattack,recall_normal,precision_normal,recall_physical_fault,precision_physical_fault,recall_sd,precision_sd
0,LSTM,net_scaled_cc,phy_att_1,0.703,0.522,0.562,0.265,0.667,0.779,0.740,NaN,NaN,0.276,0.284
1,LSTM,net_scaled_cc,phy_att_2,0.345,0.380,0.326,0.168,0.076,0.622,0.864,0.350,0.096,0.130,0.029
2,LSTM,net_scaled_cc,phy_att_3,0.402,0.378,0.362,0.175,0.469,0.959,0.736,0.000,0.000,0.048,0.121
3,LSTM,phys_scaled_cc,phy_att_1,0.517,0.441,0.401,0.014,0.333,0.868,0.702,NaN,NaN,0.024,0.055
4,LSTM,phys_scaled_cc,phy_att_2,0.329,0.367,0.312,0.051,0.031,0.643,0.852,0.407,0.103,0.067,0.011
5,LSTM,phys_scaled_cc,phy_att_3,0.413,0.355,0.328,0.088,0.385,0.967,0.720,0.009,0.133,0.019,0.105
6,LSTM,raw,phy_att_1,0.404,0.396,0.397,0.258,0.315,0.747,0.688,0.184,0.209,0.043,0.037
7,LSTM,raw,phy_att_2,0.347,0.395,0.296,0.545,0.133,0.449,0.843,0.192,0.065,0.124,0.020
8,LSTM,raw,phy_att_3,0.364,0.371,0.362,0.328,0.247,0.707,0.705,0.079,0.139,0.031,0.016
9,LSTM,scaled,phy_att_1,0.392,0.379,0.381,0.306,0.410,0.770,0.677,0.061,0.087,0.047,0.032



C) AVERAGE OVER THE THREE FOLDS (simple mean of the fold means)


accuracy  balanced_acc  precision  recall  specificity     f1  roc_auc  pr_auc  recall_at_5pct_fpr  macro_f1
model     variant                                                                                                                            
LSTM      net_scaled_cc             0.652         0.530      0.345   0.273        0.787  0.241    0.581   0.350               0.098     0.417
          phys_scaled_cc            0.647         0.482      0.185   0.137        0.826  0.104    0.468   0.235               0.063     0.347
          raw                       0.547         0.496      0.254   0.357        0.635  0.267    0.514   0.269               0.039     0.352
          scaled                    0.590         0.547      0.301   0.424        0.671  0.322    0.611   0.334               0.081     0.395
Qwen3-32B Q_raw                     0.548         0.513      0.378   0.062        0.964  0.105    0.529   0.487               0.060     0.301
          Q_ref                     0.549         0.518      0.391   0.222        0.813  0.280    0.463   0.491               0.000     0.289
          Q_ref_net                 0.575         0.558      0.534   0.622        0.493  0.543    0.596   0.575               0.082     0.370
XGBoost   baseline_all_rows         0.724         0.572      0.409   0.244        0.899  0.288    0.643   0.378               0.163     0.424
          physical_covered_rows     0.782         0.559      0.437   0.130        0.989  0.174    0.632   0.452               0.353     0.426
          physical_plus_network     0.796         0.624      0.714   0.281        0.966  0.339    0.782   0.587               0.452     0.539


D) POOLED CONFUSION MATRICES (attack vs normal, first seed, all three folds together)
  XGBoost / baseline_all_rows: TN=3885 FP=434 FN=1148 TP=311
  XGBoost / physical_covered_rows: TN=2721 FP=57 FN=549 TP=139
  XGBoost / physical_plus_network: TN=2596 FP=182 FN=445 TP=243
  LSTM / raw: TN=2464 FP=1828 FN=920 TP=539
  LSTM / scaled: TN=2427 FP=1865 FN=873 TP=586
  LSTM / phys_scaled_cc: TN=2075 FP=676 FN=561 TP=127
  LSTM / net_scaled_cc: TN=1958 FP=793 FN=488 TP=200
  Qwen3-32B / Q_raw: TN=217 FP=8 FN=185 TP=14
  Qwen3-32B / Q_ref: TN=183 FP=42 FN=149 TP=50
  Qwen3-32B / Q_ref_net: TN=111 FP=114 FN=66 TP=133

Step 18 complete. No verdicts changed. phy_att_4 was not used.


In [38]:
# STEP 19a: Clock-offset check between physical and network timestamps
# Label-free and model-free: it only counts how many physical seconds have a network
# record at each time shift. Development acquisitions only (phy_att_4 is never touched).

import numpy as np
import pandas as pd

for v in ["physical_multiclass", "WORK_DIR", "DEV_FOLDS"]:
    assert v in globals(), f"Missing {v}: run the earlier steps first."
assert "phy_att_4" not in DEV_FOLDS

EPOCH = pd.Timestamp("1970-01-01")
ONE_S = pd.Timedelta("1s")
MAX_SHIFT = 2 * 86400            # search +/- 2 days (covers time-zone and date mix-ups)
NAMES = ["phy_norm"] + list(DEV_FOLDS)

def fmt(sec):
    return str(pd.Timestamp(int(sec), unit="s"))

def shift_profile(phys_sec, net_sec, max_shift):
    """counts[k] = number of physical seconds p with (p + shift) in net seconds, shift = k - max_shift"""
    net_sec = np.unique(net_sec)
    counts = np.zeros(2 * max_shift + 1, dtype=np.int64)
    lo = np.searchsorted(net_sec, phys_sec - max_shift, side="left")
    hi = np.searchsorted(net_sec, phys_sec + max_shift, side="right")
    for p, a, b in zip(phys_sec, lo, hi):
        if b > a:
            d = net_sec[a:b] - p + max_shift
            counts[d] += 1
    return counts

rows = []
for name in NAMES:
    ph = physical_multiclass[name]["Time"]
    assert pd.api.types.is_datetime64_any_dtype(ph), "Run the dayfirst patch cell first."
    phys_sec = np.unique(((ph.dt.floor("s") - EPOCH) // ONE_S).to_numpy())
    net = pd.read_csv(WORK_DIR / f"step14a_network_features_{name}.csv", usecols=["sec"])
    net_sec = np.unique(net["sec"].to_numpy())
    counts = shift_profile(phys_sec, net_sec, MAX_SHIFT)
    cov = counts / len(phys_sec)
    zero = cov[MAX_SHIFT]
    cmax = cov.max()
    plateau = np.where(cov >= cmax - 0.001)[0] - MAX_SHIFT          # all shifts that tie for best
    p_lo, p_hi = int(plateau.min()), int(plateau.max())
    outside = np.ones_like(cov, dtype=bool)
    outside[max(0, p_lo + MAX_SHIFT - 5):p_hi + MAX_SHIFT + 6] = False
    second = float(cov[outside].max()) if outside.any() else 0.0    # best coverage away from the plateau
    hours = np.arange(-48, 49) * 3600
    hcov = cov[hours + MAX_SHIFT]
    top_h = [(int(h // 3600), round(float(c), 4)) for h, c in sorted(zip(hours, hcov), key=lambda x: -x[1])[:3]]
    print(f"\n{name}: {len(phys_sec):,} physical seconds, {len(net_sec):,} network seconds")
    print(f"  physical span: {fmt(phys_sec[0])} -> {fmt(phys_sec[-1])}")
    print(f"  network  span: {fmt(net_sec[0])} -> {fmt(net_sec[-1])}")
    print(f"  coverage at shift 0: {zero:.4f}")
    print(f"  best coverage {cmax:.4f} for shifts {p_lo:+d} s to {p_hi:+d} s "
          f"({p_lo / 3600:+.2f} h to {p_hi / 3600:+.2f} h); best elsewhere {second:.4f}")
    print(f"  whole-hour shifts (hours, coverage), top 3: {top_h}")
    rows.append({"acquisition": name, "phys_seconds": len(phys_sec), "net_seconds": len(net_sec),
                 "coverage_shift0": round(float(zero), 4), "best_coverage": round(float(cmax), 4),
                 "best_shift_from_s": p_lo, "best_shift_to_s": p_hi,
                 "best_elsewhere": round(second, 4), "top_whole_hours": str(top_h)})

res = pd.DataFrame(rows)
print("\n", res.to_string(index=False))
res.to_csv(WORK_DIR / "step19a_clock_check.csv", index=False)

print("""
How to read this:
  - best range containing 0 (and coverage already high) -> clocks agree, nothing to fix.
  - a shift is believable only if coverage jumps close to 1.0, the tied range is narrow
    (a few seconds wide, not hundreds), and "best elsewhere" is far lower. A whole-hour
    shift with high coverage points to a time-zone difference.
  - A wide tied range means the capture is simply shorter than the run: no single shift is
    identifiable, and nothing should be corrected.
  - If nothing for phy_att_1 stands out, its network capture simply covers little of the run.
Nothing is modified by this cell. Send me this output before any correction is applied.""")


phy_norm: 3,428 physical seconds, 3,425 network seconds
  physical span: 2021-04-09 11:30:50 -> 2021-04-09 12:27:57
  network  span: 2021-04-09 11:30:52 -> 2021-04-09 12:27:56
  coverage at shift 0: 0.9991
  best coverage 0.9991 for shifts -4 s to +5 s (-0.00 h to +0.00 h); best elsewhere 0.9965
  whole-hour shifts (hours, coverage), top 3: [(0, 0.9991), (-48, 0.0), (-47, 0.0)]

phy_att_1: 2,420 physical seconds, 189 network seconds
  physical span: 2021-04-09 18:23:28 -> 2021-04-09 19:03:47
  network  span: 2021-04-09 18:23:28 -> 2021-04-09 18:26:36
  coverage at shift 0: 0.0781
  best coverage 0.0781 for shifts -2233 s to +2 s (-0.62 h to +0.00 h); best elsewhere 0.0748
  whole-hour shifts (hours, coverage), top 3: [(0, 0.0781), (-48, 0.0), (-47, 0.0)]

phy_att_2: 2,104 physical seconds, 2,096 network seconds
  physical span: 2021-04-19 15:37:12 -> 2021-04-19 16:12:16
  network  span: 2021-04-19 15:37:19 -> 2021-04-19 16:12:14
  coverage at shift 0: 0.9962
  best coverage 0.9962 for

In [39]:
# STEP 19b: Decision-threshold calibration (same rule for the baseline and the fused arm)
# Development folds only; attack_4 untouched. Needs Step 14b and Step 18 run in this kernel.
#
# Frozen before any result: target false-alarm rate = 5% on NORMAL rows.
# For each outer fold the cutoff is chosen using TRAINING acquisitions only:
#   inner fold = hold out one of the other attack acquisitions, train on the remaining attack
#   acquisition + phy_norm, score the held-out one, pool those out-of-sample scores, and take the
#   attack-score cutoff at which 5% of the held-out NORMAL rows are flagged.
# (phy_norm is never an inner test fold: it has no attacks, and scoring it would be session-leaky.)
# The test fold is only scored with that cutoff; nothing is tuned on it.

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.utils.class_weight import compute_sample_weight

for v in ["data", "NET_COLS", "feature_cols", "DEV_FOLDS", "SEEDS", "ALL_CLASSES", "WORK_DIR",
          "all_metrics", "CLASSES3"]:
    assert v in globals(), f"Missing {v}: run Steps 14b and 18 first."
assert "phy_att_4" not in DEV_FOLDS

TARGET_FPR = 0.05
MIN_INNER_NORMAL = 50
VARIANTS = {"baseline_all_rows": (False, False), "physical_covered_rows": (True, False),
            "physical_plus_network": (True, True)}

def fit_proba(tr, te, cols, seed):
    classes = sorted(set(tr["y"]))
    idx = {c: i for i, c in enumerate(classes)}
    ytr = tr["y"].map(idx).to_numpy()
    m = xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1, subsample=0.8,
                          colsample_bytree=0.8, tree_method="hist", n_jobs=-1, random_state=seed)
    m.fit(tr[cols].astype(float), ytr, sample_weight=compute_sample_weight("balanced", ytr))
    return classes, m.predict_proba(te[cols].astype(float))

def attack_score(classes, proba):
    return 1.0 - np.asarray(proba)[:, list(classes).index("normal")]

def decide(classes, proba, flag):
    """3-class labels consistent with the binary decision: flagged rows get the best non-normal class."""
    classes = list(classes)
    p = np.asarray(proba).copy()
    ni = classes.index("normal")
    p_nonnormal = p.copy(); p_nonnormal[:, ni] = -1
    best_attack = np.array(classes)[p_nonnormal.argmax(1)]
    return np.where(flag, best_attack, "normal")

rows, thresholds = [], []
for variant, (cov_only, use_net) in VARIANTS.items():
    cols = list(feature_cols) + (list(NET_COLS) if use_net else [])
    for held in DEV_FOLDS:
        others = [n for n in DEV_FOLDS if n != held]
        sub = lambda d: d[d["covered"]] if cov_only else d
        te = sub(data[held])
        train_all = pd.concat([sub(data[n]) for n in others + ["phy_norm"]])
        for seed in SEEDS:
            # ---- inner calibration (training acquisitions only)
            oof_score, oof_norm = [], []
            for inner in others:
                itr = pd.concat([sub(data[n]) for n in others if n != inner] + [sub(data["phy_norm"])])
                ite = sub(data[inner])
                if len(set(itr["y"])) < 2:
                    continue
                cl, pr = fit_proba(itr, ite, cols, seed)
                oof_score.append(attack_score(cl, pr))
                oof_norm.append((ite["y"] == "normal").to_numpy())
            s, isn = np.concatenate(oof_score), np.concatenate(oof_norm)
            assert isn.sum() >= MIN_INNER_NORMAL, f"{variant}/{held}: only {isn.sum()} inner normal rows"
            thr = float(np.quantile(s[isn], 1 - TARGET_FPR))
            thresholds.append({"variant": variant, "fold": held, "seed": seed, "cutoff": thr,
                               "inner_normal_rows": int(isn.sum())})
            # ---- outer fit and both decision rules on the untouched test fold
            classes, proba = fit_proba(train_all, te, cols, seed)
            sc = attack_score(classes, proba)
            truth = te["y"].to_numpy()
            argmax_pred = np.array(classes)[np.asarray(proba).argmax(1)]
            for rule, flag in [("argmax (Step 18 rule)", argmax_pred != "normal"),
                               ("train-calibrated 5% FPR", sc >= thr)]:
                pred3 = decide(classes, proba, flag)
                rows.append({"variant": variant, "fold": held, "seed": seed, "rule": rule,
                             **all_metrics(truth, pred3, sc, flag)})
    print("done:", variant)

res = pd.DataFrame(rows)
res.to_csv(WORK_DIR / "step19b_threshold_by_seed.csv", index=False)
pd.DataFrame(thresholds).to_csv(WORK_DIR / "step19b_cutoffs.csv", index=False)
num = [c for c in res.columns if c not in ("variant", "fold", "seed", "rule")]
full = res.groupby(["variant", "rule", "fold"])[num].mean().reset_index()
full["realized_fpr"] = 1 - full["specificity"]
full.to_csv(WORK_DIR / "step19b_threshold_metrics.csv", index=False)

pd.set_option("display.width", 240); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 100)
show = ["variant", "rule", "fold", "n", "attack_share", "precision", "recall", "realized_fpr", "f1",
        "balanced_acc", "roc_auc", "recall_at_5pct_fpr"]
print("\nA) PER FOLD (means over seeds). Target false-alarm rate is 5%: compare it with realized_fpr.")
display(full[show].round(3))
cols_avg = ["precision", "recall", "realized_fpr", "f1", "balanced_acc", "roc_auc", "recall_at_5pct_fpr"]
print("\nB) AVERAGE OVER THE THREE FOLDS")
display(full.groupby(["variant", "rule"])[cols_avg].mean().round(3))
print("\nC) SENSITIVITY: average over phy_att_2 and phy_att_3 only (phy_att_1 excluded as a test fold;"
      " decided after seeing results, because its network capture covers only 189 rows)")
display(full[full["fold"] != "phy_att_1"].groupby(["variant", "rule"])[cols_avg].mean().round(3))
print("\nD) Cutoffs chosen from training data only (attack score; 1 = certain attack)")
display(pd.DataFrame(thresholds).groupby(["variant", "fold"])["cutoff"].agg(["mean", "std"]).round(3))
print("\nStep 19b complete. No verdicts changed. phy_att_4 was not used.")

done: baseline_all_rows
done: physical_covered_rows
done: physical_plus_network

A) PER FOLD (means over seeds). Target false-alarm rate is 5%: compare it with realized_fpr.


,variant,rule,fold,n,attack_share,precision,recall,realized_fpr,f1,balanced_acc,roc_auc,recall_at_5pct_fpr
0,baseline_all_rows,argmax (Step 18 rule),phy_att_1,2420.0,0.335,0.486,0.144,0.076,0.222,0.534,0.646,0.112
1,baseline_all_rows,argmax (Step 18 rule),phy_att_2,2104.0,0.145,0.461,0.446,0.089,0.453,0.678,0.725,0.367
2,baseline_all_rows,argmax (Step 18 rule),phy_att_3,1254.0,0.274,0.279,0.143,0.136,0.188,0.503,0.559,0.011
3,baseline_all_rows,train-calibrated 5% FPR,phy_att_1,2420.0,0.335,0.476,0.234,0.130,0.313,0.552,0.646,0.112
4,baseline_all_rows,train-calibrated 5% FPR,phy_att_2,2104.0,0.145,0.566,0.237,0.031,0.334,0.603,0.725,0.367
5,baseline_all_rows,train-calibrated 5% FPR,phy_att_3,1254.0,0.274,0.240,0.113,0.129,0.153,0.492,0.559,0.011
6,physical_covered_rows,argmax (Step 18 rule),phy_att_1,189.0,0.259,0.000,0.000,0.000,0.000,0.500,0.540,0.347
7,physical_covered_rows,argmax (Step 18 rule),phy_att_2,2096.0,0.146,0.643,0.356,0.034,0.458,0.661,0.709,0.407
8,physical_covered_rows,argmax (Step 18 rule),phy_att_3,1181.0,0.282,0.667,0.033,0.000,0.063,0.517,0.647,0.305
9,physical_covered_rows,train-calibrated 5% FPR,phy_att_1,189.0,0.259,0.000,0.000,0.007,0.000,0.496,0.540,0.347



B) AVERAGE OVER THE THREE FOLDS


precision  recall  realized_fpr     f1  balanced_acc  roc_auc  recall_at_5pct_fpr
variant               rule                                                                                                      
baseline_all_rows     argmax (Step 18 rule)        0.409   0.244         0.101  0.288         0.572    0.643               0.163
                      train-calibrated 5% FPR      0.427   0.195         0.097  0.267         0.549    0.643               0.163
physical_covered_rows argmax (Step 18 rule)        0.437   0.130         0.011  0.174         0.559    0.632               0.353
                      train-calibrated 5% FPR      0.276   0.302         0.116  0.274         0.593    0.632               0.353
physical_plus_network argmax (Step 18 rule)        0.714   0.281         0.034  0.339         0.624    0.782               0.452
                      train-calibrated 5% FPR      0.588   0.552         0.209  0.418         0.672    0.782               0.452


C) SENSITIVITY: average over phy_att_2 and phy_att_3 only (phy_att_1 excluded as a test fold; decided after seeing results, because its network capture covers only 189 rows)


precision  recall  realized_fpr     f1  balanced_acc  roc_auc  recall_at_5pct_fpr
variant               rule                                                                                                      
baseline_all_rows     argmax (Step 18 rule)        0.370   0.294         0.113  0.321         0.591    0.642               0.189
                      train-calibrated 5% FPR      0.403   0.175         0.080  0.244         0.548    0.642               0.189
physical_covered_rows argmax (Step 18 rule)        0.655   0.195         0.017  0.261         0.589    0.678               0.356
                      train-calibrated 5% FPR      0.413   0.453         0.170  0.411         0.641    0.678               0.356
physical_plus_network argmax (Step 18 rule)        0.760   0.387         0.043  0.447         0.672    0.828               0.501
                      train-calibrated 5% FPR      0.503   0.764         0.307  0.517         0.729    0.828               0.501


D) Cutoffs chosen from training data only (attack score; 1 = certain attack)


mean    std
variant               fold                   
baseline_all_rows     phy_att_1  0.115  0.007
                      phy_att_2  0.939  0.001
                      phy_att_3  0.602  0.068
physical_covered_rows phy_att_1  0.102  0.013
                      phy_att_2  0.004  0.000
                      phy_att_3  0.001  0.000
physical_plus_network phy_att_1  0.258  0.042
                      phy_att_2  0.006  0.000
                      phy_att_3  0.004  0.001


Step 19b complete. No verdicts changed. phy_att_4 was not used.


In [40]:
# STEP 20: Qwen3 proposes the FUSION preprocessing (network -> per-second features -> joined to physical rows)
# Frozen task: evidence comes from the development acquisitions only (phy_att_4 never read), contains NO model
# results, and the agent may only choose from a fixed menu of operations that Step 21 will implement exactly.
# No critic pass (it missed errors last time). A deterministic validator plus HUMAN ADJUDICATION replaces it.

import json
import re
from collections import Counter
from datetime import datetime

import pandas as pd

for v in ["client", "MODEL_ID", "DEV_FOLDS", "WORK_DIR", "physical_multiclass"]:
    assert v in globals(), f"Missing {v}: run the earlier steps first."
assert "phy_att_4" not in DEV_FOLDS
assert pd.api.types.is_datetime64_any_dtype(physical_multiclass["phy_att_1"]["Time"]), "Run the dayfirst patch cell."

# ---------------- Settings ----------------
N_SAMPLES = 3
ANALYSIS_MAX_TOKENS = 14000
FORMAT_MAX_TOKENS = 5000
TEMPERATURE = 0.6
TOP_P = 0.95
PRINT_TEXT = False          # set True to read each full analysis in the notebook
ALLOWED_METRICS = ["roc_auc_attack_vs_normal", "recall_at_5pct_fpr", "pr_auc_attack_vs_normal",
                   "subtype_detection_rate"]

# ---------------- Helpers (same as Step 13) ----------------
def to_json(obj):
    return json.dumps(obj, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))

def ascii_fix(s):
    for a, b in {"’": "'", "‘": "'", "“": '"', "”": '"', "–": "-", "—": "-",
                 "×": "x", "≥": ">=", "≤": "<=", "→": "->", " ": " "}.items():
        s = s.replace(a, b)
    return s

def stream_chat(messages, max_tokens, temperature, top_p):
    content_parts, reasoning_parts, finish = [], [], None
    try:
        stream = client.chat.completions.create(model=MODEL_ID, messages=messages, tool_choice="none",
                                                temperature=temperature, top_p=top_p,
                                                max_tokens=max_tokens, stream=True)
        for chunk in stream:
            if not chunk.choices:
                continue
            choice = chunk.choices[0]
            delta = choice.delta
            c = getattr(delta, "content", None)
            r = getattr(delta, "reasoning", None) or getattr(delta, "reasoning_content", None)
            if c:
                content_parts.append(c)
            if r:
                reasoning_parts.append(r)
            if getattr(choice, "finish_reason", None):
                finish = choice.finish_reason
    except Exception as e:
        finish = f"error:{type(e).__name__}"
        print("Stream interrupted:", str(e)[:200])
    content = "".join(content_parts)
    inline = re.findall(r"<think>(.*?)</think>", content, flags=re.DOTALL)
    clean = re.sub(r"<think>.*?</think>", "", content, flags=re.DOTALL).strip()
    return ascii_fix(clean), "".join(reasoning_parts) + "\n".join(inline), finish

def parse_json(text):
    s, e = text.find("{"), text.rfind("}")
    if s == -1 or e <= s:
        return None
    try:
        return json.loads(text[s:e + 1])
    except json.JSONDecodeError:
        return None

# ---------------- The per-second network columns that already exist (built in Step 14a, labels not used) ----------------
BASE_FEATURES = [c for c in pd.read_csv(WORK_DIR / "step14a_network_features_phy_norm.csv", nrows=0).columns
                 if c != "sec"]
assert 10 <= len(BASE_FEATURES) <= 30, BASE_FEATURES

# ---------------- The menu (Step 21 will implement exactly these, fitted on training folds only) ----------------
MENU = {
    "rolling_stat":
        {"params": {"stat": ["mean", "std", "max", "min", "sum"], "window_s": [5, 10, 30, 60]},
         "needs_features": True,
         "meaning": "causal statistic over the last window_s seconds (current second included) of the chosen network columns"},
    "diff":
        {"params": {"lag_s": [1, 5]}, "needs_features": True,
         "meaning": "value now minus value lag_s seconds ago"},
    "ratio_to_recent_baseline":
        {"params": {"window_s": [60, 300]}, "needs_features": True,
         "meaning": "value now divided by (median of the previous window_s seconds + small constant); causal, no session identity"},
    "log1p":
        {"params": {}, "needs_features": True, "meaning": "log(1 + x) on non-negative columns"},
    "robust_scale_network":
        {"params": {}, "needs_features": False,
         "meaning": "subtract median and divide by IQR of ALL network columns, statistics from training normal rows only"},
    "missing_handling":
        {"params": {"method": ["keep_nan", "ffill_causal"], "max_gap_s": [1, 2, 5]}, "needs_features": False,
         "meaning": "what to do for physical seconds with no network packets: leave missing, or carry the last value forward up to max_gap_s seconds"},
    "alignment_tolerance":
        {"params": {"tolerance_s": [0, 1, 2]}, "needs_features": False,
         "meaning": "join a physical second to the nearest network second within tolerance_s seconds (0 = exact second only)"},
    "drop_features":
        {"params": {}, "needs_features": True, "meaning": "remove the chosen base network columns from the fused table"},
    "function_entropy":
        {"params": {}, "needs_features": False,
         "meaning": "add one column: entropy of the Modbus-function share columns in that second"},
}

# ---------------- Evidence (development data only; no model results) ----------------
profile = json.loads((WORK_DIR / "step12_data_profile.json").read_text(encoding="utf-8"))
net_profile = json.loads((WORK_DIR / "step12b_network_profile.json").read_text(encoding="utf-8"))
SEALED = re.compile(r"att(ack)?_?4", re.IGNORECASE)

def scrub(obj, log, path=""):
    if isinstance(obj, dict):
        out = {}
        for k, v in obj.items():
            if SEALED.search(str(k)) or (isinstance(v, str) and SEALED.search(v)):
                log.append(f"{path}/{k}")
                continue
            out[k] = scrub(v, log, f"{path}/{k}")
        return out
    if isinstance(obj, list):
        return [scrub(v, log, path) for v in obj if not (isinstance(v, str) and SEALED.search(v))]
    return obj

scrub_log = []
by_kind = {}
for r in profile["per_feature"]:
    by_kind.setdefault(r["kind"], []).append(r["feature"])
evidence = {
    "task": ("Design the FUSION preprocessing that turns packet-level network data into per-second features and "
             "joins them to the per-second physical rows. Only the fusion stage is in scope."),
    "current_fusion_design_to_improve_or_keep": {
        "per_second_network_columns": BASE_FEATURES,
        "how_built": ("packets grouped by the second of their timestamp; counts, size statistics, share of "
                      "responses with a null response field, number of distinct source ports, destination ports, "
                      "source IPs and destination IPs, and the share of packets for each of the 8 most common "
                      "Modbus functions (the rest pooled as net_fn_other)"),
        "join": "exact epoch-second match with the physical row; labels come from the physical file",
        "missing_network_second": "left missing for XGBoost; the row is excluded for the LSTM and for Qwen",
        "excluded_by_researchers": ["a network-availability flag (coverage shortcut)",
                                    "any acquisition or session identifier"],
    },
    "menu_of_allowed_operations": MENU,
    "physical_feature_names_by_kind": by_kind,
    "network_packet_profile_development_acquisitions": net_profile,
    "timestamp_alignment_physical_vs_network": {
        "note": ("Physical rows are 1 per second. Share of physical rows that have a network second at the same "
                 "second (within_0s) or within 2 seconds (within_2s)."),
        "phy_norm":  {"physical_rows": 3428, "within_0s": 0.9991, "within_2s": 1.0000},
        "phy_att_1": {"physical_rows": 2420, "within_0s": 0.0781, "within_2s": 0.0789,
                      "network_seconds_with_packets": 189,
                      "network_capture": "18:23:28 to 18:26:36 only; physical run lasts until 19:03:47"},
        "phy_att_2": {"physical_rows": 2104, "within_0s": 0.9962, "within_2s": 0.9981},
        "phy_att_3": {"physical_rows": 1254, "within_0s": 0.9418, "within_2s": 0.9442,
                      "network_capture_ends_before_physical_s": 72},
        "clock_check": "no constant clock offset exists between physical and network time in any acquisition",
    },
    "label_free_diagnostics_on_current_columns": {
        "session_fingerprinting": ("Using ONLY normal rows, a classifier could tell which acquisition a row came from "
                                   "using the current per-second network columns with ROC-AUC 1.000 for some pairs "
                                   "(including pairs with phy_att_2). The network columns therefore carry a "
                                   "session signature, not only attack information."),
        "pooled_missingness": ("Missing-network status separated attack from normal rows with ROC-AUC 0.73 when all "
                               "acquisitions were pooled, but about 0.50 within a single acquisition: pooled "
                               "coverage is a shortcut."),
    },
    "how_results_will_be_judged": ("After the proposal is implemented it is tested once, per development fold, "
                                   "against the same model without it. A step is kept only if it improves the "
                                   "primary metric in ALL three folds by more than that fold's baseline seed standard "
                                   "deviation. This rule is applied AFTER testing and is not a reason to propose or "
                                   "withhold a step. Note phy_att_1 has only 189 rows with network data."),
    "models_that_must_work_with_the_result": ["XGBoost", "LSTM", "Qwen (text prompts, per row)"],
}
evidence = scrub(evidence, scrub_log)
evidence_text = to_json(evidence)
assert not re.findall(r"[^\n]*att(?:ack)?_?4[^\n]*", evidence_text, flags=re.I), "sealed acquisition in evidence"
ev_numbers = set(re.findall(r"\d+\.\d+|\d{3,}", evidence_text))
print("Removed from the evidence (sealed acquisition):", scrub_log or "nothing")
print(f"Evidence: {len(evidence_text):,} characters (about {len(evidence_text) // 3:,} tokens). Base network columns: {len(BASE_FEATURES)}")

# ---------------- Prompts ----------------
prompt_analysis = f"""
You are the independent preprocessing agent for the WDT physical-network research project.
Review this verified evidence:

{evidence_text}

Design the FUSION preprocessing. You may ONLY use operations from "menu_of_allowed_operations"
and ONLY the base columns in "per_second_network_columns". Work through:

A. OBSERVATIONS: 5 to 7 decision-relevant facts, each with exact numbers and the evidence key.
B. PROBLEMS with the current fusion design that the evidence supports (for example: noisy per-second
   values, session signature, missing seconds, alignment). Say which you can and cannot judge.
C. CANDIDATE STEPS: for each menu operation you consider, say whether to use it, with which
   parameter values and columns, and the evidence for and against. It is allowed to propose NO steps
   if the evidence does not support any.
D. CROSS-CHECK: (1) every step is causal (uses only the current and past seconds) and fitted on
   training folds only; (2) no step uses labels, acquisition identity or a network-availability flag;
   (3) the output works for XGBoost, an LSTM and Qwen; (4) steps do not conflict.
E. FINAL PROPOSAL: an ordered list of steps. For each step: operation, exact parameter values,
   the columns it applies to, the evidence with exact numbers, and a test: the metric
   (one of {ALLOWED_METRICS}), the folds, and a numeric falsification criterion stated in the
   correct direction (the step is rejected if the metric gain is below the stated number).
   Also list menu operations you considered and rejected, with a reason.

Rules:
- Use only numbers that appear in the evidence. Re-check arithmetic. Do not invent facts.
- Never propose rules tied to one acquisition by name. A rule must make sense for any new session.
- Do not use or request phy_att_4. Folds are whole acquisitions.
- Do not propose any step that is not in the menu. Do not propose an availability flag.
- Plain ASCII text only.
"""

def format_prompt(analysis):
    return f"""/no_think
Convert ONLY section E (FINAL PROPOSAL) of the analysis below into one JSON object. Do not add, remove or
reinterpret anything; copy numbers exactly. Plain ASCII. Return only the JSON object.

Keys: "summary" (string), "steps" (list), "rejected" (list), "risks" (list of strings).
Each item of "steps": {{"id": "S1", "operation": one of {list(MENU)}, "params": object using only the parameter
names of that operation, "features": list of base column names or [] when the operation does not take columns,
"rationale": string, "evidence": list of strings each citing exact numbers,
"test": {{"metric": one of {ALLOWED_METRICS}, "folds": {list(DEV_FOLDS)} or a subset (never the word all),
"falsification": string with a numeric criterion}}}}.
Each item of "rejected": {{"operation": string, "reason": string}}.
If the analysis proposes no steps, "steps" is an empty list.

ANALYSIS:
{analysis}
"""

# ---------------- Validator (structure + menu + the traps we saw in Step 13) ----------------
def validate(p, raw):
    problems, warnings = [], []
    if not isinstance(p, dict):
        return ["not a JSON object"], [], 0
    for k in ["summary", "steps", "rejected", "risks"]:
        if k not in p:
            problems.append(f"missing key: {k}")
    if not ascii_fix(raw).isascii():
        problems.append("non-ASCII characters in reply")
    steps = p.get("steps") if isinstance(p.get("steps"), list) else []
    if len(steps) > 8:
        problems.append("more than 8 steps")
    ids = [s.get("id") for s in steps if isinstance(s, dict)]
    if len(set(ids)) != len(ids):
        problems.append("duplicate step ids")
    odd = set()
    for s in steps:
        if not isinstance(s, dict):
            problems.append("step is not an object")
            continue
        sid, op = s.get("id", "?"), s.get("operation")
        if op not in MENU:
            problems.append(f"{sid}: operation '{op}' is not in the menu")
            continue
        spec = MENU[op]
        params = s.get("params") if isinstance(s.get("params"), dict) else {}
        for k, v in params.items():
            if k not in spec["params"]:
                problems.append(f"{sid}: parameter '{k}' is not allowed for {op}")
            elif v not in spec["params"][k]:
                problems.append(f"{sid}: {k}={v} is outside the allowed values {spec['params'][k]}")
        for k in spec["params"]:
            if k not in params:
                problems.append(f"{sid}: missing parameter '{k}'")
        feats = s.get("features")
        if not isinstance(feats, list):
            problems.append(f"{sid}: features must be a list")
        else:
            bad = [f for f in feats if f not in BASE_FEATURES]
            if bad:
                problems.append(f"{sid}: unknown columns {bad}")
            if spec["needs_features"] and not feats:
                problems.append(f"{sid}: {op} needs at least one column")
        ev = s.get("evidence")
        if not isinstance(ev, list) or not ev or any(not re.search(r"\d", str(e)) for e in ev):
            problems.append(f"{sid}: evidence must be a non-empty list citing numbers")
        else:
            for e in ev:
                odd |= {n for n in re.findall(r"\d+\.\d+|\d{3,}", str(e)) if n not in ev_numbers}
        t = s.get("test")
        if not isinstance(t, dict):
            problems.append(f"{sid}: missing test object")
        else:
            if t.get("metric") not in ALLOWED_METRICS:
                problems.append(f"{sid}: metric not in the allowed list")
            f = t.get("folds")
            if isinstance(f, list) and any(re.fullmatch(r"(?i)all( development)?( folds)?", str(x).strip()) for x in f):
                t["folds"] = f = list(DEV_FOLDS)
                warnings.append(f"{sid}: 'all folds' expanded to the development folds")
            if not isinstance(f, list) or not f or not set(f) <= set(DEV_FOLDS):
                problems.append(f"{sid}: folds must be a subset of the development folds")
            fal = str(t.get("falsification", ""))
            if not re.search(r"\d", fal):
                problems.append(f"{sid}: falsification has no numeric criterion")
            if re.search(r"(?i)\bexceed|greater than|more than|above\b", fal) and re.search(r"(?i)reject|falsif", fal):
                warnings.append(f"{sid}: check the falsification direction by hand: '{fal[:90]}'")
    blob = json.dumps(p).lower()
    if SEALED.search(blob):
        warnings.append("mentions the sealed acquisition: confirm it is only a prohibition")
    if re.search(r"has_network|availab|coverage (flag|indicator)|missing(ness)? (flag|indicator)|is_missing", blob):
        warnings.append("mentions an availability/missingness flag (confirmed coverage shortcut)")
    if re.search(r"phy_att_[123]|phy_norm", " ".join(str(s.get("params", "")) + str(s.get("rationale", "")) for s in steps if isinstance(s, dict)).lower()):
        warnings.append("a step's rationale or parameters name a specific acquisition: check it generalizes to a new session")
    if re.search(r"acquisition id|session id|session_id|acquisition_id", blob):
        warnings.append("acquisition or session identity as a feature")
    if odd:
        warnings.append("numbers in evidence items not found verbatim in the evidence: " + str(sorted(odd)[:8]))
    return problems, warnings, len(steps)

# ---------------- Frozen record, written BEFORE the first call ----------------
(WORK_DIR / "step20_frozen_task.md").write_text(
    f"# Step 20 frozen task\nWritten {datetime.now().isoformat(timespec='seconds')} before any call.\n"
    f"- Scope: fusion preprocessing only; operations restricted to the menu below.\n"
    f"- Evidence: development acquisitions only; no model results; sealed acquisition removed ({len(scrub_log)} items).\n"
    f"- Samples: {N_SAMPLES}; model {MODEL_ID}; temperature {TEMPERATURE}; no critic pass.\n"
    f"- Next: human adjudication of every step, then Step 21 implements the ACCEPTED steps and tests them once "
    f"with the Step 14 rule.\n- Menu: {json.dumps(list(MENU))}\n- Base columns: {BASE_FEATURES}\n", encoding="utf-8")
(WORK_DIR / "step20_evidence.json").write_text(evidence_text, encoding="utf-8")
(WORK_DIR / "step20_prompt.txt").write_text(prompt_analysis, encoding="utf-8")

# ---------------- Run ----------------
results, adjud = [], []
for i in range(1, N_SAMPLES + 1):
    print("\n" + "=" * 60 + f"\nSAMPLE {i} of {N_SAMPLES}\n" + "=" * 60)
    analysis, reasoning, finish = stream_chat(
        [{"role": "system", "content": "You are an independent preprocessing research agent. Base every claim on the supplied evidence and check your arithmetic."},
         {"role": "user", "content": prompt_analysis}], ANALYSIS_MAX_TOKENS, TEMPERATURE, TOP_P)
    print(f"[analysis] finish={finish} reasoning_chars={len(reasoning)} text_chars={len(analysis)}")
    (WORK_DIR / f"step20_sample{i}_analysis.txt").write_text(analysis, encoding="utf-8")
    problems, warnings, plan, n_steps = [], [], None, 0
    if finish != "stop" or not analysis:
        problems.append(f"analysis incomplete (finish={finish}); raise ANALYSIS_MAX_TOKENS and rerun")
    else:
        if PRINT_TEXT:
            print(analysis)
        fmt_text, _, ffinish = stream_chat(
            [{"role": "system", "content": "You convert a written proposal into JSON. You never add, remove or change content."},
             {"role": "user", "content": format_prompt(analysis)}], FORMAT_MAX_TOKENS, 0.0, 1.0)
        plan = parse_json(fmt_text)
        if plan is None:
            problems.append(f"no parseable JSON from the formatting call (finish={ffinish})")
        else:
            problems, warnings, n_steps = validate(plan, fmt_text)
            (WORK_DIR / f"step20_sample{i}_plan.json").write_text(to_json(plan), encoding="utf-8")
    print("VALIDATION problems:", problems or "none")
    print("VALIDATION warnings:", warnings or "none")
    results.append({"i": i, "plan": plan, "problems": problems, "warnings": warnings, "n_steps": n_steps})
    for s in (plan or {}).get("steps", []) if isinstance(plan, dict) else []:
        if isinstance(s, dict):
            adjud.append({"sample": i, "id": s.get("id"), "operation": s.get("operation"),
                          "params": json.dumps(s.get("params")), "columns": ",".join(s.get("features") or []),
                          "metric": (s.get("test") or {}).get("metric"),
                          "falsification": (s.get("test") or {}).get("falsification"),
                          "rationale": s.get("rationale"),
                          "evidence": " | ".join(map(str, s.get("evidence") or [])),
                          "human_decision": "", "human_reason": ""})

# ---------------- Review tables ----------------
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 90); pd.set_option("display.max_rows", 200)
print("\n" + "=" * 60 + "\nSAMPLES AT A GLANCE\n" + "=" * 60)
display(pd.DataFrame([{"sample": r["i"], "steps": r["n_steps"], "problems": len(r["problems"]),
                       "warnings": len(r["warnings"]),
                       "operations": ", ".join(s.get("operation", "?") for s in ((r["plan"] or {}).get("steps", []) if r["plan"] else []) if isinstance(s, dict)) or "none"}
                      for r in results]))
ad = pd.DataFrame(adjud)
if len(ad):
    print("\nEVERY PROPOSED STEP (to be adjudicated; human_decision is filled in together, not by the model):")
    display(ad[["sample", "id", "operation", "params", "columns", "metric", "falsification"]])
    print("\nWHERE THE SAMPLES AGREE (same operation, columns and parameters):")
    key = ad["operation"] + " | " + ad["params"] + " | " + ad["columns"]
    agree = key.value_counts()
    display(agree[agree > 1].rename("samples proposing it").to_frame())
    ad.to_csv(WORK_DIR / "step20_adjudication_template.csv", index=False)
else:
    print("\nNo steps were proposed in any sample: the agent found no supported fusion change.")
rej = [(r["i"], x.get("operation"), x.get("reason")) for r in results if isinstance(r["plan"], dict)
       for x in r["plan"].get("rejected", []) if isinstance(x, dict)]
if rej:
    print("\nREJECTED BY THE AGENT:")
    display(pd.DataFrame(rej, columns=["sample", "operation", "reason"]))
print("\nNothing has been implemented or tested. Paste this output; we adjudicate step by step, then freeze (Step 21).")

Removed from the evidence (sealed acquisition): nothing
Evidence: 32,366 characters (about 10,788 tokens). Base network columns: 14

SAMPLE 1 of 3
[analysis] finish=stop reasoning_chars=13096 text_chars=4961
VALIDATION problems: ["S1: unknown columns ['all']", "S2: unknown columns ['all']", "S5: unknown columns ['all']", 'S5: evidence must be a non-empty list citing numbers']
VALIDATION warnings: ["numbers in evidence items not found verbatim in the evidence: ['7.89']"]

SAMPLE 2 of 3
[analysis] finish=stop reasoning_chars=7438 text_chars=5572
VALIDATION problems: ['non-ASCII characters in reply', "S3: parameter 'columns' is not allowed for drop_features", "S4: unknown columns ['net_fn_*']", 'S4: evidence must be a non-empty list citing numbers']
VALIDATION warnings: ["S1: 'all folds' expanded to the development folds", "S2: 'all folds' expanded to the development folds", "S3: 'all folds' expanded to the development folds", "S4: 'all folds' expanded to the development folds", "S5: 'all

,sample,steps,problems,warnings,operations
0,1,5,4,1,"alignment_tolerance, missing_handling, ratio_to_recent_baseline, function_entropy, rob..."
1,2,7,4,8,"alignment_tolerance, missing_handling, drop_features, function_entropy, robust_scale_n..."
2,3,6,5,1,"alignment_tolerance, missing_handling, robust_scale_network, ratio_to_recent_baseline,..."



EVERY PROPOSED STEP (to be adjudicated; human_decision is filled in together, not by the model):


,sample,id,operation,params,columns,metric,falsification
0,1,S1,alignment_tolerance,"{""tolerance_s"": 2}",all,roc_auc_attack_vs_normal,gain < 0.05
1,1,S2,missing_handling,"{""method"": ""ffill_causal"", ""max_gap_s"": 2}",all,recall_at_5pct_fpr,gain < 0.03
2,1,S3,ratio_to_recent_baseline,"{""window_s"": 60}","net_size_mean,net_pkts",pr_auc_attack_vs_normal,gain < 0.04
3,1,S4,function_entropy,{},,subtype_detection_rate,gain < 0.10
4,1,S5,robust_scale_network,{},all,roc_auc_attack_vs_normal,gain < 0.05
5,2,S1,alignment_tolerance,"{""tolerance_s"": 2}",,roc_auc_attack_vs_normal,gain < 0.01
6,2,S2,missing_handling,"{""method"": ""ffill_causal"", ""max_gap_s"": 2}",,pr_auc_attack_vs_normal,gain < 0.02
7,2,S3,drop_features,"{""columns"": [""net_distinct_sport"", ""net_distinct_dport"", ""net_distinct_ip_s"", ""net_dis...","net_distinct_sport,net_distinct_dport,net_distinct_ip_s,net_distinct_ip_d",roc_auc_attack_vs_normal,gain < 0.03
8,2,S4,function_entropy,{},net_fn_*,recall_at_5pct_fpr,gain < 0.05
9,2,S5,robust_scale_network,{},,roc_auc_attack_vs_normal,gain < 0.02



WHERE THE SAMPLES AGREE (same operation, columns and parameters):


,samples proposing it
function_entropy | {} |,2



REJECTED BY THE AGENT:


,sample,operation,reason
0,1,rolling_stat,Label-mixing risk.
1,1,log1p,Size distribution already narrow.
2,2,log1p,"Not needed; most columns are non-negative but already stable (e.g., size_mean has std ..."
3,2,rolling_stat(window_s=10),Over-smoothing; attack detection requires short-term dynamics.
4,3,log1p,Size distribution has no extreme outliers (A.4)
5,3,rolling_stat,Attack-specific size spikes are meaningful signals (A.4)
6,3,diff,Per-second changes may not add value for attack detection



Nothing has been implemented or tested. Paste this output; we adjudicate step by step, then freeze (Step 21).


In [41]:
import json
for i in (1, 2, 3):
    p = json.load(open(WORK_DIR / f"step20_sample{i}_plan.json"))
    for s in p["steps"]:
        ev = " | ".join(map(str, s.get("evidence", [])))
        if any(x in ev for x in ["2268", "2276", "77.53", "795", "046", "7.89", "78.9"]):
            print(i, s["id"], s["operation"], "->", ev, "\n")

1 S1 alignment_tolerance -> phy_att_1's 7.89% coverage 

2 S1 alignment_tolerance -> phy_att_1/3 coverage improves to 78.9%/94.4% within 2s. 

2 S5 robust_scale_network -> packet counts differ across acquisitions (normal: 2268 vs attack_3: 2276). 

3 S4 ratio_to_recent_baseline -> Attack_3's size_mean=795 (A.4) 

3 S5 function_entropy -> Attack_3 has 77.53% null responses (A.6) 

3 S6 drop_features -> Attack_2 has 25,046 distinct sports (A.5) 



In [42]:
# STEP 21: Test the ADJUDICATED fusion steps from Step 20 (XGBoost). Development folds only; phy_att_4 untouched.
# Needs Step 14b in this kernel (data, NET_COLS, feature_cols, DEV_FOLDS, SEEDS, ALL_CLASSES, WORK_DIR).
# Rule frozen before any result (same as Step 14): a step is KEPT only if its ROC-AUC (attack vs normal) beats the paired
# baseline in ALL three folds by more than that fold's baseline seed SD. Sensitivity run: phy_att_2 and phy_att_3 only.

import json
from datetime import datetime

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score, roc_curve
from sklearn.utils.class_weight import compute_sample_weight

for v in ["data", "NET_COLS", "feature_cols", "DEV_FOLDS", "SEEDS", "ALL_CLASSES", "WORK_DIR"]:
    assert v in globals(), f"Missing {v}: run Step 14b first."
assert "phy_att_4" not in DEV_FOLDS

EPOCH, ONE_S = pd.Timestamp("1970-01-01"), pd.Timedelta("1s")
RATIO_COLS = ["net_pkts", "net_size_mean", "net_size_std", "net_size_max", "net_resp_null_share"]
DROP_COLS = ["net_distinct_sport", "net_distinct_dport"]
FN_COLS = [c for c in NET_COLS if c.startswith("net_fn_")]
RATIO_WINDOW, RATIO_MIN_PERIODS = 60, 10
assert all(c in NET_COLS for c in RATIO_COLS + DROP_COLS) and FN_COLS

# ------------------------------------------------------------ the adjudicated decisions (frozen record)
ADJUDICATION = [
    ("alignment_tolerance 2 s", "3 of 3", "ACCEPT", "cheap, shared; expected near-null (coverage gain 0.0008 to 0.0024). NOTE: uses up to 2 future seconds, so it is NOT causal (offline only)."),
    ("missing_handling ffill_causal 2 s", "3 of 3", "ACCEPT", "causal, low risk; affects only short gaps."),
    ("ratio_to_recent_baseline 60 s", "3 of 3", "MODIFY", "columns differed per sample; human took the union (net_pkts, net_size_mean/std/max, net_resp_null_share); ratios replace the raw columns. Risk: a long attack becomes its own baseline."),
    ("function_entropy", "3 of 3", "ACCEPT", "one extra column, no leakage risk."),
    ("robust_scale_network", "3 of 3", "ACCEPT for LSTM only", "XGBoost trees are scale-invariant; Qwen reads text. Tested in Step 22."),
    ("drop_features sport+dport", "1 of 3", "ACCEPT narrow version", "sample 3 version; sample 2's four-column version rejected (IP counts likely carry MITM/scan signal)."),
    ("rolling_stat mean 5 s", "1 of 3", "REJECT", "two samples rejected rolling statistics; physical window test already failed."),
]
ERRORS_FOUND_BY_HUMAN = [
    "Sample 2, S1: wrote 78.9% for phy_att_1 within-2s coverage; the evidence says 7.89% (tenfold slip).",
    "Sample 2, S5: cited packet counts 2268 vs 2276 as 'differing'; the numbers are not in the evidence and barely differ.",
    "Samples 1-3: 'all' / 'net_fn_*' used as column lists, parameters nested incorrectly; normalized mechanically.",
    "Falsification criteria in sample 3 read in the opposite direction to samples 1-2; the frozen rule decides, not Qwen's thresholds.",
]
(WORK_DIR / "step21_frozen_plan.md").write_text(
    f"# Step 21 frozen plan\nWritten {datetime.now().isoformat(timespec='seconds')} BEFORE any result.\n\n"
    "## Adjudication of Qwen's fusion proposals (human)\n" +
    "\n".join(f"- {a} [{b}] -> {c}: {d}" for a, b, c, d in ADJUDICATION) +
    "\n\n## Qwen errors found by human review\n" + "\n".join(f"- {e}" for e in ERRORS_FOUND_BY_HUMAN) +
    "\n\n## Test design\n- Paired baseline: physical + the 14 raw network columns, exact-second join.\n"
    "- Evaluation rows for EVERY variant: seconds that have network data within 2 s (superset of exact coverage), so all variants are scored on the same rows; baseline keeps NaN where its exact join finds nothing.\n"
    "- Training rows: the same row set, other development acquisitions + phy_norm.\n"
    f"- Ratio step: median of the previous {RATIO_WINDOW} seconds (>= {RATIO_MIN_PERIODS} observed), denominator + 0.01 x training q99 of the column.\n"
    "- Rule: ROC-AUC gain > baseline seed SD in all three folds. Secondary: recall at 5% FPR, PR-AUC, cyberattack recall.\n"
    "- Sensitivity: phy_att_2 and phy_att_3 only (phy_att_1 has about 189 network rows).\n",
    encoding="utf-8")
pd.DataFrame(ADJUDICATION, columns=["step", "samples", "decision", "reason"]).to_csv(
    WORK_DIR / "step21_adjudication_final.csv", index=False)

# ------------------------------------------------------------ per-second network timelines
def load_timeline(name):
    net = pd.read_csv(WORK_DIR / f"step14a_network_features_{name}.csv").set_index("sec")
    assert list(net.columns) == list(NET_COLS), f"{name}: network columns differ from NET_COLS"
    phys = ((data[name]["Time"].dt.floor("s") - EPOCH) // ONE_S).to_numpy()
    lo = int(min(net.index.min(), phys.min())) - RATIO_WINDOW
    hi = int(max(net.index.max(), phys.max())) + 3
    full = net.reindex(np.arange(lo, hi + 1))
    med = full[RATIO_COLS].shift(1).rolling(RATIO_WINDOW, min_periods=RATIO_MIN_PERIODS).median().add_suffix("_med")
    return pd.concat([full, med], axis=1), phys

def align(tl, tol, ff):
    f = tl
    if tol > 0:                                   # nearest second within +/- tol (uses future seconds: not causal)
        f = tl.copy()
        for s in range(1, tol + 1):
            f = f.combine_first(tl.shift(s)).combine_first(tl.shift(-s))
    if ff > 0:                                    # causal carry-forward
        f = f.ffill(limit=ff)
    return f

ALL_NAMES = ["phy_norm"] + list(DEV_FOLDS)
TL = {n: load_timeline(n) for n in ALL_NAMES}
NF = {}
for n in ALL_NAMES:
    tl, phys = TL[n]
    for tol, ff in [(0, 0), (2, 0), (0, 2), (2, 2)]:
        f = align(tl, tol, ff).reindex(phys)
        f.index = data[n].index
        NF[(n, tol, ff)] = f
    exact = NF[(n, 0, 0)][NET_COLS]
    ref = data[n][NET_COLS]
    assert np.allclose(exact.to_numpy(float), ref.to_numpy(float), equal_nan=True), \
        f"{n}: exact join does not reproduce the Step 14b network columns"
MASK = {n: NF[(n, 2, 0)]["net_pkts"].notna().to_numpy() for n in ALL_NAMES}
print("Rows with network data: exact join vs within 2 s (evaluation rows)")
display(pd.DataFrame({n: {"physical_rows": len(data[n]), "exact": int(NF[(n, 0, 0)]["net_pkts"].notna().sum()),
                          "within_2s (eval rows)": int(MASK[n].sum())} for n in ALL_NAMES}).T)

# ------------------------------------------------------------ variants
BASE_CFG = dict(tol=0, ff=0, ratio=False, entropy=False, drop=[])
VARIANTS = {
    "base":     dict(BASE_CFG),
    "align":    dict(BASE_CFG, tol=2),
    "ffill":    dict(BASE_CFG, ff=2),
    "ratio":    dict(BASE_CFG, ratio=True),
    "entropy":  dict(BASE_CFG, entropy=True),
    "drop":     dict(BASE_CFG, drop=DROP_COLS),
    "plan_all": dict(tol=2, ff=2, ratio=True, entropy=True, drop=DROP_COLS),
}
COMPONENTS = ["align", "ffill", "ratio", "entropy", "drop", "plan_all"]

def make_X(name, cfg, eps):
    nf = NF[(name, cfg["tol"], cfg["ff"])]
    keep = [c for c in NET_COLS if c not in cfg["drop"] and not (cfg["ratio"] and c in RATIO_COLS)]
    parts = [data[name][feature_cols].astype(float), nf[keep]]
    if cfg["ratio"]:
        parts.append(pd.DataFrame({c + "_ratio": nf[c] / (nf[c + "_med"] + eps[c]) for c in RATIO_COLS}, index=nf.index))
    if cfg["entropy"]:
        p = nf[FN_COLS].clip(lower=0)
        q = p.div(p.sum(axis=1).replace(0, np.nan), axis=0)
        parts.append((-(q * np.log(q.where(q > 0))).sum(axis=1, min_count=1)).rename("net_fn_entropy").to_frame())
    return pd.concat(parts, axis=1)

def recall_at_fpr(y, s, lim=0.05):
    fpr, tpr, _ = roc_curve(y, s)
    return float(tpr[fpr <= lim].max())

rows, sub_rows = [], []
for held in DEV_FOLDS:
    names = [n for n in DEV_FOLDS if n != held] + ["phy_norm"]
    eps = {}
    for c in RATIO_COLS:
        vals = pd.concat([data[n][c][MASK[n]] for n in names]).to_numpy(float)
        eps[c] = max(0.01 * float(np.nanquantile(vals, 0.99)), 1e-6)
    for vname, cfg in VARIANTS.items():
        Xtr = pd.concat([make_X(n, cfg, eps)[MASK[n]] for n in names])
        ytr_lab = pd.concat([data[n]["y"][MASK[n]] for n in names]).to_numpy()
        Xte = make_X(held, cfg, eps)[MASK[held]]
        te = data[held][MASK[held]]
        classes = sorted(set(ytr_lab)); idx = {c: i for i, c in enumerate(classes)}
        ytr = np.array([idx[c] for c in ytr_lab])
        truth = te["y"].to_numpy(); is_att = (truth != "normal").astype(int)
        for seed in SEEDS:
            m = xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1, subsample=0.8,
                                  colsample_bytree=0.8, tree_method="hist", n_jobs=-1, random_state=seed)
            m.fit(Xtr, ytr, sample_weight=compute_sample_weight("balanced", ytr))
            proba = m.predict_proba(Xte)
            pred = np.array(classes)[proba.argmax(1)]
            p_att = 1.0 - proba[:, classes.index("normal")]
            ok = 0 < is_att.sum() < len(is_att)
            present = [c for c in ALL_CLASSES if c in set(truth)]
            rows.append({"variant": vname, "fold": held, "seed": seed, "n_test": len(te), "n_features": Xtr.shape[1],
                         "roc_auc": roc_auc_score(is_att, p_att) if ok else np.nan,
                         "pr_auc": average_precision_score(is_att, p_att) if ok else np.nan,
                         "recall_at_5pct_fpr": recall_at_fpr(is_att, p_att) if ok else np.nan,
                         "cyber_recall": float(np.mean(pred[truth == "cyberattack"] == "cyberattack")) if (truth == "cyberattack").any() else np.nan,
                         "macro_f1": f1_score(truth, pred, labels=present, average="macro", zero_division=0)})
            if vname in ("base", "entropy", "plan_all"):
                d = pd.DataFrame({"sub": te["target_class"].astype(str).to_numpy(), "det": pred != "normal"})
                for sub, g in d.groupby("sub"):
                    sub_rows.append({"variant": vname, "fold": held, "subtype": sub, "n": len(g), "detected": g["det"].mean()})
    print("done fold:", held)

res = pd.DataFrame(rows)
res.to_csv(WORK_DIR / "step21_results.csv", index=False)

# ------------------------------------------------------------ verdicts (frozen rule)
def tab(metric, v):
    return res[res.variant == v].groupby("fold")[metric].agg(["mean", "std"])

def verdicts(folds, metric="roc_auc"):
    out = []
    for comp in COMPONENTS:
        b, v = tab(metric, "base"), tab(metric, comp)
        row, ok = {"step": comp}, True
        for f in folds:
            d, sd = v.loc[f, "mean"] - b.loc[f, "mean"], b.loc[f, "std"]
            row[f"{f} delta"], row[f"{f} base_sd"] = round(float(d), 4), round(float(sd), 4)
            ok &= bool(np.isfinite(d) and np.isfinite(sd) and d > 0 and d > sd)
        row["verdict"] = "KEEP" if ok else "DROP"
        out.append(row)
    return pd.DataFrame(out)

pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
V3 = verdicts(DEV_FOLDS); V3.to_csv(WORK_DIR / "step21_verdicts.csv", index=False)
V2 = verdicts([f for f in DEV_FOLDS if f != "phy_att_1"]); V2.to_csv(WORK_DIR / "step21_verdicts_sensitivity.csv", index=False)
print("\nVERDICTS, frozen rule (ROC-AUC attack vs normal, all three folds, paired baseline = current fused XGBoost)")
display(V3)
print("\nSENSITIVITY (phy_att_2 and phy_att_3 only; labeled post-hoc, because phy_att_1 has about 189 network rows)")
display(V2)
print("\nSECONDARY: recall at 5% false alarms, delta vs baseline per fold (not used for verdicts)")
sec = []
for comp in COMPONENTS:
    b, v = tab("recall_at_5pct_fpr", "base"), tab("recall_at_5pct_fpr", comp)
    sec.append({"step": comp, **{f: round(float(v.loc[f, "mean"] - b.loc[f, "mean"]), 4) for f in DEV_FOLDS}})
display(pd.DataFrame(sec))
print("\nFOLD MEANS, base vs all steps together")
display(res[res.variant.isin(["base", "plan_all"])].groupby(["variant", "fold"])[
    ["n_test", "n_features", "roc_auc", "pr_auc", "recall_at_5pct_fpr", "cyber_recall", "macro_f1"]].mean().round(3))
st = pd.DataFrame(sub_rows).groupby(["fold", "subtype", "variant"])[["n", "detected"]].mean().reset_index()
st = st.pivot_table(index=["fold", "subtype", "n"], columns="variant", values="detected").reset_index().round(3)
print("\nDETECTION BY SUBTYPE (argmax is not normal; for 'normal' this is the false-alarm rate)")
display(st)
st.to_csv(WORK_DIR / "step21_subtype_detection.csv", index=False)
print("\nStep 21 complete (XGBoost). Scaling for the LSTM is Step 22. phy_att_4 was not used.")

Rows with network data: exact join vs within 2 s (evaluation rows)


,physical_rows,exact,within_2s (eval rows)
phy_norm,3428,3425,3428
phy_att_1,2420,189,191
phy_att_2,2104,2096,2100
phy_att_3,1254,1181,1184


done fold: phy_att_1
done fold: phy_att_2
done fold: phy_att_3

VERDICTS, frozen rule (ROC-AUC attack vs normal, all three folds, paired baseline = current fused XGBoost)


,step,phy_att_1 delta,phy_att_1 base_sd,phy_att_2 delta,phy_att_2 base_sd,phy_att_3 delta,phy_att_3 base_sd,verdict
0,align,0.0148,0.0388,0.0097,0.0049,0.0066,0.0078,DROP
1,ffill,0.0196,0.0388,0.0065,0.0049,0.0043,0.0078,DROP
2,ratio,-0.0105,0.0388,0.0329,0.0049,-0.0072,0.0078,DROP
3,entropy,0.0132,0.0388,-0.0010,0.0049,-0.0016,0.0078,DROP
4,drop,0.0205,0.0388,-0.0125,0.0049,-0.0028,0.0078,DROP
5,plan_all,0.0044,0.0388,0.0310,0.0049,-0.0083,0.0078,DROP



SENSITIVITY (phy_att_2 and phy_att_3 only; labeled post-hoc, because phy_att_1 has about 189 network rows)


,step,phy_att_2 delta,phy_att_2 base_sd,phy_att_3 delta,phy_att_3 base_sd,verdict
0,align,0.0097,0.0049,0.0066,0.0078,DROP
1,ffill,0.0065,0.0049,0.0043,0.0078,DROP
2,ratio,0.0329,0.0049,-0.0072,0.0078,DROP
3,entropy,-0.0010,0.0049,-0.0016,0.0078,DROP
4,drop,-0.0125,0.0049,-0.0028,0.0078,DROP
5,plan_all,0.0310,0.0049,-0.0083,0.0078,DROP


... (3 outputs left out)

n_test  n_features  roc_auc  pr_auc  recall_at_5pct_fpr  cyber_recall  macro_f1
variant  fold                                                                                      
base     phy_att_1   191.0        54.0    0.686   0.577               0.464         0.105     0.521
         phy_att_2  2100.0        54.0    0.851   0.468               0.410         0.332     0.644
         phy_att_3  1184.0        54.0    0.797   0.739               0.580         0.395     0.477
plan_all phy_att_1   191.0        53.0    0.690   0.515               0.294         0.085     0.500
         phy_att_2  2100.0        53.0    0.882   0.538               0.535         0.231     0.663
         phy_att_3  1184.0        53.0    0.789   0.728               0.544         0.350     0.460


DETECTION BY SUBTYPE (argmax is not normal; for 'normal' this is the false-alarm rate)


variant,fold,subtype,n,base,entropy,plan_all
0,phy_att_1,MITM,51.0,0.105,0.111,0.085
1,phy_att_1,normal,140.0,0.007,0.014,0.014
2,phy_att_2,DoS,80.0,0.604,0.638,0.450
3,phy_att_2,MITM,96.0,0.205,0.208,0.087
4,phy_att_2,normal,1794.0,0.091,0.094,0.043
5,phy_att_2,physical fault,123.0,0.802,0.789,0.829
6,phy_att_2,scan,7.0,0.333,0.333,0.286
7,phy_att_3,DoS,77.0,0.913,0.918,0.779
8,phy_att_3,MITM,106.0,0.019,0.019,0.038
9,phy_att_3,normal,849.0,0.001,0.001,0.001



Step 21 complete (XGBoost). Scaling for the LSTM is Step 22. phy_att_4 was not used.


In [43]:
# STEP 22: The adjudicated fusion steps on the LSTM (development folds only; phy_att_4 untouched).
# Needs Step 14b, Step 16 (SEQ, train_predict) and Step 21 (NF, make_X, RATIO_COLS, DROP_COLS) run in this kernel.
# Rule frozen before any result: same as Steps 14, 16 and 21 (ROC-AUC gain > paired-baseline seed SD in ALL three folds).

from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score, roc_curve

for v in ["data", "NET_COLS", "feature_cols", "DEV_FOLDS", "SEEDS", "ALL_CLASSES", "WORK_DIR",
          "SEQ", "train_predict", "NF", "make_X", "RATIO_COLS", "DROP_COLS", "FN_COLS"]:
    assert v in globals(), f"Missing {v}: run Steps 14b, 16 and 21 first."
assert "phy_att_4" not in DEV_FOLDS

NP = len(feature_cols)
CLASS_ORDER = sorted(ALL_CLASSES)

# ---------------------------------------------------------------- frozen design
VARIANT_CFG = {
    "base":     dict(ratio=False, entropy=False, drop=[],        robust=False),   # = Step 16 net_scaled_cc
    "ratio":    dict(ratio=True,  entropy=False, drop=[],        robust=False),
    "entropy":  dict(ratio=False, entropy=True,  drop=[],        robust=False),
    "drop":     dict(ratio=False, entropy=False, drop=DROP_COLS, robust=False),
    "robust":   dict(ratio=False, entropy=False, drop=[],        robust=True),
    "plan_lstm": dict(ratio=True, entropy=True,  drop=DROP_COLS, robust=True),
}
COMPONENTS = ["ratio", "entropy", "drop", "robust", "plan_lstm"]
(WORK_DIR / "step22_frozen_plan.md").write_text(
    f"# Step 22 frozen LSTM fusion test\nWritten {datetime.now().isoformat(timespec='seconds')} BEFORE any result.\n"
    "- Model and training: exactly Step 16 (1-layer LSTM, hidden 64, 15 epochs, 10-row sequences, class-weighted loss).\n"
    "- Paired baseline: Step 16 net_scaled_cc (physical + 14 raw network columns, min-max scaled).\n"
    "- Steps tested: ratio to 60 s baseline, function entropy, drop port columns, robust scaling of the network columns "
    "(median/IQR from training NORMAL rows), and all four together (plan_lstm).\n"
    "- NOT tested here: alignment tolerance and forward-fill. The LSTM only accepts sequences whose 10 rows all have network "
    "data, so these steps only change which rows are eligible and cannot be paired; they were tested in Step 21 (XGBoost).\n"
    "- Same sequences for every variant: a sequence is eligible if all 10 rows have network data AND finite ratio and entropy values.\n"
    "- Rule: ROC-AUC gain > baseline seed SD in all three folds. Sensitivity: phy_att_2 and phy_att_3 only.\n", encoding="utf-8")

# ---------------------------------------------------------------- eligible rows (identical for every variant)
FULL = dict(tol=0, ff=0, ratio=True, entropy=True, drop=[])
ONES = {c: 1.0 for c in RATIO_COLS}
OK = {}
for n in ["phy_norm"] + list(DEV_FOLDS):
    X = make_X(n, FULL, ONES).to_numpy(float)
    OK[n] = np.isfinite(X).all(axis=1) & data[n]["covered"].to_numpy()
print("Eligible rows per acquisition (network present, ratio and entropy defined):",
      {n: int(m.sum()) for n, m in OK.items()})

def sequences(X, y, ok):
    n = len(X)
    idx = np.arange(SEQ - 1, n)
    csum = np.concatenate([[0], np.cumsum(ok.astype(int))])
    idx = idx[(csum[idx + 1] - csum[idx + 1 - SEQ]) == SEQ]
    if not len(idx):
        return np.empty((0, SEQ, X.shape[1]), dtype=np.float32), np.empty(0, dtype=object)
    return np.stack([X[i - SEQ + 1:i + 1] for i in idx]).astype(np.float32), y[idx]

def build22(variant, held):
    cfg = VARIANT_CFG[variant]
    names = [n for n in DEV_FOLDS if n != held] + ["phy_norm"]
    eps = {}
    for c in RATIO_COLS:
        vals = pd.concat([data[n][c][OK[n]] for n in names]).to_numpy(float)
        eps[c] = max(0.01 * float(np.nanquantile(vals, 0.99)), 1e-6)
    mk = dict(tol=0, ff=0, ratio=cfg["ratio"], entropy=cfg["entropy"], drop=cfg["drop"])
    frames = {n: make_X(n, mk, eps) for n in names + [held]}
    cols = list(frames[held].columns)
    Xtr_l, ytr_l = [], []
    for n in names:
        w, y = sequences(frames[n].to_numpy(float), data[n]["y"].to_numpy(), OK[n])
        Xtr_l.append(w); ytr_l.append(y)
    Xtr, ytr = np.concatenate(Xtr_l), np.concatenate(ytr_l)
    Xte, yte = sequences(frames[held].to_numpy(float), data[held]["y"].to_numpy(), OK[held])
    flat = Xtr.reshape(-1, Xtr.shape[-1])
    lo, hi = flat.min(axis=0), flat.max(axis=0)
    span = np.where(hi - lo == 0, 1.0, hi - lo)
    center, scale = lo.copy(), span.copy()          # min-max everywhere ...
    if cfg["robust"]:                                # ... except network columns: median/IQR of training NORMAL rows
        rows = np.concatenate([frames[n].to_numpy(float)[OK[n] & (data[n]["y"].to_numpy() == "normal")] for n in names])
        med = np.median(rows, axis=0)
        iqr = np.quantile(rows, 0.75, axis=0) - np.quantile(rows, 0.25, axis=0)
        net_idx = np.arange(NP, len(cols))
        center[net_idx] = med[net_idx]
        scale[net_idx] = np.where(iqr[net_idx] == 0, 1.0, iqr[net_idx])
    Xtr = ((Xtr - center) / scale).astype(np.float32)
    Xte = ((Xte - center) / scale).astype(np.float32)
    assert np.isfinite(Xtr).all() and np.isfinite(Xte).all(), "non-finite values reached the LSTM"
    return Xtr, ytr, Xte, yte

def recall_at_fpr(y, s, lim=0.05):
    fpr, tpr, _ = roc_curve(y, s)
    return float(tpr[fpr <= lim].max())

# ---------------------------------------------------------------- run
rows = []
for variant in VARIANT_CFG:
    for held in DEV_FOLDS:
        Xtr, ytr, Xte, yte = build22(variant, held)
        for seed in SEEDS:
            classes, proba = train_predict(Xtr, ytr, Xte, seed)
            pred = np.array(classes)[proba.argmax(1)]
            is_att = (yte != "normal").astype(int)
            p_att = 1.0 - proba[:, classes.index("normal")]
            present = [c for c in CLASS_ORDER if c in set(yte)]
            ok = 0 < is_att.sum() < len(is_att) and np.isfinite(p_att).all()
            rows.append({"variant": variant, "fold": held, "seed": seed, "n_train": len(ytr), "n_test": len(yte),
                         "n_features": Xtr.shape[-1],
                         "roc_auc": roc_auc_score(is_att, p_att) if ok else np.nan,
                         "pr_auc": average_precision_score(is_att, p_att) if ok else np.nan,
                         "recall_at_5pct_fpr": recall_at_fpr(is_att, p_att) if ok else np.nan,
                         "cyber_recall": float(np.mean(pred[yte == "cyberattack"] == "cyberattack")) if (yte == "cyberattack").any() else np.nan,
                         "specificity": float(np.mean(pred[yte == "normal"] == "normal")) if (yte == "normal").any() else np.nan,
                         "macro_f1": f1_score(yte, pred, labels=present, average="macro", zero_division=0)})
    print("done:", variant)
res = pd.DataFrame(rows)
res.to_csv(WORK_DIR / "step22_results.csv", index=False)
nt = res.groupby(["fold", "variant"])["n_test"].first().unstack()
assert (nt.nunique(axis=1) == 1).all(), "variants were not scored on the same test sequences"

# ---------------------------------------------------------------- verdicts
def tab(metric, v):
    return res[res.variant == v].groupby("fold")[metric].agg(["mean", "std"])

def verdicts(folds):
    out = []
    for comp in COMPONENTS:
        b, v = tab("roc_auc", "base"), tab("roc_auc", comp)
        row, ok = {"step": comp}, True
        for f in folds:
            d, sd = v.loc[f, "mean"] - b.loc[f, "mean"], b.loc[f, "std"]
            row[f"{f} delta"], row[f"{f} base_sd"] = round(float(d), 4), round(float(sd), 4)
            ok &= bool(np.isfinite(d) and np.isfinite(sd) and d > 0 and d > sd)
        row["verdict"] = "KEEP" if ok else "DROP"
        out.append(row)
    return pd.DataFrame(out)

pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
V3 = verdicts(DEV_FOLDS); V3.to_csv(WORK_DIR / "step22_verdicts.csv", index=False)
V2 = verdicts([f for f in DEV_FOLDS if f != "phy_att_1"]); V2.to_csv(WORK_DIR / "step22_verdicts_sensitivity.csv", index=False)
print("\nLSTM VERDICTS, frozen rule (ROC-AUC, all three folds, paired baseline = Step 16 net_scaled_cc)")
display(V3)
print("\nSENSITIVITY (phy_att_2 and phy_att_3 only; labeled post-hoc)")
display(V2)
print("\nSECONDARY: recall at 5% false alarms, delta vs baseline (not used for verdicts)")
display(pd.DataFrame([{"step": c, **{f: round(float(tab("recall_at_5pct_fpr", c).loc[f, "mean"] - tab("recall_at_5pct_fpr", "base").loc[f, "mean"]), 4) for f in DEV_FOLDS}} for c in COMPONENTS]))
print("\nFOLD MEANS (base vs all four steps together)")
display(res[res.variant.isin(["base", "plan_lstm"])].groupby(["variant", "fold"])[
    ["n_train", "n_test", "n_features", "roc_auc", "pr_auc", "recall_at_5pct_fpr", "cyber_recall", "specificity", "macro_f1"]].mean().round(3))
print("\nStep 22 complete. phy_att_4 was not used.")

Eligible rows per acquisition (network present, ratio and entropy defined): {'phy_norm': 3415, 'phy_att_1': 179, 'phy_att_2': 2086, 'phy_att_3': 1171}
done: base
done: ratio
done: entropy
done: drop
done: robust
done: plan_lstm

LSTM VERDICTS, frozen rule (ROC-AUC, all three folds, paired baseline = Step 16 net_scaled_cc)


,step,phy_att_1 delta,phy_att_1 base_sd,phy_att_2 delta,phy_att_2 base_sd,phy_att_3 delta,phy_att_3 base_sd,verdict
0,ratio,-0.0602,0.2309,-0.0163,0.0299,-0.0739,0.0117,DROP
1,entropy,-0.0648,0.2309,0.0177,0.0299,-0.0326,0.0117,DROP
2,drop,-0.1419,0.2309,-0.0012,0.0299,-0.0454,0.0117,DROP
3,robust,0.0259,0.2309,0.0338,0.0299,0.0666,0.0117,DROP
4,plan_lstm,0.1472,0.2309,0.0053,0.0299,-0.0154,0.0117,DROP



SENSITIVITY (phy_att_2 and phy_att_3 only; labeled post-hoc)


,step,phy_att_2 delta,phy_att_2 base_sd,phy_att_3 delta,phy_att_3 base_sd,verdict
0,ratio,-0.0163,0.0299,-0.0739,0.0117,DROP
1,entropy,0.0177,0.0299,-0.0326,0.0117,DROP
2,drop,-0.0012,0.0299,-0.0454,0.0117,DROP
3,robust,0.0338,0.0299,0.0666,0.0117,KEEP
4,plan_lstm,0.0053,0.0299,-0.0154,0.0117,DROP



SECONDARY: recall at 5% false alarms, delta vs baseline (not used for verdicts)


,step,phy_att_1,phy_att_2,phy_att_3
0,ratio,0.0000,-0.0261,-0.1431
1,entropy,0.0000,-0.0033,-0.0030
2,drop,0.0000,0.0414,-0.0891
3,robust,0.0476,0.2353,0.0841
4,plan_lstm,0.0816,0.1569,-0.1151



FOLD MEANS (base vs all four steps together)


n_train  n_test  n_features  roc_auc  pr_auc  recall_at_5pct_fpr  cyber_recall  specificity  macro_f1
variant   fold                                                                                                            
base      phy_att_1   6645.0   170.0        54.0    0.354   0.242               0.000         0.095        0.595     0.378
          phy_att_2   4738.0  2077.0        54.0    0.581   0.164               0.034         0.135        0.586     0.315
          phy_att_3   5653.0  1162.0        54.0    0.714   0.534               0.224         0.166        0.962     0.360
plan_lstm phy_att_1   6645.0   170.0        53.0    0.501   0.326               0.082         0.340        0.719     0.607
          phy_att_2   4738.0  2077.0        53.0    0.587   0.303               0.191         1.000        0.000     0.054
          phy_att_3   5653.0  1162.0        53.0    0.698   0.503               0.109         0.199        0.940     0.378


Step 22 complete. phy_att_4 was not used.


In [45]:
# STEP 23a: Freeze the final-test protocol, then build attack_4's per-second network features and check coverage.
# No model is trained and no attack_4 LABEL or physical feature value is read here (only the physical Time column).
# Needs Step 14a in this kernel (scan, clean_name, PAIRS) and its saved step14a_network_features_meta.json.

import json
from datetime import datetime

import numpy as np
import pandas as pd

for v in ["network_files", "physical_multiclass", "WORK_DIR", "scan", "clean_name", "PAIRS", "DEV_FOLDS"]:
    assert v in globals(), f"Missing {v}: run Step 14a first."
assert "phy_att_4" in physical_multiclass, "phy_att_4 was not loaded in Step 5."
assert pd.api.types.is_datetime64_any_dtype(physical_multiclass["phy_att_4"]["Time"]), \
    "Run the day-first date fix on physical_multiclass first (it covers every acquisition)."
meta = json.loads((WORK_DIR / "step14a_network_features_meta.json").read_text(encoding="utf-8"))
TOP_FN, NET_COLS_DEV = meta["top_modbus_fn"], meta["net_cols"]

# ---------------------------------------------------------------- 1) the protocol, written BEFORE attack_4 is touched
PROTOCOL = f"""# Step 23 frozen final-test protocol
Written {datetime.now().isoformat(timespec='seconds')} BEFORE attack_4 network data or phy_att_4 data were read.

## Data
- Training: phy_att_1, phy_att_2, phy_att_3 and phy_norm (all development data). Test: phy_att_4, once.
- Evaluation rows: seconds that have network data by exact-second join (the development protocol). 
  LSTM: 10-row sequences whose rows all have network data. The raw physical arm is also scored on all rows.
- The run goes ahead as declared whatever attack_4's coverage turns out to be; the covered row count is reported.

## Arms (every configuration below is reported; none is chosen after seeing attack_4)
| Model | Primary (chosen by the frozen rule on development folds) | Reference arms | Secondary (declared, exploratory) |
|---|---|---|---|
| XGBoost | A3: physical + 14 raw per-second network columns | A1: raw physical (all rows and covered rows); A2: physical + packets/s + mean packet size | agent's full fusion plan (Step 21 plan_all) |
| LSTM | A3: net_scaled_cc (physical + 14 network columns, min-max scaled) | A1: raw physical (unscaled, all rows) and phys_scaled_cc; A2: physical + 2 minimal columns, min-max scaled (LSTM cannot train on unscaled packet counts) | robust-scaled network columns (post-hoc observation, Step 22) |
| Qwen3-32B | Q_raw (no component passed the rule) | none | Q_raw_netmin (raw + 2 network numbers), Q_ref_net (reference summary + network) |
- Models and settings exactly as in development: XGBoost 300 trees depth 5 lr 0.1, balanced weights; LSTM Step 16 (hidden 64, 15 epochs,
  10-row sequences); seeds 0, 1, 2. Qwen: temperature 0, thinking off, 75 attack + 75 normal covered rows, decision cutoff 50, Step 17 prompts.
- No tuning, no threshold calibration, no retraining after the test. Decision rule for the binary call: argmax class is not normal (Qwen: score >= 50).

## Metrics
- All models: ROC-AUC (attack vs normal), recall at 5% false alarms, recall, specificity. XGBoost and LSTM additionally: precision, F1, PR-AUC, balanced accuracy, 3-class macro-F1.
- Qwen's balanced subsample inflates precision, accuracy, F1 and PR-AUC; those are not compared across models.
- Uncertainty: seed mean and SD for XGBoost/LSTM; block-bootstrap 95% interval (60-second blocks, 1,000 resamples) for ROC-AUC and recall at 5% false alarms.
- A single test acquisition cannot support the KEEP/DROP rule, so comparisons between arms are descriptive, with intervals.

## One-time rule
- The final-test cell is run once. All result files are written before anything is printed.
- A rerun is allowed only if the run crashes before any result file is written, and any such rerun is disclosed.
- If it finishes, the numbers stand; a later-found bug is reported as an addendum and the original stays primary.
- A rehearsal switch runs the identical code with a development fold standing in for phy_att_4; it never reads phy_att_4.
"""
(WORK_DIR / "step23_frozen_protocol.md").write_text(PROTOCOL, encoding="utf-8")
print("Protocol frozen:", WORK_DIR / "step23_frozen_protocol.md")

# ---------------------------------------------------------------- 2) attack_4 per-second network features (fixed function vocabulary)
path = {p.stem: p for p in network_files}.get("attack_4")
assert path is not None, "attack_4 network file not found among network_files"
b, f, dist, n_packets = scan(path)
print(f"attack_4: {n_packets:,} packets -> {len(b):,} seconds with packets")
out = pd.DataFrame(index=b.index)
out["net_pkts"] = b["n"]
mean = b["size_sum"] / b["n"]
out["net_size_mean"] = mean
out["net_size_std"] = np.sqrt(np.maximum(b["size_sq"] / b["n"] - mean ** 2, 0))
out["net_size_max"] = b["size_max"]
out["net_resp_null_share"] = b["resp_null"] / b["n"]
for k, s in dist.items():
    out[f"net_distinct_{k}"] = s.reindex(out.index).fillna(0)
f = f.reindex(out.index).fillna(0)
tot = f.sum(axis=1).replace(0, np.nan)
for fn_name in TOP_FN:
    out[f"net_fn_{clean_name(fn_name)}"] = (f[fn_name] / tot) if fn_name in f.columns else 0.0
out["net_fn_other"] = 1.0 - sum(out[f"net_fn_{clean_name(x)}"] for x in TOP_FN)
out.index.name = "sec"
assert list(out.columns) == list(NET_COLS_DEV), "attack_4 columns differ from the development columns"
out.reset_index().to_csv(WORK_DIR / "step14a_network_features_phy_att_4.csv", index=False)
out[["net_pkts", "net_size_mean"]].reset_index().to_csv(WORK_DIR / "step10b_minimal_network_phy_att_4.csv", index=False)
unseen = sorted(set(f.columns) - set(TOP_FN))
print("Modbus functions in attack_4 outside the fixed development vocabulary (pooled into net_fn_other):", unseen or "none")

# ---------------------------------------------------------------- 3) coverage and clock check (physical Time column only)
EPOCH, ONE_S = pd.Timestamp("1970-01-01"), pd.Timedelta("1s")
ph = physical_multiclass["phy_att_4"]["Time"]
phys_sec = np.unique(((ph.dt.floor("s") - EPOCH) // ONE_S).to_numpy())
net_sec = np.unique(out.index.to_numpy())
net_set = set(net_sec.tolist())
exact = np.mean([s in net_set for s in phys_sec])
w2 = np.mean([any((s + d) in net_set for d in range(-2, 3)) for s in phys_sec])
fmt = lambda s: str(pd.Timestamp(int(s), unit="s"))
print(f"\nphy_att_4 physical rows (unique seconds): {len(phys_sec):,}   physical span {fmt(phys_sec[0])} -> {fmt(phys_sec[-1])}")
print(f"attack_4 network span: {fmt(net_sec[0])} -> {fmt(net_sec[-1])}   ({len(net_sec):,} seconds with packets)")
print(f"Share of physical seconds with network data: exact {exact:.4f}, within 2 s {w2:.4f}   (development: 0.9991, 0.0781, 0.9962, 0.9418)")
MAX_SHIFT = 3600
counts = np.zeros(2 * MAX_SHIFT + 1, dtype=np.int64)
lo = np.searchsorted(net_sec, phys_sec - MAX_SHIFT, side="left"); hi = np.searchsorted(net_sec, phys_sec + MAX_SHIFT, side="right")
for p, a, c in zip(phys_sec, lo, hi):
    if c > a:
        counts[net_sec[a:c] - p + MAX_SHIFT] += 1
cov = counts / len(phys_sec); plateau = np.where(cov >= cov.max() - 0.001)[0] - MAX_SHIFT
print(f"Clock check (+/- 1 hour): best coverage {cov.max():.4f} for shifts {plateau.min():+d} s to {plateau.max():+d} s "
      f"(coverage at 0: {cov[MAX_SHIFT]:.4f}). A narrow range far from 0 would mean a clock offset; a range containing 0 means aligned.")
pd.DataFrame([{"phys_seconds": len(phys_sec), "net_seconds": len(net_sec), "exact": round(exact, 4), "within_2s": round(w2, 4),
               "best_shift_from_s": int(plateau.min()), "best_shift_to_s": int(plateau.max()), "best_coverage": round(float(cov.max()), 4)}]
             ).to_csv(WORK_DIR / "step23a_attack4_coverage.csv", index=False)
print("\nStep 23a complete. No model was trained; no attack_4 labels or physical feature values were read.")

Protocol frozen: C:\WDT\agentic_multiclass\step23_frozen_protocol.md
attack_4: 2,033,686 packets -> 614 seconds with packets
Modbus functions in attack_4 outside the fixed development vocabulary (pooled into net_fn_other): none

phy_att_4 physical rows (unique seconds): 1,717   physical span 2022-02-21 14:45:18 -> 2022-02-21 15:13:54
attack_4 network span: 2022-02-21 14:45:25 -> 2022-02-21 14:55:38   (614 seconds with packets)
Share of physical seconds with network data: exact 0.3576, within 2 s 0.3599   (development: 0.9991, 0.0781, 0.9962, 0.9418)
Clock check (+/- 1 hour): best coverage 0.3576 for shifts -1097 s to +8 s (coverage at 0: 0.3576). A narrow range far from 0 would mean a clock offset; a range containing 0 means aligned.

Step 23a complete. No model was trained; no attack_4 labels or physical feature values were read.


## Test

In [48]:
# STEP 23b: THE FINAL TEST (attack_4), or a rehearsal of it on a development fold.
# Protocol: step23_frozen_protocol.md (written in Step 23a before attack_4 was read).
# REHEARSAL = True  -> identical code, a development fold stands in for attack_4; attack_4 is never read; reruns allowed.
# REHEARSAL = False -> the real run: attack_4 is the test set; each stage is run once and saved before anything is printed.
# Needs, in this kernel: Steps 9, 14b, 16, 17 (functions), 18 (metrics), 21, 22, 23a.

import json
import pickle
import time
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore", category=UserWarning)   # single-class warnings are expected when a class is absent

# ------------------------------------------------------------------ switches
REHEARSAL = False
REHEARSAL_FOLD = "phy_att_3"
CONFIRM_FINAL = "RUN ONCE"                         # must be exactly "RUN ONCE" when REHEARSAL is False
RUN_STAGES = ["xgboost", "lstm", "qwen"]   # a stage that already finished in the real run is never run again
QWEN_REHEARSAL_CALLS = 6                   # real calls during a rehearsal (parsing check only)
FINAL_CAP_USD = 1.00                       # Qwen spend cap for the real run (assumed ceiling prices from Step 17)
BOOTSTRAPS, BLOCK, BOOT_SEED = 1000, 60, 0
MIN_CLASS_ROWS = 20

need = ["data", "NET_COLS", "feature_cols", "DEV_FOLDS", "SEEDS", "ALL_CLASSES", "WORK_DIR", "frames", "physical_multiclass",
        "build", "make_samples", "train_predict", "SEQ", "build22", "OK", "NF", "TL", "MASK", "make_X", "FULL", "ONES",
        "load_timeline", "align", "all_metrics", "recall_at_fpr", "CLASSES3", "QWEN_CUTOFF",
        "make_prompt", "build_reference", "reference_text", "row_text", "fmt", "TASK", "call_qwen", "parse_reply",
        "run_items", "client", "ATT_PER_FOLD", "NORM_PER_FOLD", "SAMPLE_SEED", "PRICE_IN_PER_M", "PRICE_OUT_PER_M"]
missing = [v for v in need if v not in globals()]
assert not missing, f"Missing {missing}: run Steps 9, 14b, 16, 17, 18, 21, 22 and 23a in this kernel first."
assert "phy_att_4" not in DEV_FOLDS

TEST_NAME = REHEARSAL_FOLD if REHEARSAL else "phy_att_4"
PFX = "step23_rehearsal_" if REHEARSAL else "step23_"
SEEDS_USED = list(SEEDS[:1]) if REHEARSAL else list(SEEDS)
MIN_COLS = ["net_pkts", "net_size_mean"]
PHYS = list(feature_cols)
if REHEARSAL:
    assert REHEARSAL_FOLD in DEV_FOLDS
    print(f"REHEARSAL: {REHEARSAL_FOLD} stands in for attack_4. attack_4 is not read. Files are prefixed '{PFX}'.")
else:
    assert CONFIRM_FINAL == "RUN ONCE", 'Real run blocked: set CONFIRM_FINAL = "RUN ONCE" to confirm this is the one-time test.'
    amend = (f"\n## Amendment 1 (written {datetime.now().isoformat(timespec='seconds')}, before any attack_4 label or feature value was read)\n"
             "- Step 23a showed attack_4's network capture covers only about 36% of the physical run (first ~10 minutes), as for phy_att_1. The run proceeds as declared on the covered rows.\n"
             "- Qwen subsample sizes are 'up to' 75 attack and 75 normal covered rows (as in Step 17).\n"
             "- If the covered rows contain fewer than 20 rows of either class, ROC-AUC and related metrics for the networked arms are reported as undefined; the raw all-rows arm is still reported; nothing else changes.\n"
             "- Secondary LSTM arm (robust scaling) is scored on its own eligible sequences (ratio and entropy defined), as in Step 22.\n"
             "- Bootstrap blocks are 60 consecutive rows of the acquisition.\n"
             "- Stage rule: a stage (XGBoost, LSTM, Qwen) that crashes before saving its own result file may be run again; a finished stage is never run again.\n")
    with open(WORK_DIR / "step23_frozen_protocol.md", "a", encoding="utf-8") as fh:
        fh.write(amend)

def marker(stage):
    return WORK_DIR / f"step23_DONE_{stage}.txt"

# ------------------------------------------------------------------ the test acquisition
def register_test_acquisition(name):
    d = frames[name].copy()
    d["Time"] = physical_multiclass[name]["Time"]
    assert pd.api.types.is_datetime64_any_dtype(d["Time"])
    sec = ((d["Time"].dt.floor("s") - pd.Timestamp("1970-01-01")) // pd.Timedelta("1s")).to_numpy()
    net = pd.read_csv(WORK_DIR / f"step14a_network_features_{name}.csv").set_index("sec")
    nf = net.reindex(sec)
    for c in NET_COLS:
        d[c] = nf[c].to_numpy()
    d["covered"] = d["net_pkts"].notna().to_numpy()
    data[name] = d
    tl, phys = load_timeline(name)
    TL[name] = (tl, phys)
    for tol, ff in [(0, 0), (2, 0), (0, 2), (2, 2)]:
        f = align(tl, tol, ff).reindex(phys)
        f.index = d.index
        NF[(name, tol, ff)] = f
    MASK[name] = NF[(name, 2, 0)]["net_pkts"].notna().to_numpy()
    OK[name] = np.isfinite(make_X(name, FULL, ONES).to_numpy(float)).all(axis=1) & d["covered"].to_numpy()

if not REHEARSAL:
    register_test_acquisition(TEST_NAME)
TRAIN_NAMES = [n for n in DEV_FOLDS if n != TEST_NAME] + ["phy_norm"]
TEST = data[TEST_NAME]
cov_pos = np.where(TEST["covered"].to_numpy())[0]
y_cov = TEST["y"].to_numpy()[cov_pos]
n_att, n_norm = int((y_cov != "normal").sum()), int((y_cov == "normal").sum())
FEASIBLE = n_att >= MIN_CLASS_ROWS and n_norm >= MIN_CLASS_ROWS
print(f"Test acquisition: {TEST_NAME} | rows {len(TEST):,} | covered {len(cov_pos):,} | covered attack {n_att:,}, normal {n_norm:,} | feasible: {FEASIBLE}")

# ------------------------------------------------------------------ containers and helpers
PRED = {}      # (model, arm) -> dict(pos, truth, sub, p (list per seed), pred (list per seed), classes)
ROWS = []      # per-seed metric rows
ROLE = {}

def record(model, arm, role, pos, truth, sub, classes, proba_list, extra=None):
    ROLE[(model, arm)] = role
    P, PR = [], []
    for proba in proba_list:
        proba = np.asarray(proba)
        pred = np.array(classes)[proba.argmax(1)]
        p_att = 1.0 - proba[:, list(classes).index("normal")]
        P.append(p_att); PR.append(pred)
        m = all_metrics(truth, pred, p_att, pred != "normal")
        ROWS.append({"model": model, "arm": arm, "role": role, "seed": len(P) - 1, **m})
    PRED[(model, arm)] = {"pos": np.asarray(pos), "truth": np.asarray(truth), "sub": np.asarray(sub), "p": P, "pred": PR, "classes": list(classes)}

def seq_positions(ok_mask):
    n = len(ok_mask)
    idx = np.arange(SEQ - 1, n)
    csum = np.concatenate([[0], np.cumsum(ok_mask.astype(int))])
    return idx[(csum[idx + 1] - csum[idx + 1 - SEQ]) == SEQ]

def stage_wanted(stage):
    if stage not in RUN_STAGES:
        return False
    if (not REHEARSAL) and marker(stage).exists():
        print(f"Stage '{stage}' already finished in the real run: skipped (never run twice).")
        return False
    return True

# ------------------------------------------------------------------ stage 1: XGBoost
def run_xgboost():
    import xgboost as xgb
    from sklearn.utils.class_weight import compute_sample_weight
    def fit_pred(Xtr, ytr_lab, Xte, seed):
        classes = sorted(set(ytr_lab)); idx = {c: i for i, c in enumerate(classes)}
        y = np.array([idx[c] for c in ytr_lab])
        m = xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                              tree_method="hist", n_jobs=-1, random_state=seed)
        m.fit(Xtr.astype(float), y, sample_weight=compute_sample_weight("balanced", y))
        return classes, m.predict_proba(Xte.astype(float))
    ARMS = {"A1_physical_all_rows": ("reference", PHYS, False), "A1_physical_covered_rows": ("reference", PHYS, True),
            "A2_physical_plus_minimal_network": ("reference", PHYS + MIN_COLS, True),
            "A3_physical_plus_14_network": ("PRIMARY", PHYS + list(NET_COLS), True)}
    for arm, (role, cols, cov_only) in ARMS.items():
        tr = pd.concat([data[n] for n in TRAIN_NAMES]); te = TEST
        if cov_only:
            tr, te = tr[tr["covered"]], te[te["covered"]]
        pos = np.where(TEST["covered"].to_numpy())[0] if cov_only else np.arange(len(TEST))
        out = [fit_pred(tr[cols], tr["y"].to_numpy(), te[cols], s) for s in SEEDS_USED]
        record("XGBoost", arm, role, pos, te["y"].to_numpy(), te["target_class"].astype(str).to_numpy(), out[0][0], [o[1] for o in out])
        print("done: XGBoost", arm)
    # secondary: the agent's full fusion plan (Step 21 plan_all), trained on tolerance-2 rows, scored on exact-covered rows
    cfg = dict(tol=2, ff=2, ratio=True, entropy=True, drop=DROP_COLS)
    eps = {}
    for c in RATIO_COLS:
        vals = pd.concat([data[n][c][MASK[n]] for n in TRAIN_NAMES]).to_numpy(float)
        eps[c] = max(0.01 * float(np.nanquantile(vals, 0.99)), 1e-6)
    Xtr = pd.concat([make_X(n, cfg, eps)[MASK[n]] for n in TRAIN_NAMES])
    ytr = pd.concat([data[n]["y"][MASK[n]] for n in TRAIN_NAMES]).to_numpy()
    Xte_all = make_X(TEST_NAME, cfg, eps)
    cm = TEST["covered"].to_numpy()
    out = [fit_pred(Xtr, ytr, Xte_all[cm], s) for s in SEEDS_USED]
    record("XGBoost", "S1_agent_full_plan", "secondary", np.where(cm)[0], TEST["y"].to_numpy()[cm],
           TEST["target_class"].astype(str).to_numpy()[cm], out[0][0], [o[1] for o in out])
    print("done: XGBoost S1_agent_full_plan")

# ------------------------------------------------------------------ stage 2: LSTM
def build_min2(held):
    names = [n for n in DEV_FOLDS if n != held] + ["phy_norm"]
    def arrs(n):
        d = data[n]
        return {"phys": d[feature_cols].astype(float).to_numpy(), "net": d[MIN_COLS].astype(float).to_numpy(),
                "y": d["y"].to_numpy(), "covered": d["covered"].to_numpy()}
    cols = list(range(len(feature_cols)))
    tl = [make_samples(arrs(n), cols, True, True) for n in names]
    Xtr, ytr = np.concatenate([a for a, _ in tl]), np.concatenate([b for _, b in tl])
    Xte, yte = make_samples(arrs(held), cols, True, True)
    flat = Xtr.reshape(-1, Xtr.shape[-1]); lo, hi = np.nanmin(flat, axis=0), np.nanmax(flat, axis=0)
    span = np.where(hi - lo == 0, 1.0, hi - lo)
    return ((Xtr - lo) / span).astype(np.float32), ytr, ((Xte - lo) / span).astype(np.float32), yte

def run_lstm():
    cm = TEST["covered"].to_numpy()
    arms = [("A1_physical_raw_all_rows", "reference", lambda: build("raw", TEST_NAME), seq_positions(np.ones(len(TEST), bool))),
            ("A1_physical_scaled_covered", "reference", lambda: build("phys_scaled_cc", TEST_NAME), seq_positions(cm)),
            ("A2_physical_plus_minimal_network_scaled", "reference", lambda: build_min2(TEST_NAME), seq_positions(cm)),
            ("A3_physical_plus_14_network_scaled", "PRIMARY", lambda: build("net_scaled_cc", TEST_NAME), seq_positions(cm)),
            ("S2_robust_scaled_network", "secondary", lambda: build22("robust", TEST_NAME), seq_positions(OK[TEST_NAME]))]
    for arm, role, fn, pos in arms:
        Xtr, ytr, Xte, yte = fn()
        assert len(yte) == len(pos), f"{arm}: test sequences {len(yte)} vs positions {len(pos)}"
        sub = TEST["target_class"].astype(str).to_numpy()[pos]
        res = [train_predict(Xtr, ytr, Xte, s) for s in SEEDS_USED]
        record("LSTM", arm, role, pos, yte, sub, res[0][0], [r[1] for r in res])
        print("done: LSTM", arm)

# ------------------------------------------------------------------ stage 3: Qwen
def run_qwen():
    te = TEST[TEST["covered"]]
    pos_all = np.where(TEST["covered"].to_numpy())[0]
    a = np.where(te["y"].to_numpy() != "normal")[0]; nrm = np.where(te["y"].to_numpy() == "normal")[0]
    if len(a) == 0 or len(nrm) == 0:
        print("Qwen stage skipped: one class absent from the covered rows."); return None
    rng = np.random.default_rng(SAMPLE_SEED)
    pick = np.concatenate([rng.choice(a, min(ATT_PER_FOLD, len(a)), replace=False), rng.choice(nrm, min(NORM_PER_FOLD, len(nrm)), replace=False)])
    ref = build_reference(TEST_NAME, PHYS + list(NET_COLS))
    cache = {"phys": reference_text({c: ref[c] for c in PHYS}), "both": reference_text(ref)}
    items = []
    for p in pick:
        row = te.iloc[p].copy(); row[PHYS] = row[PHYS].astype(float)
        for var in ["Q_raw", "Q_raw_netmin", "Q_ref_net"]:
            prompt = (TASK + "\n\nObservation:\n" + row_text(row, PHYS + MIN_COLS) + "\n/no_think") if var == "Q_raw_netmin" \
                else make_prompt(var, row, ref, cache)
            items.append({"variant": var, "pos": int(pos_all[p]), "truth": str(row["y"]), "subtype": str(row["target_class"]), "prompt": prompt})
    if REHEARSAL:
        items = items[::max(1, len(items) // QWEN_REHEARSAL_CALLS)][:QWEN_REHEARSAL_CALLS]
    est_in = sum(len(i["prompt"]) for i in items) / 3.0
    print(f"Qwen: {len(items)} prompts, rough estimate ${est_in / 1e6 * PRICE_IN_PER_M + len(items) * 25 / 1e6 * PRICE_OUT_PER_M:.2f} (cap ${FINAL_CAP_USD})")
    results, spent, cost, stopped = run_items(items, FINAL_CAP_USD)
    rows = []
    for it, res in results:
        if res is None:
            continue
        text, tin, tout = res
        cls, score = parse_reply(text)
        rows.append({"variant": it["variant"], "pos": it["pos"], "truth": it["truth"], "subtype": it["subtype"], "reply": text[:200],
                     "cls": cls, "score": score, "tokens_in": tin, "tokens_out": tout})
    out = pd.DataFrame(rows)
    out.to_csv(WORK_DIR / f"{PFX}qwen_replies.csv", index=False)
    print(f"Qwen calls: {len(results)} | estimated spend ${cost:.3f}" + (" | STOPPED AT CAP" if stopped else "") +
          f" | unparsable replies: {int(out['score'].isna().sum())}")
    assert not stopped or REHEARSAL, "Qwen hit the spend cap before finishing: raise FINAL_CAP_USD and run only the 'qwen' stage again."
    for var, d in out.groupby("variant"):
        truth3 = d["truth"].to_numpy()
        pred3 = np.array([c if c in CLASSES3 else "normal" for c in d["cls"].fillna("normal")])
        score = d["score"].fillna(50.0).to_numpy()
        m = all_metrics(truth3, pred3, score / 100.0, score >= QWEN_CUTOFF)
        ROLE[("Qwen3-32B", var)] = "PRIMARY" if var == "Q_raw" else "secondary"
        ROWS.append({"model": "Qwen3-32B", "arm": var, "role": ROLE[("Qwen3-32B", var)], "seed": 0, **m})
        PRED[("Qwen3-32B", var)] = {"pos": d["pos"].to_numpy(), "truth": truth3, "sub": d["subtype"].to_numpy(),
                                    "p": [score / 100.0], "pred": [np.where(score >= QWEN_CUTOFF, pred3, "normal")], "classes": CLASSES3}
    return out

STAGE_FN = {"xgboost": run_xgboost, "lstm": run_lstm, "qwen": run_qwen}
FAILED = {}
if REHEARSAL:                                   # a rehearsal always starts clean
    for st in STAGE_FN:
        (WORK_DIR / f"{PFX}state_{st}.pkl").unlink(missing_ok=True)
for st in ["xgboost", "lstm", "qwen"]:
    if not stage_wanted(st):
        continue
    t0 = time.time()
    try:
        before, keys_before = len(ROWS), set(PRED)
        STAGE_FN[st]()
        pd.DataFrame(ROWS[before:]).to_csv(WORK_DIR / f"{PFX}metrics_by_seed_{st}.csv", index=False)   # saved before anything is printed
        new_keys = [k for k in PRED if k not in keys_before]
        with open(WORK_DIR / f"{PFX}state_{st}.pkl", "wb") as fh:
            pickle.dump({"pred": {k: PRED[k] for k in new_keys}, "role": {k: ROLE[k] for k in new_keys}, "rows": ROWS[before:]}, fh)
        if not REHEARSAL:
            marker(st).write_text(f"finished {datetime.now().isoformat(timespec='seconds')}\n", encoding="utf-8")
        print(f"stage '{st}' finished in {time.time() - t0:.0f}s and saved")
    except Exception as e:
        FAILED[st] = f"{type(e).__name__}: {str(e)[:200]}"
        print(f"stage '{st}' FAILED before saving: {FAILED[st]}")

# ------------------------------------------------------------------ aggregate every finished stage from its saved file
PRED, ROLE, ROWS = {}, {}, []
MISSING_STAGES = []
for st in ["xgboost", "lstm", "qwen"]:
    f = WORK_DIR / f"{PFX}state_{st}.pkl"
    if f.exists():
        with open(f, "rb") as fh:
            blob = pickle.load(fh)
        PRED.update(blob["pred"]); ROLE.update(blob["role"]); ROWS.extend(blob["rows"])
    else:
        MISSING_STAGES.append(st)
assert PRED, "No stage has finished yet: nothing to summarize."

# ------------------------------------------------------------------ uncertainty (block bootstrap over 60-row blocks)
res = pd.DataFrame(ROWS)
res.to_csv(WORK_DIR / f"{PFX}metrics_by_seed.csv", index=False)
rng_b = np.random.default_rng(BOOT_SEED)

def metric_pair(y, p):
    if not (0 < y.sum() < len(y)):
        return np.nan, np.nan
    return roc_auc_score(y, p), recall_at_fpr(y, p)

def boot(arms):
    """arms: list of (y_att, pos, p). Returns draws[B, len(arms), 2] (ROC-AUC, recall at 5% FPR), shared resamples."""
    pos0 = arms[0][1]
    blocks = pos0 // BLOCK
    ub = np.unique(blocks)
    groups = [np.where(blocks == b)[0] for b in ub]
    draws = np.full((BOOTSTRAPS, len(arms), 2), np.nan)
    for i in range(BOOTSTRAPS):
        idx = np.concatenate([groups[j] for j in rng_b.integers(0, len(groups), len(groups))])
        for k, (y, pos, p) in enumerate(arms):
            draws[i, k, :] = metric_pair(y[idx], p[idx])
    return draws

def ci(x):
    x = x[np.isfinite(x)]
    return (float(np.percentile(x, 2.5)), float(np.percentile(x, 97.5))) if len(x) > 20 else (np.nan, np.nan)

summ_rows, pair_rows, pred_rows = [], [], []
for (model, arm), d in PRED.items():
    y = (d["truth"] != "normal").astype(int); pm = np.mean(d["p"], axis=0)
    sub = res[(res.model == model) & (res.arm == arm)]
    dr = boot([(y, d["pos"], pm)])
    rc, r5 = ci(dr[:, 0, 0]), ci(dr[:, 0, 1])
    summ_rows.append({"model": model, "arm": arm, "role": ROLE[(model, arm)], "n": len(y), "attack_share": round(float(y.mean()), 3),
                      "roc_auc": sub["roc_auc"].mean(), "roc_auc_ci_lo": rc[0], "roc_auc_ci_hi": rc[1],
                      "recall_at_5pct_fpr": sub["recall_at_5pct_fpr"].mean(), "r5_ci_lo": r5[0], "r5_ci_hi": r5[1],
                      "recall": sub["recall"].mean(), "specificity": sub["specificity"].mean(),
                      "precision": sub["precision"].mean(), "f1": sub["f1"].mean(), "pr_auc": sub["pr_auc"].mean(),
                      "balanced_acc": sub["balanced_acc"].mean(), "macro_f1": sub["macro_f1"].mean(),
                      "accuracy": sub["accuracy"].mean(), "majority_acc": sub["majority_acc"].mean(),
                      "recall_physical_fault": sub["recall_physical_fault"].mean() if "recall_physical_fault" in sub else np.nan,
                      "recall_cyberattack": sub["recall_cyberattack"].mean() if "recall_cyberattack" in sub else np.nan,
                      **dict(zip(["TP", "FP", "FN", "TN"], np.mean([[int((f & (y == 1)).sum()), int((f & (y == 0)).sum()),
                                                                      int((~f & (y == 1)).sum()), int((~f & (y == 0)).sum())]
                                                                     for f in [pr != "normal" for pr in d["pred"]]], axis=0).round(1))),
                      "roc_auc_seed_sd": sub["roc_auc"].std() if len(sub) > 1 else np.nan})
    for i in range(len(y)):
        pred_rows.append({"model": model, "arm": arm, "pos": int(d["pos"][i]), "truth": d["truth"][i], "subtype": d["sub"][i], "p_attack": float(pm[i])})
PAIRS_SPEC = [("XGBoost", "A3_physical_plus_14_network", "A2_physical_plus_minimal_network"),
              ("XGBoost", "A2_physical_plus_minimal_network", "A1_physical_covered_rows"),
              ("XGBoost", "A3_physical_plus_14_network", "A1_physical_covered_rows"),
              ("LSTM", "A3_physical_plus_14_network_scaled", "A2_physical_plus_minimal_network_scaled"),
              ("LSTM", "A2_physical_plus_minimal_network_scaled", "A1_physical_scaled_covered"),
              ("LSTM", "A3_physical_plus_14_network_scaled", "A1_physical_scaled_covered")]
for model, hi, lo in PAIRS_SPEC:
    if (model, hi) in PRED and (model, lo) in PRED:
        a, b = PRED[(model, hi)], PRED[(model, lo)]
        if len(a["pos"]) == len(b["pos"]) and (a["pos"] == b["pos"]).all():
            y = (a["truth"] != "normal").astype(int)
            dr = boot([(y, a["pos"], np.mean(a["p"], axis=0)), (y, b["pos"], np.mean(b["p"], axis=0))])
            for j, metric in enumerate(["roc_auc", "recall_at_5pct_fpr"]):
                dd = dr[:, 0, j] - dr[:, 1, j]
                lo_, hi_ = ci(dd)
                pair_rows.append({"model": model, "higher_arm": hi, "lower_arm": lo, "metric": metric,
                                  "difference": float(np.nanmean(dr[:, 0, j]) - np.nanmean(dr[:, 1, j])) if np.isfinite(dd).any() else np.nan,
                                  "ci_lo": lo_, "ci_hi": hi_, "interval_excludes_0": bool(np.isfinite(lo_) and (lo_ > 0 or hi_ < 0))})
summ = pd.DataFrame(summ_rows); pairs = pd.DataFrame(pair_rows)
summ.to_csv(WORK_DIR / f"{PFX}summary.csv", index=False); pairs.to_csv(WORK_DIR / f"{PFX}pairwise.csv", index=False)
pd.DataFrame(pred_rows).to_csv(WORK_DIR / f"{PFX}predictions.csv", index=False)

# ------------------------------------------------------------------ print (everything above is already saved)
pd.set_option("display.width", 260); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 100)
print("\n" + "=" * 78 + f"\nRESULTS on {'REHEARSAL fold ' + TEST_NAME if REHEARSAL else 'attack_4 (phy_att_4)'}"
      + f"  | covered rows {len(cov_pos):,} (attack {n_att:,}, normal {n_norm:,})" + "\n" + "=" * 78)
if FAILED:
    print("STAGES THAT FAILED BEFORE SAVING (may be run again, nothing else):", FAILED)
if MISSING_STAGES:
    print("Stages without saved results yet:", MISSING_STAGES)
if not FEASIBLE:
    print("Covered rows have fewer than 20 rows of one class: ROC-AUC and related metrics for networked arms are not meaningful (declared in Amendment 1).")
thr = ["model", "arm", "role", "n", "attack_share", "roc_auc", "roc_auc_ci_lo", "roc_auc_ci_hi", "recall_at_5pct_fpr", "r5_ci_lo", "r5_ci_hi"]
std = ["model", "arm", "role", "n", "attack_share", "majority_acc", "accuracy", "precision", "recall", "specificity", "f1", "balanced_acc"]
SUBSAMPLE_NOTE = ("Qwen rows (*) are scored on a balanced subsample (about 50% attacks), so its accuracy, precision, F1 and the "
                  "'always normal' accuracy are NOT comparable with XGBoost/LSTM; its recall and specificity are.")
summ["arm"] = np.where(summ["model"] == "Qwen3-32B", "*" + summ["arm"], summ["arm"])
print("\nA1) THRESHOLD-FREE QUALITY, primary arms (does the score rank attacks above normal rows? comparable across models)")
display(summ[summ.role == "PRIMARY"][thr].round(3))
print("\nA2) STANDARD METRICS AT EACH MODEL'S OWN DECISION RULE, primary arms")
print("    accuracy should be compared with majority_acc (the score of always answering 'normal').")
display(summ[summ.role == "PRIMARY"][std].round(3))
print("    " + SUBSAMPLE_NOTE)
print("\nB1) ALL ARMS, threshold-free (none chosen after seeing the result)")
display(summ[thr].round(3))
print("\nB2) ALL ARMS, standard metrics: accuracy, precision, recall, specificity, F1 (XGBoost/LSTM: argmax is not normal; Qwen: score >= 50)")
display(summ[std].round(3))
print("    " + SUBSAMPLE_NOTE)
print("\nC) CONFUSION COUNTS (attack = positive; mean over seeds) AND PER-CLASS RECALL")
display(summ[["model", "arm", "TP", "FP", "FN", "TN", "recall_physical_fault", "recall_cyberattack", "macro_f1", "pr_auc"]].round(3))
print("\nD) PAIRED DIFFERENCES (same rows, block bootstrap, 95% interval)")
display(pairs.round(3))
print("\nE) DETECTION BY SUBTYPE at each arm's decision rule (for 'normal' this is the false-alarm rate)")
st = []
for (model, arm), d in PRED.items():
    flag = np.mean([pr != "normal" for pr in d["pred"]], axis=0)
    st.append(pd.DataFrame({"model": model, "arm": arm, "subtype": d["sub"], "flag": flag}))
display(pd.concat(st).groupby(["model", "arm", "subtype"])["flag"].agg(["mean", "size"]).round(3).unstack("subtype")["mean"])
if not REHEARSAL and not FAILED and not MISSING_STAGES:
    (WORK_DIR / "step23_FINAL_DONE.txt").write_text(f"all stages finished {datetime.now().isoformat(timespec='seconds')}\n", encoding="utf-8")
print("\nStep 23b complete." + (" (rehearsal: nothing here is a result; attack_4 was not read)" if REHEARSAL else " The numbers stand; they are not rerun or tuned."))

Test acquisition: phy_att_4 | rows 1,717 | covered 614 | covered attack 47, normal 567 | feasible: True
done: XGBoost A1_physical_all_rows
done: XGBoost A1_physical_covered_rows
done: XGBoost A2_physical_plus_minimal_network
done: XGBoost A3_physical_plus_14_network
done: XGBoost S1_agent_full_plan
stage 'xgboost' finished in 36s and saved
done: LSTM A1_physical_raw_all_rows
done: LSTM A1_physical_scaled_covered
done: LSTM A2_physical_plus_minimal_network_scaled
done: LSTM A3_physical_plus_14_network_scaled
done: LSTM S2_robust_scaled_network
stage 'lstm' finished in 89s and saved
Qwen: 366 prompts, rough estimate $0.44 (cap $1.0)
Qwen calls: 366 | estimated spend $0.914 | unparsable replies: 0
stage 'qwen' finished in 153s and saved

RESULTS on attack_4 (phy_att_4)  | covered rows 614 (attack 47, normal 567)

A1) THRESHOLD-FREE QUALITY, primary arms (does the score rank attacks above normal rows? comparable across models)


,model,arm,role,n,attack_share,roc_auc,roc_auc_ci_lo,roc_auc_ci_hi,recall_at_5pct_fpr,r5_ci_lo,r5_ci_hi
3,XGBoost,A3_physical_plus_14_network,PRIMARY,614,0.077,0.920,0.777,1.000,0.043,0.011,1.000
8,LSTM,A3_physical_plus_14_network_scaled,PRIMARY,605,0.078,0.858,0.543,0.905,0.518,0.000,0.837
10,Qwen3-32B,*Q_raw,PRIMARY,122,0.385,0.427,0.377,0.472,0.000,0.000,0.000



A2) STANDARD METRICS AT EACH MODEL'S OWN DECISION RULE, primary arms
    accuracy should be compared with majority_acc (the score of always answering 'normal').


,model,arm,role,n,attack_share,majority_acc,accuracy,precision,recall,specificity,f1,balanced_acc
3,XGBoost,A3_physical_plus_14_network,PRIMARY,614,0.077,0.923,0.600,0.161,1.000,0.567,0.277,0.783
8,LSTM,A3_physical_plus_14_network_scaled,PRIMARY,605,0.078,0.922,0.829,0.287,0.801,0.831,0.422,0.816
10,Qwen3-32B,*Q_raw,PRIMARY,122,0.385,0.615,0.533,0.000,0.000,0.867,0.000,0.433


    Qwen rows (*) are scored on a balanced subsample (about 50% attacks), so its accuracy, precision, F1 and the 'always normal' accuracy are NOT comparable with XGBoost/LSTM; its recall and specificity are.

B1) ALL ARMS, threshold-free (none chosen after seeing the result)


,model,arm,role,n,attack_share,roc_auc,roc_auc_ci_lo,roc_auc_ci_hi,recall_at_5pct_fpr,r5_ci_lo,r5_ci_hi
0,XGBoost,A1_physical_all_rows,reference,1717,0.325,0.627,0.481,0.765,0.041,0.003,0.328
1,XGBoost,A1_physical_covered_rows,reference,614,0.077,0.600,0.384,0.835,0.021,0.000,0.400
2,XGBoost,A2_physical_plus_minimal_network,reference,614,0.077,0.900,0.663,0.996,0.021,0.000,0.988
3,XGBoost,A3_physical_plus_14_network,PRIMARY,614,0.077,0.920,0.777,1.000,0.043,0.011,1.000
4,XGBoost,S1_agent_full_plan,secondary,614,0.077,0.944,0.593,0.999,0.738,0.101,1.000
5,LSTM,A1_physical_raw_all_rows,reference,1708,0.327,0.484,0.340,0.616,0.026,0.001,0.056
6,LSTM,A1_physical_scaled_covered,reference,605,0.078,0.328,0.250,0.640,0.014,0.000,0.429
7,LSTM,A2_physical_plus_minimal_network_scaled,reference,605,0.078,0.506,0.388,0.769,0.014,0.000,0.333
8,LSTM,A3_physical_plus_14_network_scaled,PRIMARY,605,0.078,0.858,0.543,0.905,0.518,0.000,0.837
9,LSTM,S2_robust_scaled_network,secondary,595,0.079,0.930,0.405,0.971,0.879,0.000,0.977


... (5 outputs left out)

,model,higher_arm,lower_arm,metric,difference,ci_lo,ci_hi,interval_excludes_0
0,XGBoost,A3_physical_plus_14_network,A2_physical_plus_minimal_network,roc_auc,0.056,0.001,0.206,True
1,XGBoost,A3_physical_plus_14_network,A2_physical_plus_minimal_network,recall_at_5pct_fpr,0.196,0.000,0.940,False
2,XGBoost,A2_physical_plus_minimal_network,A1_physical_covered_rows,roc_auc,0.251,-0.062,0.466,False
3,XGBoost,A2_physical_plus_minimal_network,A1_physical_covered_rows,recall_at_5pct_fpr,0.184,-0.167,0.984,False
4,XGBoost,A3_physical_plus_14_network,A1_physical_covered_rows,roc_auc,0.301,0.117,0.481,True
5,XGBoost,A3_physical_plus_14_network,A1_physical_covered_rows,recall_at_5pct_fpr,0.350,0.000,0.994,False
6,LSTM,A3_physical_plus_14_network_scaled,A2_physical_plus_minimal_network_scaled,roc_auc,0.228,-0.146,0.425,False
7,LSTM,A3_physical_plus_14_network_scaled,A2_physical_plus_minimal_network_scaled,recall_at_5pct_fpr,0.220,0.000,0.841,False
8,LSTM,A2_physical_plus_minimal_network_scaled,A1_physical_scaled_covered,roc_auc,0.124,-0.009,0.336,False
9,LSTM,A2_physical_plus_minimal_network_scaled,A1_physical_scaled_covered,recall_at_5pct_fpr,0.007,-0.256,0.245,False



E) DETECTION BY SUBTYPE at each arm's decision rule (for 'normal' this is the false-alarm rate)


subtype                                              DoS   MITM  normal  physical fault   scan
model     arm                                                                                 
LSTM      A1_physical_raw_all_rows                 0.673  0.220   0.441           0.363  0.571
          A1_physical_scaled_covered               0.000    NaN   0.167             NaN  0.400
          A2_physical_plus_minimal_network_scaled  0.000    NaN   0.165             NaN  0.400
          A3_physical_plus_14_network_scaled       0.849    NaN   0.169             NaN  0.400
          S2_robust_scaled_network                 1.000    NaN   0.112             NaN  0.067
Qwen3-32B Q_raw                                    0.000    NaN   0.133             NaN  0.000
          Q_raw_netmin                             0.500    NaN   0.573             NaN  0.400
          Q_ref_net                                1.000    NaN   0.867             NaN  1.000
XGBoost   A1_physical_all_rows                     0.000  0.288   0.095           0.030  0.429
          A1_physical_covered_rows                 0.000    NaN   0.082             NaN  0.400
          A2_physical_plus_minimal_network         1.000    NaN   0.126             NaN  0.400
          A3_physical_plus_14_network              1.000    NaN   0.433             NaN  1.000
          S1_agent_full_plan                       0.825    NaN   0.066             NaN  0.600


Step 23b complete. The numbers stand; they are not rerun or tuned.


## Summary

In [49]:
# ======================================================================
# FINAL RESULTS SUMMARY  (reads the saved final-test file; runs nothing new,
# does not touch attack_4, and downloads nothing)
# ======================================================================
from pathlib import Path
import pandas as pd

try:
    base = WORK_DIR                      # defined by the earlier setup cells
except NameError:
    base = Path(r"C:\WDT\agentic_multiclass")
summary_file = Path(base) / "step23_summary.csv"
s = pd.read_csv(summary_file)

LABELS = {
    ("XGBoost", "A1_physical_covered_rows"):             ("XGBoost", "No preprocessing: physical only"),
    ("XGBoost", "A2_physical_plus_minimal_network"):     ("XGBoost", "Minimal network (2 counts, no agent)"),
    ("XGBoost", "A3_physical_plus_14_network"):          ("XGBoost", "Agent's 14 network features (main)"),
    ("XGBoost", "S1_agent_full_plan"):                   ("XGBoost", "Agent's full fusion plan"),
    ("LSTM", "A1_physical_scaled_covered"):              ("LSTM", "No preprocessing: physical only"),
    ("LSTM", "A2_physical_plus_minimal_network_scaled"): ("LSTM", "Minimal network (2 counts, no agent)"),
    ("LSTM", "A3_physical_plus_14_network_scaled"):      ("LSTM", "Agent's 14 network features (main)"),
    ("LSTM", "S2_robust_scaled_network"):                ("LSTM", "Agent's full fusion plan (robust scaling)"),
    ("Qwen3-32B", "Q_raw"):                              ("Qwen3-32B*", "No preprocessing: physical only"),
    ("Qwen3-32B", "Q_raw_netmin"):                       ("Qwen3-32B*", "Minimal network (2 counts, no agent)"),
    ("Qwen3-32B", "Q_ref_net"):                          ("Qwen3-32B*", "Physical + network features"),
}
rows = []
for (model, arm), (m_lab, a_lab) in LABELS.items():
    r = s[(s["model"] == model) & (s["arm"].astype(str).str.lstrip("*") == arm)]
    if len(r) == 0:
        continue
    r = r.iloc[0]
    rows.append({"Model": m_lab, "Preprocessing / input": a_lab,
                 "ROC-AUC": r["roc_auc"], "Recall": r["recall"], "Specificity": r["specificity"],
                 "Precision": r["precision"], "F1": r["f1"],
                 "Accuracy": r["accuracy"], "Macro-F1": r["macro_f1"]})
table = pd.DataFrame(rows).round(3)
first = s[(s["model"] == "XGBoost") & (s["arm"] == "A3_physical_plus_14_network")].iloc[0]

print("FINAL TEST on attack_4 (one run, nothing retuned)")
print(f"Models are 3-class; scores treat any non-normal answer as 'attack'. "
      f"{int(first['n'])} test seconds with network data, attack share {first['attack_share']:.3f}; "
      f"'always normal' accuracy = {first['majority_acc']:.3f}.")
display(table.style.hide(axis="index").format({c: "{:.3f}" for c in table.columns[2:]}))
print("(main) = arm chosen before the test; 'agent' = preprocessing designed by Qwen3-32B and reviewed by a human.")
print("*Qwen was scored on a smaller balanced sample, so its accuracy, precision, F1 and macro-F1 are not comparable with XGBoost/LSTM.")
print("Covered attack rows are almost all DoS, so physical-fault and MITM detection could not be tested here.")

FINAL TEST on attack_4 (one run, nothing retuned)
Models are 3-class; scores treat any non-normal answer as 'attack'. 614 test seconds with network data, attack share 0.077; 'always normal' accuracy = 0.923.


Model,Preprocessing / input,ROC-AUC,Recall,Specificity,Precision,F1,Accuracy,Macro-F1
XGBoost,No preprocessing: physical only,0.600,0.043,0.918,0.041,0.042,0.851,0.481
XGBoost,"Minimal network (2 counts, no agent)",0.900,0.936,0.874,0.381,0.541,0.878,0.736
XGBoost,Agent's 14 network features (main),0.920,1.000,0.567,0.161,0.277,0.600,0.500
XGBoost,Agent's full fusion plan,0.944,0.801,0.934,0.502,0.617,0.924,0.787
LSTM,No preprocessing: physical only,0.328,0.043,0.833,0.021,0.028,0.771,0.453
LSTM,"Minimal network (2 counts, no agent)",0.506,0.043,0.835,0.021,0.028,0.774,0.454
LSTM,Agent's 14 network features (main),0.858,0.801,0.831,0.287,0.422,0.829,0.673
LSTM,Agent's full fusion plan (robust scaling),0.930,0.901,0.888,0.408,0.562,0.889,0.798
Qwen3-32B*,No preprocessing: physical only,0.427,0.000,0.867,0.000,0.000,0.533,0.348
Qwen3-32B*,"Minimal network (2 counts, no agent)",0.458,0.489,0.427,0.348,0.407,0.451,0.448


(main) = arm chosen before the test; 'agent' = preprocessing designed by Qwen3-32B and reviewed by a human.
*Qwen was scored on a smaller balanced sample, so its accuracy, precision, F1 and macro-F1 are not comparable with XGBoost/LSTM.
Covered attack rows are almost all DoS, so physical-fault and MITM detection could not be tested here.
